In [ ]:
import sys
from pathlib import Path
import pandas as pd

print("Python:", sys.executable)

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "Dataset_Final"

train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("\nProject:", PROJECT_ROOT)
print("Data:", DATA_DIR)
print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())

print("\nFirst 5 training rows:")
display(train.head())

In [ ]:
# Verify that CSV filenames match the actual audio files

train_audio_dir = DATA_DIR / "train"
test_audio_dir = DATA_DIR / "test"

train_missing = [
    f for f in train["filename"]
    if not (train_audio_dir / f).exists()
]

test_missing = [
    f for f in test["filename"]
    if not (test_audio_dir / f).exists()
]

print("Train audio files:", len(list(train_audio_dir.glob("*.wav"))))
print("Test audio files:", len(list(test_audio_dir.glob("*.wav"))))

print("\nMissing train files:", len(train_missing))
print("Missing test files:", len(test_missing))

if train_missing:
    print("Example missing train files:", train_missing[:5])

if test_missing:
    print("Example missing test files:", test_missing[:5])

In [ ]:
import numpy as np
import soundfile as sf

# -----------------------------
# Label distribution
# -----------------------------
print("TRAIN LABEL DISTRIBUTION")
print(train["label"].value_counts().sort_index())

print("\nLabel statistics:")
print(train["label"].describe())

# -----------------------------
# Audio duration
# -----------------------------
def get_duration(path):
    info = sf.info(path)
    return info.frames / info.samplerate

train_durations = np.array([
    get_duration(train_audio_dir / f)
    for f in train["filename"]
])

test_durations = np.array([
    get_duration(test_audio_dir / f)
    for f in test["filename"]
])

print("\nAUDIO DURATION")
print(f"Train mean: {train_durations.mean():.2f} sec")
print(f"Train min : {train_durations.min():.2f} sec")
print(f"Train max : {train_durations.max():.2f} sec")

print(f"\nTest mean : {test_durations.mean():.2f} sec")
print(f"Test min  : {test_durations.min():.2f} sec")
print(f"Test max  : {test_durations.max():.2f} sec")

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Use the observed score values as categorical strata
strata = train["label"].astype(str)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train["fold"] = -1

for fold, (_, val_idx) in enumerate(
    skf.split(train, strata)
):
    train.loc[val_idx, "fold"] = fold

print("Fold distribution:")
print(train["fold"].value_counts().sort_index())

print("\nLabel distribution by fold:")
print(
    pd.crosstab(
        train["fold"],
        train["label"],
        normalize="index"
    ).round(3)
)

In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import torch

MODEL_NAME = "microsoft/wavlm-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
wavlm = WavLMModel.from_pretrained(MODEL_NAME).to(device)
wavlm.eval()

print("WavLM loaded successfully")
print("Device:", device)

In [ ]:
import librosa
import numpy as np

# Load one training audio file
sample_file = train_audio_dir / train.iloc[0]["filename"]

audio, sr = librosa.load(
    sample_file,
    sr=16000,
    mono=True
)

print("Audio shape:", audio.shape)
print("Sample rate:", sr)
print("Duration:", round(len(audio) / sr, 2), "seconds")

# Prepare input for WavLM
inputs = feature_extractor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

input_values = inputs.input_values.to(device)

# Extract WavLM representation
with torch.no_grad():
    outputs = wavlm(input_values)

hidden_states = outputs.last_hidden_state

print("\nWavLM output shape:", hidden_states.shape)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import librosa
import soundfile as sf

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "Dataset_Final"

train_audio_dir = DATA_DIR / "train"
test_audio_dir = DATA_DIR / "test"

# Load datasets
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)
print("Train:", train.shape)
print("Test:", test.shape)
print("Device:", device)

In [ ]:
from transformers import WavLMModel, Wav2Vec2FeatureExtractor

MODEL_NAME = "microsoft/wavlm-base"

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    MODEL_NAME
)

wavlm = WavLMModel.from_pretrained(
    MODEL_NAME
).to(device)

wavlm.eval()

print("WavLM loaded successfully")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CACHE_DIR = OUTPUT_DIR / "wavlm_cache"

print("Project:", PROJECT_ROOT)
print("\nExisting output files:")

for folder in [OUTPUT_DIR, CACHE_DIR]:
    print(f"\n{folder}")
    if folder.exists():
        files = list(folder.iterdir())
        if files:
            for file in files:
                if file.is_file():
                    print(f"  {file.name} ({file.stat().st_size / 1024**2:.2f} MB)")
                else:
                    print(f"  {file.name}/")
        else:
            print("  Empty")
    else:
        print("  Folder does not exist")

In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"

print("Project exists:", PROJECT_DIR.exists())
print("Dataset directory:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())

if DATA_DIR.exists():
    for name in ["train.csv", "test.csv", "sample_submission.csv"]:
        path = DATA_DIR / name
        print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")

    for folder in ["train", "test"]:
        audio_dir = DATA_DIR / folder

        if audio_dir.is_dir():
            wav_files = list(audio_dir.rglob("*.wav"))
            wav_files += list(audio_dir.rglob("*.WAV"))
            print(f"{folder} WAV count: {len(wav_files)}")
        else:
            print(f"{folder} folder: MISSING")

    train_csv = DATA_DIR / "train.csv"
    test_csv = DATA_DIR / "test.csv"

    if train_csv.is_file():
        train_df = pd.read_csv(train_csv)
        print("\nTraining CSV shape:", train_df.shape)
        print("Training columns:", train_df.columns.tolist())
        print("Training label statistics:")
        print(train_df["label"].describe())

    if test_csv.is_file():
        test_df = pd.read_csv(test_csv)
        print("\nTest CSV shape:", test_df.shape)
        print("Test columns:", test_df.columns.tolist())
else:
    print("\nDataset path is incorrect. We must locate it before continuing.")


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

def audit_split(split_name, df):
    audio_dir = DATA_DIR / split_name

    # One search pattern only; Windows matching is case-insensitive.
    wav_paths = list(audio_dir.rglob("*.wav"))
    wav_by_name = {}

    for path in wav_paths:
        wav_by_name.setdefault(path.name.lower(), []).append(path)

    csv_names = df["filename"].astype(str).tolist()

    missing = [
        name for name in csv_names
        if Path(name).name.lower() not in wav_by_name
    ]

    duplicate_csv_names = (
        pd.Series([Path(n).name.lower() for n in csv_names])
        .duplicated()
        .sum()
    )

    print(f"\n--- {split_name.upper()} AUDIT ---")
    print("CSV rows:", len(df))
    print("Actual WAV files:", len(wav_paths))
    print("Unique WAV filenames:", len(wav_by_name))
    print("CSV filenames missing from audio folder:", len(missing))
    print("Repeated filenames in CSV:", duplicate_csv_names)

    if missing:
        print("Missing examples:", missing[:5])

    print("CSV filename examples:", csv_names[:3])
    print("Audio filename examples:", [p.name for p in wav_paths[:3]])

audit_split("train", train_df)
audit_split("test", test_df)

print("\n--- LABEL CHECK ---")
print("Training labels missing:", train_df["label"].isna().sum())
print("Training labels outside [0, 5]:",
      ((train_df["label"] < 0) | (train_df["label"] > 5)).sum())
print("Test labels:", sorted(test_df["label"].dropna().unique().tolist()))

print("\nAudit complete. No files modified.")


In [ ]:

from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)

search_dirs = [
    PROJECT_DIR / "outputs",
    PROJECT_DIR / "features",
    PROJECT_DIR / "checkpoints",
]

extensions = {".npy", ".npz", ".pt", ".pth", ".pkl", ".joblib", ".parquet"}

for folder in search_dirs:
    print(f"\n--- {folder} ---")

    if not folder.exists():
        print("Folder does not exist.")
        continue

    matches = [
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in extensions
    ]

    if matches:
        for path in matches[:20]:
            print(f"{path.name} ({path.stat().st_size / (1024**2):.2f} MB)")
        if len(matches) > 20:
            print(f"... and {len(matches) - 20} more files")
    else:
        print("No saved feature or checkpoint files found.")

print("\nSearch complete.")


In [ ]:

import sys
import importlib
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)

print("Python:", sys.version)
print("Executable:", sys.executable)

packages = [
    "torch",
    "transformers",
    "numpy",
    "pandas",
    "sklearn",
    "librosa",
    "soundfile",
]

print("\n--- PACKAGE VERSIONS ---")

for package in packages:
    try:
        module = importlib.import_module(package)
        version = getattr(module, "__version__", "version unavailable")
        print(f"{package}: {version}")
    except Exception as exc:
        print(f"{package}: ERROR — {type(exc).__name__}: {exc}")

print("\n--- GPU STATUS ---")

try:
    import torch

    print("PyTorch CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("Using CPU for local inference.")
except Exception as exc:
    print("PyTorch check failed:", repr(exc))

for folder in ["features", "checkpoints", "outputs/wavlm_cache"]:
    path = PROJECT_DIR / folder
    path.mkdir(parents=True, exist_ok=True)
    print("Directory ready:", path)

print("\nEnvironment check complete.")


In [ ]:

import torch

print("Potential model variables:")

for name, value in list(globals().items()):
    if name.startswith("_"):
        continue

    if isinstance(value, torch.nn.Module):
        print(f"Model candidate: {name} ({type(value).__name__})")

    elif (
        "processor" in name.lower()
        or "wavlm" in name.lower()
    ):
        print(f"Related variable: {name} ({type(value).__name__})")


In [ ]:

import transformers
import torch

print("Loaded WavLM model:", type(wavlm).__name__)
print("Model device:", next(wavlm.parameters()).device)

print("\nProcessor / feature extractor candidates:")

for name, value in list(globals().items()):
    if name.startswith("_"):
        continue

    if isinstance(value, transformers.ProcessorMixin):
        print(f"{name}: {type(value).__name__}")
    elif isinstance(value, transformers.FeatureExtractionMixin):
        print(f"{name}: {type(value).__name__}")

print("\nProcessor discovery complete.")


In [ ]:

import time
import numpy as np
import soundfile as sf
import pandas as pd
import torch
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"
TRAIN_DIR = DATA_DIR / "train"

train_df = pd.read_csv(DATA_DIR / "train.csv")
sample_names = train_df["filename"].head(3).tolist()

wavlm.eval()
times = []

for name in sample_names:
    audio_path = TRAIN_DIR / name
    audio, sr = sf.read(audio_path, dtype="float32")

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    if sr != 16000:
        raise ValueError(f"Unexpected sample rate: {sr}")

    inputs = feature_extractor(
        audio,
        sampling_rate=sr,
        return_tensors="pt"
    )

    start = time.perf_counter()

    with torch.inference_mode():
        output = wavlm(**inputs)

    elapsed = time.perf_counter() - start
    times.append(elapsed)

    embedding = output.last_hidden_state.mean(dim=1)

    print(
        f"{name}: {elapsed:.2f} seconds | "
        f"embedding shape: {tuple(embedding.shape)}"
    )

avg_seconds = float(np.mean(times))

print("\nAverage inference time:", round(avg_seconds, 2), "seconds/file")
print(
    "Estimated inference-only time for 985 files:",
    round(avg_seconds * 985 / 3600, 2),
    "hours"
)


In [ ]:

import os
import platform
import torch

print("Processor:", platform.processor() or "Not reported")
print("Logical CPU cores:", os.cpu_count())
print("PyTorch intra-op threads:", torch.get_num_threads())
print("PyTorch inter-op threads:", torch.get_num_interop_threads())

print("\nCurrent model device:", next(wavlm.parameters()).device)
print("Model in evaluation mode:", not wavlm.training)

print("\nCPU configuration check complete.")


In [ ]:

import time
import soundfile as sf
import pandas as pd
import torch
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"

train_df = pd.read_csv(DATA_DIR / "train.csv")
audio_path = DATA_DIR / "train" / train_df.iloc[0]["filename"]

audio, sr = sf.read(audio_path, dtype="float32")

if audio.ndim > 1:
    audio = audio.mean(axis=1)

inputs = feature_extractor(
    audio,
    sampling_rate=sr,
    return_tensors="pt"
)

wavlm.eval()
results = []

for num_threads in [2, 4, 6, 8]:
    torch.set_num_threads(num_threads)

    start = time.perf_counter()

    with torch.inference_mode():
        output = wavlm(**inputs)

    elapsed = time.perf_counter() - start

    results.append({
        "threads": num_threads,
        "seconds": round(elapsed, 2)
    })

    print(f"Threads: {num_threads} | Inference: {elapsed:.2f} seconds")

print("\nBenchmark summary:")
for result in sorted(results, key=lambda x: x["seconds"]):
    print(f'{result["threads"]} threads: {result["seconds"]} seconds')

best = min(results, key=lambda x: x["seconds"])
print(f'\nFastest tested setting: {best["threads"]} threads')
print(f'Estimated inference time for 985 files: '
      f'{best["seconds"] * 985 / 3600:.2f} hours')


In [ ]:

from pathlib import Path
import shutil
import torch
import pandas as pd

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"
CACHE_DIR = PROJECT_DIR / "outputs" / "wavlm_cache"

CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Use the fastest setting measured on your machine.
torch.set_num_threads(6)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

disk = shutil.disk_usage(PROJECT_DIR)

print("CPU threads:", torch.get_num_threads())
print("Training samples:", len(train_df))
print("Test samples:", len(test_df))
print("Total audio files:", len(train_df) + len(test_df))
print("Cache directory:", CACHE_DIR)
print(f"Free disk space: {disk.free / (1024**3):.2f} GB")
print("Existing cache files:", sum(1 for p in CACHE_DIR.iterdir() if p.is_file()))

print("\nCache setup complete. No audio processed yet.")


In [ ]:

# ============================================================
# SHL GRAMMAR SCORING — RESUMABLE WAVLM FEATURE EXTRACTION
# Frozen microsoft/wavlm-base | CPU | 6 threads
# ============================================================

import os
import json
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch

# ---------- 1. Project configuration ----------

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"
CACHE_ROOT = PROJECT_DIR / "outputs" / "wavlm_cache"
LOG_DIR = PROJECT_DIR / "outputs"

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"

MODEL_ID = "microsoft/wavlm-base"
POOLING = "mean"
EXPECTED_DIM = 768

TRAIN_CACHE = CACHE_ROOT / "train"
TEST_CACHE = CACHE_ROOT / "test"

TRAIN_CACHE.mkdir(parents=True, exist_ok=True)
TEST_CACHE.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ---------- 2. Validate existing model and processor ----------

if "wavlm" not in globals():
    raise RuntimeError("WavLM model 'wavlm' is not loaded.")

if "feature_extractor" not in globals():
    raise RuntimeError("Feature extractor is not loaded.")

if not TRAIN_CSV.is_file() or not TEST_CSV.is_file():
    raise FileNotFoundError("Training or test CSV is missing.")

device = torch.device("cpu")
wavlm = wavlm.to(device)
wavlm.eval()
torch.set_num_threads(6)

# ---------- 3. Load and validate manifests ----------

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

for split_name, df in [("train", train_df), ("test", test_df)]:
    if "filename" not in df.columns:
        raise ValueError(f"{split_name}.csv has no filename column.")

    if df["filename"].isna().any():
        raise ValueError(f"{split_name}.csv contains missing filenames.")

    if df["filename"].duplicated().any():
        raise ValueError(f"{split_name}.csv contains duplicate filenames.")

    if split_name == "train":
        if "label" not in df.columns:
            raise ValueError("Training labels are missing.")
        if df["label"].isna().any():
            raise ValueError("Training labels contain missing values.")
        if not df["label"].between(0, 5).all():
            raise ValueError("Training labels outside the range 0–5.")

print("Model device:", next(wavlm.parameters()).device)
print("CPU threads:", torch.get_num_threads())
print("Training files:", len(train_df))
print("Test files:", len(test_df))
print("Checkpoint folders:", TRAIN_CACHE, TEST_CACHE)

# ---------- 4. Safe checkpoint helpers ----------

def checkpoint_paths(cache_dir, filename):
    # CSV filenames are simple WAV basenames in this dataset.
    stem = Path(filename).stem
    return cache_dir / f"{stem}.npy", cache_dir / f"{stem}.json"


def valid_checkpoint(cache_dir, filename, audio_path):
    npy_path, json_path = checkpoint_paths(cache_dir, filename)

    if not npy_path.is_file() or not json_path.is_file():
        return False

    try:
        metadata = json.loads(json_path.read_text(encoding="utf-8"))
        embedding = np.load(npy_path, allow_pickle=False)

        return (
            metadata.get("filename") == filename
            and metadata.get("model") == MODEL_ID
            and metadata.get("pooling") == POOLING
            and metadata.get("source_size") == audio_path.stat().st_size
            and metadata.get("source_mtime_ns") == audio_path.stat().st_mtime_ns
            and embedding.shape == (EXPECTED_DIM,)
            and embedding.dtype == np.float32
            and np.isfinite(embedding).all()
        )
    except Exception:
        return False


def save_checkpoint(cache_dir, filename, audio_path, embedding, sample_rate):
    npy_path, json_path = checkpoint_paths(cache_dir, filename)
    tmp_npy = npy_path.with_name(npy_path.stem + ".tmp.npy")
    tmp_json = json_path.with_name(json_path.stem + ".tmp")

    metadata = {
        "filename": filename,
        "model": MODEL_ID,
        "pooling": POOLING,
        "embedding_dimension": EXPECTED_DIM,
        "sample_rate": int(sample_rate),
        "source_size": audio_path.stat().st_size,
        "source_mtime_ns": audio_path.stat().st_mtime_ns,
    }

    # Write temporary files first; completed checkpoints have both files.
    np.save(tmp_npy, embedding, allow_pickle=False)
    tmp_json.write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8"
    )

    tmp_npy.replace(npy_path)
    tmp_json.replace(json_path)


# ---------- 5. Process one split with resume support ----------

failure_log = LOG_DIR / "wavlm_failures.csv"
failed_rows = []
total_start = time.perf_counter()
processed_total = 0
skipped_total = 0

def extract_split(split_name, df, audio_folder, cache_dir):
    global processed_total, skipped_total

    completed = 0
    skipped = 0
    failures = []

    print(f"\n{'=' * 60}", flush=True)
    print(f"STARTING {split_name.upper()} EXTRACTION", flush=True)
    print(f"{'=' * 60}", flush=True)

    for i, filename_value in enumerate(df["filename"], start=1):
        filename = str(filename_value)

        # Prevent paths in the CSV from escaping the audio directory.
        if Path(filename).name != filename:
            failures.append({
                "split": split_name,
                "filename": filename,
                "error": "Filename is not a basename"
            })
            continue

        audio_path = audio_folder / filename

        if not audio_path.is_file():
            failures.append({
                "split": split_name,
                "filename": filename,
                "error": "Audio file missing"
            })
            continue

        if valid_checkpoint(cache_dir, filename, audio_path):
            skipped += 1
            skipped_total += 1
            continue

        try:
            audio, sr = sf.read(audio_path, dtype="float32")

            if audio.ndim == 2:
                audio = audio.mean(axis=1)

            if audio.ndim != 1 or audio.size == 0:
                raise ValueError("Audio is empty or has unsupported shape.")

            # The verified dataset is expected to be mono 16 kHz.
            if sr != 16000:
                raise ValueError(
                    f"Expected 16000 Hz, found {sr} Hz."
                )

            if not np.isfinite(audio).all():
                raise ValueError("Audio contains NaN or infinite samples.")

            inputs = feature_extractor(
                audio,
                sampling_rate=sr,
                return_tensors="pt"
            )

            with torch.inference_mode():
                output = wavlm(**inputs)

            embedding = (
                output.last_hidden_state
                .mean(dim=1)
                .squeeze(0)
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            if embedding.shape != (EXPECTED_DIM,):
                raise ValueError(
                    f"Unexpected embedding shape: {embedding.shape}"
                )

            if not np.isfinite(embedding).all():
                raise ValueError("Embedding contains invalid values.")

            save_checkpoint(
                cache_dir, filename, audio_path, embedding, sr
            )

            # Read it back before counting the file as complete.
            if not valid_checkpoint(cache_dir, filename, audio_path):
                raise IOError("Saved checkpoint verification failed.")

            completed += 1
            processed_total += 1

        except Exception as exc:
            failures.append({
                "split": split_name,
                "filename": filename,
                "error": f"{type(exc).__name__}: {exc}"
            })

        # Persist failure information during the run.
        if failures:
            existing = pd.DataFrame(failures)
            existing.to_csv(
                LOG_DIR / f"wavlm_failures_{split_name}.csv",
                index=False
            )

        if i % 10 == 0 or i == len(df):
            elapsed = time.perf_counter() - total_start
            print(
                f"{split_name}: {i}/{len(df)} examined | "
                f"{completed} newly saved | {skipped} reused | "
                f"{len(failures)} failures | "
                f"{elapsed / 3600:.2f} hours elapsed",
                flush=True
            )

    print(
        f"{split_name.upper()} DONE: "
        f"{completed} newly saved, {skipped} reused, "
        f"{len(failures)} failures",
        flush=True
    )
    return failures


# ---------- 6. Extract training and test separately ----------

train_failures = extract_split(
    "train",
    train_df,
    DATA_DIR / "train",
    TRAIN_CACHE
)

test_failures = extract_split(
    "test",
    test_df,
    DATA_DIR / "test",
    TEST_CACHE
)

all_failures = train_failures + test_failures

if all_failures:
    pd.DataFrame(all_failures).to_csv(failure_log, index=False)
elif failure_log.exists():
    failure_log.unlink()

# ---------- 7. Final integrity audit ----------

def count_valid(df, audio_folder, cache_dir):
    count = 0
    for filename in df["filename"].astype(str):
        path = audio_folder / filename
        if path.is_file() and valid_checkpoint(cache_dir, filename, path):
            count += 1
    return count

valid_train = count_valid(
    train_df, DATA_DIR / "train", TRAIN_CACHE
)
valid_test = count_valid(
    test_df, DATA_DIR / "test", TEST_CACHE
)

elapsed_hours = (time.perf_counter() - total_start) / 3600

print("\n" + "=" * 60)
print("FINAL EXTRACTION AUDIT")
print("=" * 60)
print(f"Valid training embeddings: {valid_train}/{len(train_df)}")
print(f"Valid test embeddings:     {valid_test}/{len(test_df)}")
print("New embeddings saved:", processed_total)
print("Existing embeddings reused:", skipped_total)
print("Failures in this run:", len(all_failures))
print(f"Elapsed time: {elapsed_hours:.2f} hours")
print("Training cache:", TRAIN_CACHE)
print("Test cache:", TEST_CACHE)

if valid_train == len(train_df) and valid_test == len(test_df):
    print("\nSUCCESS: All 985 embeddings are verified.")
else:
    print("\nINCOMPLETE: Fix or retry failed files before modeling.")
    if failure_log.exists():
        print("Failure log:", failure_log)


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"
CACHE_ROOT = PROJECT_DIR / "outputs" / "wavlm_cache"
FEATURE_DIR = PROJECT_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

def load_embeddings(df, split):
    cache_dir = CACHE_ROOT / split
    embeddings = []

    for filename in df["filename"].astype(str):
        path = cache_dir / f"{Path(filename).stem}.npy"

        if not path.is_file():
            raise FileNotFoundError(f"Missing embedding: {path}")

        embedding = np.load(path, allow_pickle=False)

        if embedding.shape != (768,):
            raise ValueError(
                f"{filename}: unexpected shape {embedding.shape}"
            )

        if not np.isfinite(embedding).all():
            raise ValueError(f"{filename}: non-finite embedding values")

        embeddings.append(embedding)

    return np.stack(embeddings).astype(np.float32)

X_train_wavlm = load_embeddings(train_df, "train")
X_test_wavlm = load_embeddings(test_df, "test")
y_train = train_df["label"].to_numpy(dtype=np.float32)

assert X_train_wavlm.shape == (769, 768)
assert X_test_wavlm.shape == (216, 768)
assert y_train.shape == (769,)

np.save(FEATURE_DIR / "X_train_wavlm.npy", X_train_wavlm)
np.save(FEATURE_DIR / "X_test_wavlm.npy", X_test_wavlm)
np.save(FEATURE_DIR / "y_train.npy", y_train)

print("Training embeddings:", X_train_wavlm.shape)
print("Test embeddings:", X_test_wavlm.shape)
print("Training labels:", y_train.shape)
print("Training embedding dtype:", X_train_wavlm.dtype)
print("All values finite:", 
      np.isfinite(X_train_wavlm).all()
      and np.isfinite(X_test_wavlm).all())
print("Feature matrices saved to:", FEATURE_DIR)


In [ ]:

import time
import numpy as np
import pandas as pd
import librosa
import soundfile as sf

from pathlib import Path

# --------------------------------------------------
# 1. Paths and directories
# --------------------------------------------------
PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"
CACHE_ROOT = PROJECT_DIR / "outputs" / "acoustic_cache"
FEATURE_DIR = PROJECT_DIR / "features"

for split in ["train", "test"]:
    (CACHE_ROOT / split).mkdir(parents=True, exist_ok=True)

EXPECTED_FEATURES = 47


# --------------------------------------------------
# 2. Extract acoustic and prosodic features
# --------------------------------------------------
def extract_acoustic_features(audio_path):
    audio, sr = sf.read(audio_path, dtype="float32")

    # Convert stereo audio to mono
    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    if len(audio) == 0 or not np.isfinite(audio).all():
        raise ValueError(f"Invalid audio: {audio_path}")

    # Resample to 16 kHz if required
    if sr != 16000:
        audio = librosa.resample(
            audio, orig_sr=sr, target_sr=16000
        )
        sr = 16000

    duration = len(audio) / sr

    # Frame-level acoustic features
    rms = librosa.feature.rms(
        y=audio, frame_length=1024, hop_length=512
    )[0]

    zcr = librosa.feature.zero_crossing_rate(
        audio, frame_length=1024, hop_length=512
    )[0]

    centroid = librosa.feature.spectral_centroid(
        y=audio, sr=sr, n_fft=1024, hop_length=512
    )[0]

    bandwidth = librosa.feature.spectral_bandwidth(
        y=audio, sr=sr, n_fft=1024, hop_length=512
    )[0]

    rolloff = librosa.feature.spectral_rolloff(
        y=audio, sr=sr, n_fft=1024, hop_length=512
    )[0]

    flatness = librosa.feature.spectral_flatness(
        y=audio, n_fft=1024, hop_length=512
    )[0]

    features = [
        duration,                          # 1
        float(np.mean(rms)),               # 2
        float(np.std(rms)),                # 3
        float(np.percentile(rms, 10)),     # 4
        float(np.percentile(rms, 90)),     # 5
        float(np.mean(zcr)),               # 6
        float(np.std(zcr)),                # 7
        float(np.mean(centroid)),          # 8
        float(np.std(centroid)),           # 9
        float(np.mean(bandwidth)),         # 10
        float(np.std(bandwidth)),          # 11
        float(np.mean(rolloff)),           # 12
        float(np.mean(flatness)),          # 13
        float(np.std(flatness)),           # 14
    ]

    # Approximate low-energy / silence statistics
    energy_threshold = max(float(np.max(rms)) * 0.05, 1e-5)
    low_energy = rms < energy_threshold

    transitions = np.diff(
        np.r_[False, low_energy, False].astype(int)
    )

    features.extend([
        float(np.mean(low_energy)),        # 15
        float(np.sum(transitions == 1)),   # 16
    ])

    # MFCC mean and standard deviation: 13 + 13 features
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=13,
        n_fft=1024,
        hop_length=512
    )

    features.extend(np.mean(mfcc, axis=1).tolist())  # 17–29
    features.extend(np.std(mfcc, axis=1).tolist())   # 30–42

    # Fundamental frequency statistics: 5 features
    f0 = librosa.yin(
        audio,
        fmin=70,
        fmax=400,
        sr=sr,
        frame_length=1024,
        hop_length=512
    )

    voiced = np.isfinite(f0) & (f0 >= 70) & (f0 <= 400)
    voiced_f0 = f0[voiced]

    features.extend([
        float(np.mean(voiced)) if len(voiced) else 0.0,
        float(np.mean(voiced_f0)) if len(voiced_f0) else 0.0,
        float(np.std(voiced_f0)) if len(voiced_f0) else 0.0,
        float(np.percentile(voiced_f0, 10)) if len(voiced_f0) else 0.0,
        float(np.percentile(voiced_f0, 90)) if len(voiced_f0) else 0.0,
    ])  # 43–47

    result = np.asarray(features, dtype=np.float32)

    if result.shape != (EXPECTED_FEATURES,):
        raise ValueError(
            f"{audio_path.name}: expected {EXPECTED_FEATURES} features, "
            f"got {result.shape}"
        )

    if not np.isfinite(result).all():
        raise ValueError(f"Non-finite features: {audio_path}")

    return result


# --------------------------------------------------
# 3. Build or resume the feature cache
# --------------------------------------------------
def build_acoustic_cache(split):
    df = pd.read_csv(DATA_DIR / f"{split}.csv")
    audio_dir = DATA_DIR / split
    cache_dir = CACHE_ROOT / split

    start_time = time.time()
    reused = 0
    extracted = 0

    for i, filename in enumerate(
        df["filename"].astype(str), start=1
    ):
        cache_path = cache_dir / f"{Path(filename).stem}.npy"

        # Reuse only valid cached vectors
        if cache_path.exists():
            try:
                cached = np.load(cache_path, allow_pickle=False)

                if (
                    cached.shape == (EXPECTED_FEATURES,)
                    and np.isfinite(cached).all()
                ):
                    reused += 1
                    continue
            except (ValueError, OSError):
                pass

        audio_path = audio_dir / filename

        if not audio_path.is_file():
            raise FileNotFoundError(
                f"Missing audio file: {audio_path}"
            )

        vector = extract_acoustic_features(audio_path)

        # Atomic save avoids leaving a partial cache file
        temp_path = cache_path.with_name(
            cache_path.stem + ".tmp.npy"
        )
        np.save(temp_path, vector)
        temp_path.replace(cache_path)

        extracted += 1

        if i % 25 == 0 or i == len(df):
            print(
                f"{split}: {i}/{len(df)} checked | "
                f"reused={reused} | extracted={extracted} | "
                f"elapsed={time.time() - start_time:.1f}s"
            )

    # Verify every file in the split
    for filename in df["filename"].astype(str):
        cache_path = cache_dir / f"{Path(filename).stem}.npy"

        if not cache_path.is_file():
            raise FileNotFoundError(
                f"Missing feature cache: {cache_path}"
            )

        vector = np.load(cache_path, allow_pickle=False)

        if vector.shape != (EXPECTED_FEATURES,):
            raise ValueError(
                f"{filename}: invalid cached shape {vector.shape}"
            )

        if not np.isfinite(vector).all():
            raise ValueError(
                f"{filename}: non-finite cached values"
            )

    print(f"\n{split.upper()} CACHE VERIFIED")
    print(f"Total recordings: {len(df)}")
    print(f"Reused cache files: {reused}")
    print(f"Newly extracted: {extracted}")
    print(f"Elapsed: {(time.time() - start_time) / 60:.2f} minutes")


# --------------------------------------------------
# 4. Process both splits
# --------------------------------------------------
build_acoustic_cache("train")
build_acoustic_cache("test")

print("\nSUCCESS: Acoustic feature caches verified.")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

# Paths
PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"
CACHE_ROOT = PROJECT_DIR / "outputs" / "acoustic_cache"
FEATURE_DIR = PROJECT_DIR / "features"

EXPECTED_FEATURES = 47


def load_acoustic_matrix(split):
    df = pd.read_csv(DATA_DIR / f"{split}.csv")
    cache_dir = CACHE_ROOT / split

    filenames = df["filename"].astype(str)

    # Avoid accidentally mapping different filenames to the same cache
    stems = [Path(name).stem for name in filenames]
    if len(stems) != len(set(stems)):
        raise ValueError(
            f"Duplicate filename stems found in {split} CSV."
        )

    rows = []

    for filename in filenames:
        cache_path = cache_dir / f"{Path(filename).stem}.npy"

        if not cache_path.is_file():
            raise FileNotFoundError(
                f"Missing cached features: {cache_path}"
            )

        vector = np.load(cache_path, allow_pickle=False)

        if vector.shape != (EXPECTED_FEATURES,):
            raise ValueError(
                f"{filename}: unexpected shape {vector.shape}"
            )

        if not np.isfinite(vector).all():
            raise ValueError(
                f"{filename}: non-finite feature values"
            )

        rows.append(vector)

    return np.stack(rows).astype(np.float32), filenames.tolist()


# Load features in the exact CSV order
X_train_acoustic, train_filenames = load_acoustic_matrix("train")
X_test_acoustic, test_filenames = load_acoustic_matrix("test")

# Load labels and previously saved WavLM matrices
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

X_train_wavlm = np.load(
    FEATURE_DIR / "X_train_wavlm.npy", allow_pickle=False
)
X_test_wavlm = np.load(
    FEATURE_DIR / "X_test_wavlm.npy", allow_pickle=False
)
y_train = train_df["label"].to_numpy(dtype=np.float32)

# Critical alignment and integrity checks
assert train_filenames == train_df["filename"].astype(str).tolist()
assert test_filenames == test_df["filename"].astype(str).tolist()

assert X_train_acoustic.shape == (769, 47)
assert X_test_acoustic.shape == (216, 47)
assert X_train_wavlm.shape == (769, 768)
assert X_test_wavlm.shape == (216, 768)
assert y_train.shape == (769,)

assert np.isfinite(X_train_acoustic).all()
assert np.isfinite(X_test_acoustic).all()
assert np.isfinite(X_train_wavlm).all()
assert np.isfinite(X_test_wavlm).all()
assert np.isfinite(y_train).all()
assert np.all((y_train >= 0) & (y_train <= 5))

# Save the assembled acoustic matrices
np.save(FEATURE_DIR / "X_train_acoustic.npy", X_train_acoustic)
np.save(FEATURE_DIR / "X_test_acoustic.npy", X_test_acoustic)

print("TRAIN acoustic matrix:", X_train_acoustic.shape)
print("TEST acoustic matrix: ", X_test_acoustic.shape)
print("TRAIN WavLM matrix:   ", X_train_wavlm.shape)
print("TEST WavLM matrix:    ", X_test_wavlm.shape)
print("Training labels:      ", y_train.shape)
print("Label range:          ", y_train.min(), "to", y_train.max())
print("All alignment and integrity checks passed.")
print("Feature matrices saved to:", FEATURE_DIR)


In [ ]:

import time
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------------------------------------------------
# 1. Load validated feature matrices
# --------------------------------------------------
PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
FEATURE_DIR = PROJECT_DIR / "features"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_wavlm = np.load(FEATURE_DIR / "X_train_wavlm.npy")
X_acoustic = np.load(FEATURE_DIR / "X_train_acoustic.npy")
y = np.load(FEATURE_DIR / "y_train.npy")

feature_sets = {
    "WavLM": X_wavlm,
    "Acoustic": X_acoustic,
    "Combined": np.hstack([X_wavlm, X_acoustic]).astype(np.float32),
}

assert len(y) == 769
assert all(X.shape[0] == len(y) for X in feature_sets.values())

# --------------------------------------------------
# 2. Fixed folds for fair comparison
# --------------------------------------------------
cv = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(X_wavlm, y))

results = []
oof_predictions = {}

# --------------------------------------------------
# 3. Evaluate mean baseline and ExtraTrees
# --------------------------------------------------
experiments = [
    ("MeanBaseline", None),
    ("ExtraTrees", ExtraTreesRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    )),
]

for feature_name, X in feature_sets.items():
    for model_name, model_template in experiments:
        if model_name == "MeanBaseline":
            # The mean is calculated using only each training fold
            predictions = np.zeros(len(y), dtype=np.float32)
            fold_rmse, fold_mae, fold_pearson = [], [], []
            start = time.time()

            for train_idx, valid_idx in folds:
                model = DummyRegressor(strategy="mean")
                model.fit(np.zeros((len(train_idx), 1)), y[train_idx])
                pred = model.predict(np.zeros((len(valid_idx), 1)))
                predictions[valid_idx] = pred

                actual = y[valid_idx]
                fold_rmse.append(
                    np.sqrt(mean_squared_error(actual, pred))
                )
                fold_mae.append(mean_absolute_error(actual, pred))

                corr = np.corrcoef(actual, pred)[0, 1]
                fold_pearson.append(
                    float(corr) if np.isfinite(corr) else np.nan
                )

        else:
            predictions = np.zeros(len(y), dtype=np.float32)
            fold_rmse, fold_mae, fold_pearson = [], [], []
            start = time.time()

            for fold_num, (train_idx, valid_idx) in enumerate(
                folds, start=1
            ):
                # Fresh model for each fold
                model = ExtraTreesRegressor(
                    n_estimators=400,
                    min_samples_leaf=2,
                    max_features=1.0,
                    random_state=42,
                    n_jobs=-1
                )

                model.fit(X[train_idx], y[train_idx])
                pred = model.predict(X[valid_idx])
                predictions[valid_idx] = pred

                actual = y[valid_idx]
                fold_rmse.append(
                    np.sqrt(mean_squared_error(actual, pred))
                )
                fold_mae.append(mean_absolute_error(actual, pred))

                corr = np.corrcoef(actual, pred)[0, 1]
                fold_pearson.append(
                    float(corr) if np.isfinite(corr) else np.nan
                )

                print(
                    f"{feature_name} | fold {fold_num}/5 | "
                    f"RMSE={fold_rmse[-1]:.4f} | "
                    f"MAE={fold_mae[-1]:.4f}"
                )

        # Aggregate metrics over all out-of-fold predictions
        pooled_corr = np.corrcoef(y, predictions)[0, 1]

        results.append({
            "Features": feature_name,
            "Model": model_name,
            "OOF_RMSE": np.sqrt(mean_squared_error(y, predictions)),
            "OOF_MAE": mean_absolute_error(y, predictions),
            "OOF_Pearson": pooled_corr,
            "Mean_Fold_RMSE": np.mean(fold_rmse),
            "Mean_Fold_MAE": np.mean(fold_mae),
            "Mean_Fold_Pearson": np.nanmean(fold_pearson),
            "Seconds": time.time() - start,
        })

        key = f"{feature_name}_{model_name}"
        oof_predictions[key] = predictions

# --------------------------------------------------
# 4. Display and save results
# --------------------------------------------------
results_df = pd.DataFrame(results).sort_values(
    by="OOF_RMSE", ascending=True
)

print("\nCROSS-VALIDATION RESULTS — sorted by pooled OOF RMSE")
print(results_df.round(4).to_string(index=False))

results_df.to_csv(
    OUTPUT_DIR / "cv_feature_comparison.csv", index=False
)

oof_df = pd.DataFrame({"actual": y})
for name, pred in oof_predictions.items():
    oof_df[name] = pred

oof_df.to_csv(OUTPUT_DIR / "oof_predictions.csv", index=False)

print("\nResults saved:")
print(OUTPUT_DIR / "cv_feature_comparison.csv")
print(OUTPUT_DIR / "oof_predictions.csv")


In [ ]:

import time
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------------------------------------------------
# 1. Load the combined feature matrix
# --------------------------------------------------
PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
FEATURE_DIR = PROJECT_DIR / "features"
OUTPUT_DIR = PROJECT_DIR / "outputs"

X = np.hstack([
    np.load(FEATURE_DIR / "X_train_wavlm.npy"),
    np.load(FEATURE_DIR / "X_train_acoustic.npy"),
]).astype(np.float32)

y = np.load(FEATURE_DIR / "y_train.npy")

assert X.shape == (769, 815)
assert y.shape == (769,)
assert np.isfinite(X).all() and np.isfinite(y).all()

# --------------------------------------------------
# 2. Define a limited, justified model search
# --------------------------------------------------
experiments = {
    "ExtraTrees_leaf1": ExtraTreesRegressor(
        n_estimators=400,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=42,
        n_jobs=-1,
    ),
    "ExtraTrees_leaf2": ExtraTreesRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=42,
        n_jobs=-1,
    ),
    "ExtraTrees_leaf3": ExtraTreesRegressor(
        n_estimators=400,
        min_samples_leaf=3,
        max_features=1.0,
        random_state=42,
        n_jobs=-1,
    ),
    "ExtraTrees_leaf5": ExtraTreesRegressor(
        n_estimators=400,
        min_samples_leaf=5,
        max_features=1.0,
        random_state=42,
        n_jobs=-1,
    ),
    "ExtraTrees_half_features": ExtraTreesRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        max_features=0.5,
        random_state=42,
        n_jobs=-1,
    ),
    "RandomForest_leaf2": RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=42,
        n_jobs=-1,
    ),
    "RandomForest_sqrt_features": RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        max_features=0.5,
        random_state=42,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42,
    ),
}

# Fixed folds shared by every candidate
cv = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(X, y))

results = []
oof_store = {}

# --------------------------------------------------
# 3. Evaluate each configuration
# --------------------------------------------------
for model_name, model_template in experiments.items():
    start = time.time()
    oof = np.zeros(len(y), dtype=np.float32)
    fold_rmses = []

    for fold_num, (train_idx, valid_idx) in enumerate(
        folds, start=1
    ):
        # Clone via sklearn to ensure a fresh model per fold
        from sklearn.base import clone
        model = clone(model_template)

        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[valid_idx])
        oof[valid_idx] = pred

        fold_rmse = np.sqrt(
            mean_squared_error(y[valid_idx], pred)
        )
        fold_rmses.append(fold_rmse)

        print(
            f"{model_name} | fold {fold_num}/5 | "
            f"RMSE={fold_rmse:.4f}"
        )

    pooled_corr = np.corrcoef(y, oof)[0, 1]

    results.append({
        "Model": model_name,
        "OOF_RMSE": np.sqrt(mean_squared_error(y, oof)),
        "OOF_MAE": mean_absolute_error(y, oof),
        "OOF_Pearson": pooled_corr,
        "Mean_Fold_RMSE": np.mean(fold_rmses),
        "Fold_RMSE_Std": np.std(fold_rmses),
        "Seconds": time.time() - start,
    })

    oof_store[model_name] = oof

# --------------------------------------------------
# 4. Save metrics and OOF predictions
# --------------------------------------------------
results_df = pd.DataFrame(results).sort_values(
    "OOF_RMSE", ascending=True
)

print("\nMODEL COMPARISON — sorted by pooled OOF RMSE")
print(results_df.round(4).to_string(index=False))

results_df.to_csv(
    OUTPUT_DIR / "model_tuning_results.csv",
    index=False,
)

oof_df = pd.DataFrame({"actual": y})
for model_name, predictions in oof_store.items():
    oof_df[model_name] = predictions

oof_df.to_csv(
    OUTPUT_DIR / "model_tuning_oof_predictions.csv",
    index=False,
)

print("\nSaved metrics and OOF predictions in:", OUTPUT_DIR)


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------------------------------------------------
# 1. Load saved validation results
# --------------------------------------------------
PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

results = pd.read_csv(OUTPUT_DIR / "model_tuning_results.csv")
oof = pd.read_csv(OUTPUT_DIR / "model_tuning_oof_predictions.csv")

actual = oof["actual"].to_numpy(dtype=float)
predicted = oof["HistGradientBoosting"].to_numpy(dtype=float)
residuals = actual - predicted

rmse = np.sqrt(mean_squared_error(actual, predicted))
mae = mean_absolute_error(actual, predicted)
pearson = np.corrcoef(actual, predicted)[0, 1]

print("HistGradientBoosting OOF diagnostics")
print(f"RMSE:    {rmse:.4f}")
print(f"MAE:     {mae:.4f}")
print(f"Pearson: {pearson:.4f}")
print(f"Mean residual (actual - predicted): {residuals.mean():.4f}")
print(f"Residual standard deviation: {residuals.std():.4f}")

# --------------------------------------------------
# 2. Plot RMSE comparison
# --------------------------------------------------
rmse_results = results.sort_values("OOF_RMSE")

plt.figure(figsize=(10, 6))
plt.barh(rmse_results["Model"], rmse_results["OOF_RMSE"])
plt.gca().invert_yaxis()
plt.xlabel("Out-of-fold RMSE (lower is better)")
plt.title("Model Comparison: OOF RMSE")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "model_rmse_comparison.png", dpi=160)
plt.show()

# --------------------------------------------------
# 3. Plot Pearson comparison
# --------------------------------------------------
pearson_results = results.sort_values("OOF_Pearson", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(
    pearson_results["Model"],
    pearson_results["OOF_Pearson"]
)
plt.gca().invert_yaxis()
plt.xlabel("Out-of-fold Pearson correlation (higher is better)")
plt.title("Model Comparison: OOF Pearson Correlation")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "model_pearson_comparison.png", dpi=160
)
plt.show()

# --------------------------------------------------
# 4. Actual versus predicted scores
# --------------------------------------------------
plt.figure(figsize=(7, 7))
plt.scatter(actual, predicted, alpha=0.65)

# Ideal predictions lie on this diagonal
plt.plot([0, 5], [0, 5], linestyle="--")

plt.xlim(0, 5)
plt.ylim(0, 5)
plt.xlabel("Actual grammar score")
plt.ylabel("Out-of-fold predicted score")
plt.title("HistGradientBoosting: Actual vs Predicted")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "hgb_actual_vs_predicted.png", dpi=160
)
plt.show()

# --------------------------------------------------
# 5. Residual distribution
# --------------------------------------------------
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=25, edgecolor="black")
plt.axvline(0, linestyle="--")
plt.xlabel("Residual (actual - predicted)")
plt.ylabel("Number of recordings")
plt.title("HistGradientBoosting: OOF Residual Distribution")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "hgb_residual_distribution.png", dpi=160
)
plt.show()

# --------------------------------------------------
# 6. Residuals versus predicted scores
# --------------------------------------------------
plt.figure(figsize=(8, 5))
plt.scatter(predicted, residuals, alpha=0.65)
plt.axhline(0, linestyle="--")
plt.xlabel("Out-of-fold predicted score")
plt.ylabel("Residual (actual - predicted)")
plt.title("HistGradientBoosting: Residuals vs Predicted")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "hgb_residuals_vs_predicted.png", dpi=160
)
plt.show()

print("\nFigures saved in:", FIGURE_DIR)


In [ ]:

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression

# --------------------------------------------------
# 1. Load combined training features
# --------------------------------------------------
PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
FEATURE_DIR = PROJECT_DIR / "features"
OUTPUT_DIR = PROJECT_DIR / "outputs"

X = np.hstack([
    np.load(FEATURE_DIR / "X_train_wavlm.npy"),
    np.load(FEATURE_DIR / "X_train_acoustic.npy"),
]).astype(np.float32)

y = np.load(FEATURE_DIR / "y_train.npy")

# Same model configuration as the leading experiment
def make_model():
    return HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42,
    )

# --------------------------------------------------
# 2. Nested OOF predictions
# --------------------------------------------------
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
outer_splits = list(outer_cv.split(X, y))

raw_oof = np.zeros(len(y), dtype=np.float32)
calibrated_oof = np.zeros(len(y), dtype=np.float32)

for fold, (train_idx, valid_idx) in enumerate(outer_splits, start=1):
    X_train, X_valid = X[train_idx], X[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    # Generate predictions for the outer training fold using
    # inner OOF predictions, so calibration does not use in-sample fits.
    inner_cv = KFold(n_splits=4, shuffle=True, random_state=100 + fold)
    inner_oof = np.zeros(len(train_idx), dtype=np.float32)

    for inner_train, inner_valid in inner_cv.split(X_train, y_train):
        model = make_model()
        model.fit(X_train[inner_train], y_train[inner_train])
        inner_oof[inner_valid] = model.predict(X_train[inner_valid])

    calibrator = LinearRegression()
    calibrator.fit(inner_oof.reshape(-1, 1), y_train)

    # Fit the main model on the outer training fold
    outer_model = make_model()
    outer_model.fit(X_train, y_train)
    raw_pred = outer_model.predict(X_valid)

    raw_oof[valid_idx] = raw_pred
    calibrated_oof[valid_idx] = calibrator.predict(
        raw_pred.reshape(-1, 1)
    )

    print(f"Completed outer fold {fold}/5")

# --------------------------------------------------
# 3. Compare both approaches
# --------------------------------------------------
def calculate_metrics(name, actual, pred):
    return {
        "Prediction": name,
        "RMSE": np.sqrt(mean_squared_error(actual, pred)),
        "MAE": mean_absolute_error(actual, pred),
        "Pearson": np.corrcoef(actual, pred)[0, 1],
        "Mean_prediction": np.mean(pred),
        "Prediction_std": np.std(pred),
    }

comparison = pd.DataFrame([
    calculate_metrics("Raw HGB", y, raw_oof),
    calculate_metrics("Nested calibrated HGB", y, calibrated_oof),
])

print("\nNESTED OOF CALIBRATION COMPARISON")
print(comparison.round(4).to_string(index=False))

comparison.to_csv(
    OUTPUT_DIR / "calibration_comparison.csv",
    index=False
)

pd.DataFrame({
    "actual": y,
    "raw_oof": raw_oof,
    "calibrated_oof": calibrated_oof,
}).to_csv(
    OUTPUT_DIR / "calibration_oof_predictions.csv",
    index=False
)

print("\nSaved calibration results to:", OUTPUT_DIR)


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("TRAIN CSV")
print("Shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())

print("\nTEST CSV")
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

print("\nSAMPLE SUBMISSION")
print("Shape:", sample_df.shape)
print("Columns:", sample_df.columns.tolist())
print("\nFirst five sample rows:")
print(sample_df.head().to_string(index=False))

# Compare filename sets if both files contain a filename column
if "filename" in test_df.columns and "filename" in sample_df.columns:
    test_names = test_df["filename"].astype(str)
    sample_names = sample_df["filename"].astype(str)

    test_set = set(test_names)
    sample_set = set(sample_names)

    print("\nFILENAME AUDIT")
    print("Test rows:", len(test_names))
    print("Sample submission rows:", len(sample_names))
    print("Unique test filenames:", len(test_set))
    print("Unique sample filenames:", len(sample_set))
    print("Duplicate test filenames:", test_names.duplicated().sum())
    print("Duplicate sample filenames:", sample_names.duplicated().sum())
    print("Test files missing from sample:", len(test_set - sample_set))
    print("Sample files absent from test:", len(sample_set - test_set))

    print("\nExamples missing from sample:")
    print(sorted(test_set - sample_set)[:10])

    print("\nExamples present only in sample:")
    print(sorted(sample_set - test_set)[:10])
else:
    print(
        "\nNo shared 'filename' column found. "
        "We need to inspect the displayed schemas before proceeding."
    )


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"

test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")

test_names = test_df["filename"].astype(str).tolist()
sample_names = sample_df["filename"].astype(str).tolist()

test_set = set(test_names)
sample_set = set(sample_names)

missing_from_sample = sorted(test_set - sample_set)
absent_from_test = sorted(sample_set - test_set)

print("=== EXACT SUBMISSION AUDIT ===")
print("Test rows:", len(test_names))
print("Sample rows:", len(sample_names))
print("Unique test filenames:", len(test_set))
print("Unique sample filenames:", len(sample_set))
print("Duplicate test filenames:", len(test_names) - len(test_set))
print("Duplicate sample filenames:", len(sample_names) - len(sample_set))
print("Test filenames missing from sample:", len(missing_from_sample))
print("Sample filenames absent from test:", len(absent_from_test))

print("\nAll test filenames missing from sample:")
print(missing_from_sample)

print("\nAll sample filenames absent from test:")
print(absent_from_test)

# Verify the WAV files themselves
test_audio_dir = DATA_DIR / "test"
missing_audio = [
    name for name in test_names
    if not (test_audio_dir / name).is_file()
]

print("\nTest WAV files missing on disk:", len(missing_audio))
if missing_audio:
    print(missing_audio)

print("\nFirst 10 test filenames:")
print(test_names[:10])

print("\nFirst 10 sample submission filenames:")
print(sample_names[:10])


In [ ]:

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor

# --------------------------------------------------
# 1. Paths and input files
# --------------------------------------------------
PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"
FEATURE_DIR = PROJECT_DIR / "features"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# --------------------------------------------------
# 2. Load cached features and combine them
# --------------------------------------------------
X_train_wavlm = np.load(FEATURE_DIR / "X_train_wavlm.npy")
X_test_wavlm = np.load(FEATURE_DIR / "X_test_wavlm.npy")

X_train_acoustic = np.load(FEATURE_DIR / "X_train_acoustic.npy")
X_test_acoustic = np.load(FEATURE_DIR / "X_test_acoustic.npy")

y_train = train_df["label"].to_numpy(dtype=np.float32)

X_train = np.hstack([
    X_train_wavlm,
    X_train_acoustic,
]).astype(np.float32)

X_test = np.hstack([
    X_test_wavlm,
    X_test_acoustic,
]).astype(np.float32)

# Integrity checks
assert X_train.shape == (769, 815)
assert X_test.shape == (216, 815)
assert len(y_train) == len(train_df)
assert len(test_df) == len(X_test)
assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()
assert np.isfinite(y_train).all()
assert np.all((y_train >= 0) & (y_train <= 5))
assert not test_df["filename"].duplicated().any()

print("Training matrix:", X_train.shape)
print("Test matrix:", X_test.shape)

# --------------------------------------------------
# 3. Train the selected model on all labeled samples
# --------------------------------------------------
final_model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42,
)

final_model.fit(X_train, y_train)

# --------------------------------------------------
# 4. Predict the 216 test recordings
# --------------------------------------------------
predictions = final_model.predict(X_test)

# Scores are defined on a 0–5 scale
predictions = np.clip(predictions, 0.0, 5.0)

assert predictions.shape == (216,)
assert np.isfinite(predictions).all()

# --------------------------------------------------
# 5. Create candidate submission in test.csv order
# --------------------------------------------------
submission = pd.DataFrame({
    "filename": test_df["filename"].astype(str),
    "label": predictions,
})

assert submission.columns.tolist() == ["filename", "label"]
assert len(submission) == len(test_df)
assert submission["filename"].tolist() == (
    test_df["filename"].astype(str).tolist()
)
assert not submission["filename"].duplicated().any()

submission_path = OUTPUT_DIR / "submission_candidate.csv"
submission.to_csv(submission_path, index=False)

# Save model for reproducibility
model_path = OUTPUT_DIR / "final_hist_gradient_boosting.joblib"
joblib.dump(final_model, model_path)

# --------------------------------------------------
# 6. Report results and saved artifacts
# --------------------------------------------------
print("\nFINAL MODEL TRAINED")
print("Training recordings:", len(y_train))
print("Test predictions:", len(predictions))
print(f"Prediction minimum: {predictions.min():.4f}")
print(f"Prediction maximum: {predictions.max():.4f}")
print(f"Prediction mean:    {predictions.mean():.4f}")
print(f"Prediction std:     {predictions.std():.4f}")

print("\nFirst 10 predictions:")
print(submission.head(10).to_string(index=False))

print("\nCandidate submission saved to:")
print(submission_path)

print("\nTrained model saved to:")
print(model_path)

print("\nCandidate file validated against test.csv.")
print("Do not upload until official submission requirements are confirmed.")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"
OUTPUT_DIR = PROJECT_DIR / "outputs"

test_df = pd.read_csv(DATA_DIR / "test.csv")
submission_path = OUTPUT_DIR / "submission_candidate.csv"
submission = pd.read_csv(submission_path)

assert submission.columns.tolist() == ["filename", "label"]
assert len(submission) == 216
assert submission["filename"].is_unique
assert submission["filename"].tolist() == (
    test_df["filename"].astype(str).tolist()
)
assert submission["label"].notna().all()
assert np.isfinite(submission["label"]).all()
assert submission["label"].between(0, 5).all()

print("FINAL SUBMISSION VALIDATION: PASSED")
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
print("Unique filenames:", submission["filename"].nunique())
print("Prediction range:", (
    round(submission["label"].min(), 4),
    round(submission["label"].max(), 4),
))
print("Predictions equal to 0:", (submission["label"] == 0).sum())
print("Predictions equal to 5:", (submission["label"] == 5).sum())
print("File size (KB):", round(submission_path.stat().st_size / 1024, 2))
print("\nCandidate file:")
print(submission_path)
print("\nFirst 5 rows:")
print(submission.head().to_string(index=False))


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"
OUTPUT_DIR = PROJECT_DIR / "outputs"

candidate = pd.read_csv(OUTPUT_DIR / "submission_candidate.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("Candidate rows:", len(candidate))
print("Sample rows:", len(sample))
print("Test rows:", len(test))

print("\nCandidate prediction statistics:")
print(candidate["label"].describe())

print("\nFirst 10 candidate predictions:")
print(candidate.head(10).to_string(index=False))

print("\nSample labels:")
print(sample["label"].value_counts(dropna=False).to_dict())

print("\nCandidate labels all equal to sample placeholders?",
      bool((candidate["label"] == -1).all()))

print("\nCandidate filenames match test order?",
      candidate["filename"].tolist() == test["filename"].tolist())

print("\nCandidate has unique filenames?",
      candidate["filename"].is_unique)


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
FEATURE_DIR = PROJECT_DIR / "features"
OUTPUT_DIR = PROJECT_DIR / "outputs"

X = np.hstack([
    np.load(FEATURE_DIR / "X_train_wavlm.npy"),
    np.load(FEATURE_DIR / "X_train_acoustic.npy"),
]).astype(np.float32)

y = np.load(FEATURE_DIR / "y_train.npy")

cv = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(y), dtype=np.float32)
fold_results = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    model = ExtraTreesRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X[train_idx], y[train_idx])
    pred = model.predict(X[valid_idx])
    oof[valid_idx] = pred

    fold_rmse = np.sqrt(mean_squared_error(y[valid_idx], pred))
    fold_mae = mean_absolute_error(y[valid_idx], pred)
    fold_pearson = np.corrcoef(y[valid_idx], pred)[0, 1]

    fold_results.append({
        "Fold": fold,
        "RMSE": fold_rmse,
        "MAE": fold_mae,
        "Pearson": fold_pearson,
    })

    print(
        f"Fold {fold}/5 | RMSE={fold_rmse:.4f} | "
        f"MAE={fold_mae:.4f} | Pearson={fold_pearson:.4f}"
    )

overall = {
    "OOF_RMSE": np.sqrt(mean_squared_error(y, oof)),
    "OOF_MAE": mean_absolute_error(y, oof),
    "OOF_Pearson": np.corrcoef(y, oof)[0, 1],
}

print("\nEXTRATREES VALIDATION RESULTS")
print(pd.DataFrame([overall]).round(4).to_string(index=False))

pd.DataFrame(fold_results).to_csv(
    OUTPUT_DIR / "extratrees_revalidation_folds.csv",
    index=False,
)

pd.DataFrame({
    "actual": y,
    "extratrees_oof": oof,
}).to_csv(
    OUTPUT_DIR / "extratrees_revalidation_oof.csv",
    index=False,
)

print("\nSaved revalidation results in:", OUTPUT_DIR)


In [ ]:

from pathlib import Path
import importlib.util

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)
OUTPUT_DIR = PROJECT_DIR / "outputs"

print("=== ASR ENVIRONMENT AUDIT ===")

for package in ["transformers", "torch", "whisper", "faster_whisper"]:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'installed' if installed else 'not installed'}")

print("\n=== EXISTING TRANSCRIPT ARTIFACTS ===")

for folder in [OUTPUT_DIR, OUTPUT_DIR / "transcripts"]:
    print(f"\nFolder: {folder}")
    if not folder.exists():
        print("Not found")
        continue

    matches = [
        p for p in folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".csv", ".json", ".jsonl", ".txt", ".parquet"}
        and any(word in p.name.lower()
                for word in ["transcript", "whisper", "asr"])
    ]

    if matches:
        for path in matches[:30]:
            print(path)
    else:
        print("No transcript-related files found")


In [ ]:

from pathlib import Path
import transformers
import torch

print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

try:
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    print("Whisper classes: available")
except Exception as e:
    print("Whisper classes: unavailable")
    print(type(e).__name__, str(e)[:500])

# Check the default Hugging Face cache for Whisper model files
cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
cached_whisper = []

if cache_dir.exists():
    cached_whisper = [
        p.name for p in cache_dir.iterdir()
        if p.is_dir() and "whisper" in p.name.lower()
    ]

print("Cached Whisper models:", cached_whisper or "None found")


In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import librosa
from pathlib import Path
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Paths
PROJECT_DIR = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA_DIR = PROJECT_DIR / "data" / "Dataset_Final"

train_df = pd.read_csv(DATA_DIR / "train.csv")

# Use five training files only for this pilot
pilot_df = train_df.head(5).copy()

model_name = "openai/whisper-tiny.en"
device = "cpu"

print("Loading cached Whisper model...")

processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)
model.to(device)
model.eval()

transcripts = []
start_time = time.time()

for i, row in enumerate(pilot_df.itertuples(index=False), start=1):
    audio_path = DATA_DIR / "train" / row.filename

    # Adjust this path only if your WAV files are stored elsewhere
    if not audio_path.exists():
        audio_path = DATA_DIR / row.filename

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {row.filename}")

    audio, sr = librosa.load(audio_path, sr=16000, mono=True)

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.inference_mode():
        generated_ids = model.generate(
            inputs.input_features,
            max_new_tokens=225
        )

    transcript = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()

    transcripts.append({
        "filename": row.filename,
        "true_score": row.label,
        "transcript": transcript,
        "audio_seconds": len(audio) / sr
    })

    print(f"\n{i}/5 | {row.filename} | score={row.label}")
    print(transcript if transcript else "[EMPTY TRANSCRIPT]")

elapsed = time.time() - start_time

pilot_results = pd.DataFrame(transcripts)

print("\n--- PILOT SUMMARY ---")
print(f"Files transcribed: {len(pilot_results)}")
print(f"Total time: {elapsed / 60:.2f} minutes")
print(f"Average time per file: {elapsed / len(pilot_results):.1f} seconds")
print(f"Total audio duration: {pilot_results['audio_seconds'].sum() / 60:.2f} minutes")

In [ ]:
import re
import numpy as np
import pandas as pd

def extract_text_features(text):
    text = str(text).lower().strip()

    # Tokenize words while retaining contractions
    words = re.findall(r"\b[a-z]+(?:'[a-z]+)?\b", text)
    word_count = len(words)

    sentences = [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

    sentence_count = len(sentences)
    unique_words = len(set(words))

    # Adjacent repeated words, e.g. "the the"
    repeated_adjacent = sum(
        words[i] == words[i - 1]
        for i in range(1, word_count)
    )

    fillers = {
        "um", "uh", "erm", "hmm", "uhh"
    }
    filler_count = sum(word in fillers for word in words)

    # Repeated word rate and vocabulary diversity
    repeated_word_rate = (
        repeated_adjacent / max(word_count - 1, 1)
    )
    vocabulary_diversity = (
        unique_words / max(word_count, 1)
    )

    # Transcript may end mid-sentence because of generation limits
    incomplete_ending = int(
        bool(words) and text[-1:] not in ".!?"
    )

    return {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_words_per_sentence": (
            word_count / max(sentence_count, 1)
        ),
        "unique_word_count": unique_words,
        "vocabulary_diversity": vocabulary_diversity,
        "repeated_adjacent_count": repeated_adjacent,
        "repeated_word_rate": repeated_word_rate,
        "filler_count": filler_count,
        "incomplete_ending": incomplete_ending,
    }

pilot_text_features = pd.DataFrame(
    [
        extract_text_features(text)
        for text in pilot_results["transcript"]
    ]
)

print("Pilot linguistic features:")
display(
    pd.concat(
        [
            pilot_results[["filename", "true_score"]],
            pilot_text_features
        ],
        axis=1
    )
)

print("\nFeature matrix shape:", pilot_text_features.shape)
print("Missing values:", int(pilot_text_features.isna().sum().sum()))

In [ ]:
import json
import time
import re
from pathlib import Path

import pandas as pd
import librosa
import torch

# Output locations
TRANSCRIPT_DIR = PROJECT_DIR / "outputs" / "transcripts"
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

TRANSCRIPT_CSV = TRANSCRIPT_DIR / "train_transcripts.csv"
FEATURE_CSV = TRANSCRIPT_DIR / "train_linguistic_features.csv"
PROGRESS_JSON = TRANSCRIPT_DIR / "train_transcription_progress.json"

# Reuse the five pilot transcripts; resume from saved progress if available
if TRANSCRIPT_CSV.exists():
    saved_df = pd.read_csv(TRANSCRIPT_CSV)
else:
    saved_df = pilot_results[["filename", "true_score", "transcript", "audio_seconds"]].copy()

transcript_records = saved_df.to_dict("records")
completed_files = {r["filename"] for r in transcript_records}

print(f"Already transcribed: {len(completed_files)}/{len(train_df)}")

start_time = time.time()

for row in train_df.itertuples(index=False):
    filename = row.filename

    if filename in completed_files:
        continue

    audio_path = DATA_DIR / "train" / filename
    if not audio_path.exists():
        audio_path = DATA_DIR / filename
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {filename}")

    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")

    with torch.inference_mode():
        generated_ids = model.generate(
            inputs.input_features,
            max_new_tokens=225
        )

    transcript = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()

    transcript_records.append({
        "filename": filename,
        "true_score": float(row.label),
        "transcript": transcript,
        "audio_seconds": len(audio) / sr
    })
    completed_files.add(filename)

    if len(completed_files) % 25 == 0:
        pd.DataFrame(transcript_records).to_csv(TRANSCRIPT_CSV, index=False)
        print(f"Progress: {len(completed_files)}/{len(train_df)}")

# Save all transcripts
transcript_df = pd.DataFrame(transcript_records)
transcript_df.to_csv(TRANSCRIPT_CSV, index=False)

# Extract linguistic features using the function from Step 25
linguistic_df = pd.DataFrame([
    extract_text_features(text)
    for text in transcript_df["transcript"]
])
linguistic_df.insert(0, "filename", transcript_df["filename"].values)
linguistic_df.to_csv(FEATURE_CSV, index=False)

elapsed = time.time() - start_time

print("\n--- TRAIN TRANSCRIPTION COMPLETE ---")
print("Transcript rows:", len(transcript_df))
print("Unique filenames:", transcript_df["filename"].nunique())
print("Empty transcripts:", int(transcript_df["transcript"].fillna("").str.strip().eq("").sum()))
print(f"Elapsed time: {elapsed / 60:.1f} minutes")
print("Transcript file:", TRANSCRIPT_CSV)
print("Feature file:", FEATURE_CSV)
print("Feature shape:", linguistic_df.shape)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

# Load the existing, previously extracted audio features
FEATURE_DIR = PROJECT_DIR / "features"
X_wavlm = np.load(FEATURE_DIR / "X_train_wavlm.npy")
X_acoustic = np.load(FEATURE_DIR / "X_train_acoustic.npy")
y = np.load(FEATURE_DIR / "y_train.npy")

X_audio = np.hstack([X_wavlm, X_acoustic])

# Align linguistic features by filename, never by assumed row order
ling_df = pd.read_csv(FEATURE_DIR.parent / "outputs" / "transcripts" / "train_linguistic_features.csv")
train_csv = pd.read_csv(DATA_DIR / "train.csv")

ling_columns = [c for c in ling_df.columns if c != "filename"]

assert len(train_csv) == len(y) == len(X_audio) == 769
assert train_csv["filename"].is_unique
assert ling_df["filename"].is_unique
assert set(train_csv["filename"]) == set(ling_df["filename"])

ling_df = (
    train_csv[["filename"]]
    .merge(ling_df, on="filename", how="left", validate="one_to_one")
)

X_linguistic = ling_df[ling_columns].to_numpy(dtype=np.float64)
X_experiment = np.hstack([X_audio, X_linguistic])

assert np.isfinite(X_audio).all()
assert np.isfinite(X_linguistic).all()
assert X_experiment.shape == (769, 824)

# Same folds and model configuration for a fair comparison
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_hgb(X, y, cv):
    model = HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42
    )

    oof = cross_val_predict(model, X, y, cv=cv, n_jobs=1)

    return {
        "RMSE": np.sqrt(mean_squared_error(y, oof)),
        "MAE": mean_absolute_error(y, oof),
        "Pearson": pearsonr(y, oof).statistic
    }, oof

baseline_metrics, baseline_oof = evaluate_hgb(X_audio, y, cv)
ling_metrics, ling_oof = evaluate_hgb(X_experiment, y, cv)

comparison = pd.DataFrame(
    [baseline_metrics, ling_metrics],
    index=["Audio only", "Audio + linguistic"]
)

comparison["RMSE improvement"] = (
    baseline_metrics["RMSE"] - comparison["RMSE"]
)
comparison["MAE improvement"] = (
    baseline_metrics["MAE"] - comparison["MAE"]
)
comparison["Pearson improvement"] = (
    comparison["Pearson"] - baseline_metrics["Pearson"]
)

print("Feature shapes:")
print("Audio only:", X_audio.shape)
print("Audio + linguistic:", X_experiment.shape)

print("\nFive-fold out-of-fold comparison:")
display(comparison.round(4))

print("\nInterpretation:")
if ling_metrics["RMSE"] < baseline_metrics["RMSE"] and ling_metrics["Pearson"] > baseline_metrics["Pearson"]:
    print("Linguistic features improve both RMSE and Pearson on these folds.")
elif ling_metrics["RMSE"] < baseline_metrics["RMSE"] or ling_metrics["Pearson"] > baseline_metrics["Pearson"]:
    print("Results are mixed: inspect both metrics before deciding.")
else:
    print("Linguistic features do not improve either metric on these folds.")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr
from sklearn.model_selection import KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def make_model():
    return HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42
    )

fold_results = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_audio), start=1):
    for name, X in [
        ("Audio only", X_audio),
        ("Audio + linguistic", X_experiment)
    ]:
        model = make_model()
        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[valid_idx])

        fold_results.append({
            "Fold": fold,
            "Features": name,
            "RMSE": np.sqrt(mean_squared_error(y[valid_idx], pred)),
            "MAE": mean_absolute_error(y[valid_idx], pred),
            "Pearson": pearsonr(y[valid_idx], pred).statistic
        })

fold_df = pd.DataFrame(fold_results)

fold_pivot = fold_df.pivot(
    index="Fold",
    columns="Features",
    values=["RMSE", "MAE", "Pearson"]
)

display(fold_df.round(4))

print("\nPer-fold RMSE improvement (positive means linguistic features helped):")
rmse_wide = fold_df.pivot(index="Fold", columns="Features", values="RMSE")
rmse_improvement = rmse_wide["Audio only"] - rmse_wide["Audio + linguistic"]
display(rmse_improvement.rename("RMSE improvement").round(4).to_frame())

print(
    f"\nLinguistic features improved RMSE in "
    f"{int((rmse_improvement > 0).sum())}/5 folds."
)

In [ ]:
import numpy as np
import pandas as pd

# fold_df was created in Step 28
summary = (
    fold_df
    .groupby("Features")[["RMSE", "MAE", "Pearson"]]
    .agg(["mean", "std"])
    .round(4)
)

print("Mean and standard deviation across the five validation folds:")
display(summary)

rmse_wide = fold_df.pivot(
    index="Fold",
    columns="Features",
    values="RMSE"
)

mae_wide = fold_df.pivot(
    index="Fold",
    columns="Features",
    values="MAE"
)

pearson_wide = fold_df.pivot(
    index="Fold",
    columns="Features",
    values="Pearson"
)

improvements = pd.DataFrame({
    "RMSE improvement": (
        rmse_wide["Audio only"] -
        rmse_wide["Audio + linguistic"]
    ),
    "MAE improvement": (
        mae_wide["Audio only"] -
        mae_wide["Audio + linguistic"]
    ),
    "Pearson improvement": (
        pearson_wide["Audio + linguistic"] -
        pearson_wide["Audio only"]
    )
})

print("\nFold-by-fold improvements (positive is better):")
display(improvements.round(4))

print("\nAverage improvement:")
display(improvements.mean().round(4).to_frame("Mean improvement"))

In [ ]:
from pathlib import Path
import pandas as pd

test_df = pd.read_csv(DATA_DIR / "test.csv")

test_audio_dir = DATA_DIR / "test"

if not test_audio_dir.exists():
    test_audio_dir = DATA_DIR

missing_files = [
    filename for filename in test_df["filename"]
    if not (test_audio_dir / filename).exists()
]

print("Test CSV rows:", len(test_df))
print("Unique test filenames:", test_df["filename"].nunique())
print("Test audio directory:", test_audio_dir)
print("Missing audio files:", len(missing_files))

if missing_files:
    print("First missing filenames:", missing_files[:10])
else:
    print("All test audio files are present.")

test_transcript_path = (
    PROJECT_DIR / "outputs" / "transcripts" / "test_transcripts.csv"
)

print(
    "Existing test transcript file:",
    test_transcript_path.exists()
)

In [ ]:
print("Current model type:", type(model).__name__)

print(
    "Whisper model variable exists:",
    "whisper_model" in globals()
)

print(
    "Whisper processor exists:",
    "processor" in globals()
)

if "whisper_model" in globals():
    print("Whisper model type:", type(whisper_model).__name__)

In [ ]:
from transformers import WhisperForConditionalGeneration

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-tiny.en",
    local_files_only=True
)

whisper_model.to("cpu")
whisper_model.eval()

print("Whisper model restored:", type(whisper_model).__name__)
print("Device: CPU")
print("Processor:", type(processor).__name__)

In [ ]:
import time
import pandas as pd
import librosa
import torch

TEST_TRANSCRIPT_CSV = TRANSCRIPT_DIR / "test_transcripts.csv"
TEST_FEATURE_CSV = TRANSCRIPT_DIR / "test_linguistic_features.csv"

test_df = pd.read_csv(DATA_DIR / "test.csv")

if TEST_TRANSCRIPT_CSV.exists():
    saved_test_df = pd.read_csv(TEST_TRANSCRIPT_CSV)
else:
    saved_test_df = pd.DataFrame(
        columns=["filename", "transcript", "audio_seconds"]
    )

test_records = saved_test_df.to_dict("records")
completed_test_files = {r["filename"] for r in test_records}

# Reject unexpected filenames in a stale or partial cache
valid_test_files = set(test_df["filename"])
test_records = [
    r for r in test_records if r["filename"] in valid_test_files
]
completed_test_files = {r["filename"] for r in test_records}

print(f"Already transcribed: {len(completed_test_files)}/{len(test_df)}")

start_time = time.time()

for row in test_df.itertuples(index=False):
    filename = row.filename

    if filename in completed_test_files:
        continue

    audio_path = test_audio_dir / filename
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")

    with torch.inference_mode():
        generated_ids = whisper_model.generate(
            inputs.input_features,
            max_new_tokens=225
        )

    transcript = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()

    test_records.append({
        "filename": filename,
        "transcript": transcript,
        "audio_seconds": len(audio) / sr
    })
    completed_test_files.add(filename)

    if len(completed_test_files) % 25 == 0:
        pd.DataFrame(test_records).to_csv(
            TEST_TRANSCRIPT_CSV, index=False
        )
        print(f"Progress: {len(completed_test_files)}/{len(test_df)}")

test_transcript_df = pd.DataFrame(test_records)
test_transcript_df.to_csv(TEST_TRANSCRIPT_CSV, index=False)

test_linguistic_df = pd.DataFrame([
    extract_text_features(text)
    for text in test_transcript_df["transcript"]
])
test_linguistic_df.insert(
    0, "filename", test_transcript_df["filename"].values
)
test_linguistic_df.to_csv(TEST_FEATURE_CSV, index=False)

elapsed = time.time() - start_time

assert len(test_transcript_df) == 216
assert test_transcript_df["filename"].nunique() == 216
assert set(test_transcript_df["filename"]) == set(test_df["filename"])
assert test_linguistic_df.shape == (216, 10)

print("\n--- TEST TRANSCRIPTION COMPLETE ---")
print("Transcript rows:", len(test_transcript_df))
print(
    "Empty transcripts:",
    int(test_transcript_df["transcript"].fillna("").str.strip().eq("").sum())
)
print(f"Elapsed time: {elapsed / 60:.1f} minutes")
print("Transcript file:", TEST_TRANSCRIPT_CSV)
print("Feature file:", TEST_FEATURE_CSV)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

FEATURE_DIR = PROJECT_DIR / "features"
TRANSCRIPT_DIR = PROJECT_DIR / "outputs" / "transcripts"

# Load cached audio features
X_train_wavlm = np.load(FEATURE_DIR / "X_train_wavlm.npy")
X_test_wavlm = np.load(FEATURE_DIR / "X_test_wavlm.npy")
X_train_acoustic = np.load(FEATURE_DIR / "X_train_acoustic.npy")
X_test_acoustic = np.load(FEATURE_DIR / "X_test_acoustic.npy")
y_train = np.load(FEATURE_DIR / "y_train.npy")

# Load linguistic features
train_ling = pd.read_csv(
    TRANSCRIPT_DIR / "train_linguistic_features.csv"
)
test_ling = pd.read_csv(
    TRANSCRIPT_DIR / "test_linguistic_features.csv"
)

train_csv = pd.read_csv(DATA_DIR / "train.csv")
test_csv = pd.read_csv(DATA_DIR / "test.csv")

ling_columns = [c for c in train_ling.columns if c != "filename"]

# Align by the original CSV filename order
train_ling = train_csv[["filename"]].merge(
    train_ling, on="filename", how="left", validate="one_to_one"
)
test_ling = test_csv[["filename"]].merge(
    test_ling, on="filename", how="left", validate="one_to_one"
)

X_train_ling = train_ling[ling_columns].to_numpy(dtype=np.float64)
X_test_ling = test_ling[ling_columns].to_numpy(dtype=np.float64)

# Combine features
X_train_final = np.hstack([
    X_train_wavlm,
    X_train_acoustic,
    X_train_ling
])
X_test_final = np.hstack([
    X_test_wavlm,
    X_test_acoustic,
    X_test_ling
])

# Strict validation
assert X_train_final.shape == (769, 824)
assert X_test_final.shape == (216, 824)
assert len(y_train) == 769
assert len(ling_columns) == 9
assert np.isfinite(X_train_final).all()
assert np.isfinite(X_test_final).all()
assert np.isfinite(y_train).all()
assert set(train_csv["filename"]) == set(train_ling["filename"])
assert set(test_csv["filename"]) == set(test_ling["filename"])

print("Training feature matrix:", X_train_final.shape)
print("Test feature matrix:", X_test_final.shape)
print("Linguistic feature columns:", ling_columns)
print("Missing training linguistic values:", int(np.isnan(X_train_ling).sum()))
print("Missing test linguistic values:", int(np.isnan(X_test_ling).sum()))
print("Training target range:", float(y_train.min()), "to", float(y_train.max()))
print("All feature and filename alignment checks passed.")

In [ ]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from scipy.stats import pearsonr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Train on all labeled samples
final_model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

final_model.fit(X_train_final, y_train)

# Training metrics (in-sample; not a generalization estimate)
train_predictions = final_model.predict(X_train_final)

training_rmse = np.sqrt(
    mean_squared_error(y_train, train_predictions)
)
training_mae = mean_absolute_error(
    y_train, train_predictions
)
training_pearson = pearsonr(
    y_train, train_predictions
).statistic

# Predict test samples
test_predictions = final_model.predict(X_test_final)

# Save model and predictions
MODEL_PATH = OUTPUT_DIR / "final_hgb_audio_linguistic.joblib"
PREDICTION_PATH = OUTPUT_DIR / "test_predictions.csv"

joblib.dump(final_model, MODEL_PATH)

pred_df = pd.DataFrame({
    "filename": test_csv["filename"],
    "prediction": test_predictions
})
pred_df.to_csv(PREDICTION_PATH, index=False)

print("--- TRAINING METRICS (IN-SAMPLE) ---")
print(f"Training RMSE: {training_rmse:.4f}")
print(f"Training MAE: {training_mae:.4f}")
print(f"Training Pearson: {training_pearson:.4f}")

print("\n--- TEST PREDICTIONS ---")
print("Prediction count:", len(test_predictions))
print(f"Minimum: {test_predictions.min():.4f}")
print(f"Maximum: {test_predictions.max():.4f}")
print(f"Mean: {test_predictions.mean():.4f}")
print(f"Standard deviation: {test_predictions.std():.4f}")

print("\nSaved model:", MODEL_PATH)
print("Saved predictions:", PREDICTION_PATH)

In [ ]:
import numpy as np
import pandas as pd

SUBMISSION_PATH = PROJECT_DIR / "outputs" / "submission_audio_linguistic.csv"

# Keep predictions within the valid score range
submission = pd.DataFrame({
    "filename": test_csv["filename"].values,
    "label": np.clip(test_predictions, 0.0, 5.0)
})

# Validate submission integrity
assert submission.shape == (216, 2)
assert submission.columns.tolist() == ["filename", "label"]
assert submission["filename"].is_unique
assert submission["filename"].tolist() == test_csv["filename"].tolist()
assert submission["label"].notna().all()
assert np.isfinite(submission["label"].to_numpy()).all()
assert submission["label"].between(0.0, 5.0).all()

submission.to_csv(SUBMISSION_PATH, index=False)

print("--- SUBMISSION VALIDATION PASSED ---")
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
print("Unique filenames:", submission["filename"].nunique())
print("Prediction range:", submission["label"].min(), "to", submission["label"].max())
print("Missing values:", int(submission.isna().sum().sum()))
print("Saved to:", SUBMISSION_PATH)

display(submission.head(10))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

# Audio-only features: 768 WavLM + 47 acoustic
X_audio = np.hstack([X_train_wavlm, X_train_acoustic])
X_test_audio = np.hstack([X_test_wavlm, X_test_acoustic])

assert X_audio.shape == (769, 815)
assert X_test_audio.shape == (216, 815)

# Reproduce the recorded ExtraTrees configuration
et_model = ExtraTreesRegressor(
    n_estimators=400,
    min_samples_leaf=2,
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)

# Five-fold out-of-fold evaluation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
oof_pred = cross_val_predict(
    et_model, X_audio, y_train, cv=cv, n_jobs=1
)

oof_rmse = np.sqrt(mean_squared_error(y_train, oof_pred))
oof_mae = mean_absolute_error(y_train, oof_pred)
oof_pearson = pearsonr(y_train, oof_pred).statistic

print("--- EXTRATREES FIVE-FOLD VALIDATION ---")
print(f"OOF RMSE:   {oof_rmse:.4f}")
print(f"OOF MAE:    {oof_mae:.4f}")
print(f"OOF Pearson:{oof_pearson:.4f}")

# Fit on all labeled training samples and predict test
et_model.fit(X_audio, y_train)
et_test_pred = et_model.predict(X_test_audio)

# Save a separate candidate; do not overwrite previous submissions
et_candidate_path = (
    PROJECT_DIR / "outputs" / "submission_extratrees_audio.csv"
)

et_submission = pd.DataFrame({
    "filename": test_csv["filename"].values,
    "label": np.clip(et_test_pred, 0.0, 5.0)
})

assert len(et_submission) == 216
assert et_submission["filename"].is_unique
assert et_submission["filename"].tolist() == test_csv["filename"].tolist()
assert et_submission["label"].between(0, 5).all()
assert et_submission["label"].notna().all()

et_submission.to_csv(et_candidate_path, index=False)

print("\n--- EXTRA TREES CANDIDATE ---")
print("Prediction mean:", round(et_submission["label"].mean(), 4))
print("Prediction std:", round(et_submission["label"].std(), 4))
print("Saved:", et_candidate_path)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment"
)

search_dirs = [
    PROJECT_DIR / "outputs",
    PROJECT_DIR / "features",
]

keywords = ("submission", "extratrees", "extra_trees", "prediction", "model")

print("--- EXISTING CANDIDATE AND MODEL FILES ---")

for folder in search_dirs:
    if not folder.exists():
        continue

    for path in sorted(folder.rglob("*")):
        if path.is_file() and any(
            word in path.name.lower() for word in keywords
        ):
            print(path.relative_to(PROJECT_DIR))

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = PROJECT_DIR / "outputs"

files_to_inspect = [
    "extratrees_revalidation_folds.csv",
    "extratrees_revalidation_oof.csv",
    "model_tuning_results.csv",
    "submission_candidate.csv",
]

for filename in files_to_inspect:
    path = OUTPUT_DIR / filename

    print(f"\n{'=' * 60}")
    print(filename)

    if not path.exists():
        print("File not found.")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(10))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = PROJECT_DIR / "outputs"

files = {
    "Previous candidate": OUTPUT_DIR / "submission_candidate.csv",
    "Audio + linguistic": OUTPUT_DIR / "submission_audio_linguistic.csv",
    "ExtraTrees audio": OUTPUT_DIR / "submission_extratrees_audio.csv",
}

predictions = {}

for name, path in files.items():
    if not path.exists():
        print(f"Missing: {path}")
        continue

    df = pd.read_csv(path)
    assert df["filename"].is_unique
    predictions[name] = df.set_index("filename")["label"]

pred_df = pd.DataFrame(predictions).reindex(test_csv["filename"])

print("--- PREDICTION DISTRIBUTIONS ---")
display(
    pred_df.agg(["mean", "std", "min", "max"]).round(4)
)

print("\n--- CORRELATION BETWEEN SUBMISSION PREDICTIONS ---")
display(pred_df.corr().round(4))

print("\n--- MEAN ABSOLUTE PREDICTION DIFFERENCES ---")
diffs = {}
names = list(pred_df.columns)

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        diffs[f"{a} vs {b}"] = np.mean(
            np.abs(pred_df[a] - pred_df[b])
        )

display(pd.Series(diffs, name="Mean absolute difference").round(4))

In [ ]:
from pathlib import Path

# Project directory
PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")

print("=" * 75)
print("PROJECT DIRECTORY CHECK")
print("=" * 75)
print("Project exists:", PROJECT.exists())
print("Project path:", PROJECT)

if not PROJECT.exists():
    print("\nERROR: Project folder not found. Check the path above.")
else:
    # Inspect important directories
    for folder_name in ["data", "features", "outputs", "src", "notebooks"]:
        folder = PROJECT / folder_name
        print(f"\n{'-' * 60}")
        print(f"FOLDER: {folder_name}")
        print("Exists:", folder.exists())

        if folder.exists():
            files = sorted(folder.rglob("*"))
            files = [p for p in files if p.is_file()]

            print("File count:", len(files))

            for file in files[:50]:
                try:
                    size_mb = file.stat().st_size / (1024 * 1024)
                    print(f"  {file.relative_to(PROJECT)}  ({size_mb:.2f} MB)")
                except OSError:
                    print(f"  {file.relative_to(PROJECT)}")

            if len(files) > 50:
                print(f"  ... and {len(files) - 50} more files")

    # Check dataset CSV files
    print(f"\n{'-' * 60}")
    print("DATASET CSV CHECK")

    for csv_path in PROJECT.rglob("*.csv"):
        try:
            print(f"  {csv_path.relative_to(PROJECT)}")
        except ValueError:
            pass

print("\nInspection complete. No files were changed.")

In [ ]:
from pathlib import Path

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")

print("=" * 70)
print("ALL IMPORTANT ARTIFACTS")
print("=" * 70)

# Search by filename, regardless of capitalization or subfolder
keywords = [
    "wavlm", "acoustic", "prosody", "linguistic",
    "transcript", "oof", "prediction", "submission",
    "train.csv", "test.csv", "sample_submission",
    "joblib", "checkpoint"
]

for folder_name in ["features", "outputs", "data"]:
    folder = PROJECT / folder_name
    print(f"\n--- {folder_name.upper()} ---")

    if not folder.exists():
        print("Folder not found")
        continue

    matches = []
    for path in folder.rglob("*"):
        if path.is_file():
            name = path.name.lower()
            if any(k in name for k in keywords):
                matches.append(path)

    if matches:
        for path in sorted(matches):
            size_mb = path.stat().st_size / (1024 * 1024)
            print(f"{path.relative_to(PROJECT)} | {size_mb:.2f} MB")
    else:
        print("No matching files found")

print("\n--- DATASET CSV FILES ---")
for path in (PROJECT / "data").rglob("*.csv"):
    print(path.relative_to(PROJECT))

print("\n--- NUMPY FILES ---")
for path in PROJECT.rglob("*.npy"):
    print(path.relative_to(PROJECT))

print("\nInspection complete. No files were changed.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
OUTPUTS = PROJECT / "outputs"

files_to_check = [
    "oof_predictions.csv",
    "model_tuning_oof_predictions.csv",
    "extratrees_revalidation_oof.csv",
    "calibration_oof_predictions.csv",
    "test_predictions.csv",
    "submission_candidate.csv",
    "submission_audio_linguistic.csv",
    "submission_extratrees_audio.csv",
]

print("=" * 80)
print("EXISTING PREDICTIONS AND VALIDATION FILES")
print("=" * 80)

for filename in files_to_check:
    path = OUTPUTS / filename
    print(f"\n{'-' * 70}")
    print("FILE:", filename)

    if not path.exists():
        print("NOT FOUND")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

    # Show numerical prediction statistics
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    if numeric_cols:
        print("\nNumeric column statistics:")
        print(df[numeric_cols].describe().round(4).to_string())

print("\nInspection complete. No files were changed.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
OUTPUTS = PROJECT / "outputs"

oof_files = [
    "oof_predictions.csv",
    "model_tuning_oof_predictions.csv",
    "extratrees_revalidation_oof.csv",
    "calibration_oof_predictions.csv",
]

print("=" * 90)
print("OUT-OF-FOLD VALIDATION METRICS")
print("=" * 90)

for filename in oof_files:
    path = OUTPUTS / filename
    if not path.exists():
        continue

    df = pd.read_csv(path)
    print(f"\n--- {filename} | shape={df.shape} ---")
    print("Columns:", df.columns.tolist())

    # Identify the true target column
    target_col = next(
        (c for c in df.columns if c.lower() in ["actual", "y_true", "true", "label"]),
        None
    )

    if target_col is None:
        print("Target column not identified; skipping metrics.")
        continue

    y_true = pd.to_numeric(df[target_col], errors="coerce")

    for col in df.columns:
        if col == target_col:
            continue

        y_pred = pd.to_numeric(df[col], errors="coerce")
        valid = y_true.notna() & y_pred.notna()

        if valid.sum() < 10:
            continue

        actual = y_true[valid].to_numpy()
        pred = y_pred[valid].to_numpy()

        if np.std(pred) == 0:
            pearson = np.nan
        else:
            pearson = np.corrcoef(actual, pred)[0, 1]

        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)

        print(
            f"{col:35s} | n={valid.sum():3d} | "
            f"RMSE={rmse:.4f} | MAE={mae:.4f} | Pearson={pearson:.4f}"
        )

print("\n" + "=" * 90)
print("SAVED SUBMISSION PREDICTION DISTRIBUTIONS")
print("=" * 90)

for filename in [
    "submission_candidate.csv",
    "submission_audio_linguistic.csv",
    "submission_extratrees_audio.csv",
]:
    path = OUTPUTS / filename
    if not path.exists():
        continue

    df = pd.read_csv(path)
    print(f"\n--- {filename} ---")
    print("Shape:", df.shape, "| Columns:", df.columns.tolist())

    # Find likely prediction column
    candidates = [
        c for c in df.columns
        if c.lower() in ["label", "prediction", "predictions", "score"]
    ]

    if not candidates:
        print("Prediction column not identified.")
        continue

    col = candidates[0]
    values = pd.to_numeric(df[col], errors="coerce").dropna()

    print(
        f"Column={col} | count={len(values)} | "
        f"mean={values.mean():.4f} | std={values.std():.4f} | "
        f"min={values.min():.4f} | max={values.max():.4f}"
    )
    print("Predictions outside [0, 5]:", int(((values < 0) | (values > 5)).sum()))

print("\nComplete. No files were changed.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
OUTPUTS = PROJECT / "outputs"

for filename in [
    "extratrees_revalidation_oof.csv",
    "calibration_oof_predictions.csv",
]:
    path = OUTPUTS / filename

    print(f"\n{'=' * 75}")
    print(filename)

    if not path.exists():
        print("File not found")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(5).to_string(index=False))

    if "actual" not in df.columns:
        print("No 'actual' column; cannot calculate metrics automatically.")
        continue

    y = pd.to_numeric(df["actual"], errors="coerce")

    for col in df.columns:
        if col == "actual":
            continue

        p = pd.to_numeric(df[col], errors="coerce")
        valid = y.notna() & p.notna()

        if valid.sum() < 10:
            continue

        yt, yp = y[valid].to_numpy(), p[valid].to_numpy()
        r = np.corrcoef(yt, yp)[0, 1] if np.std(yp) > 0 else np.nan

        print(
            f"{col}: RMSE={np.sqrt(mean_squared_error(yt, yp)):.4f}, "
            f"MAE={mean_absolute_error(yt, yp):.4f}, Pearson={r:.4f}"
        )

print("\nNo files were modified.")

In [ ]:
from pathlib import Path
import numpy as np
import joblib

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")

print("=" * 70)
print("FEATURE SHAPES")
print("=" * 70)

for filename in [
    "X_train_wavlm.npy",
    "X_test_wavlm.npy",
    "X_train_acoustic.npy",
    "X_test_acoustic.npy",
]:
    path = PROJECT / "features" / filename
    if path.exists():
        arr = np.load(path, mmap_mode="r")
        print(f"{filename}: shape={arr.shape}, dtype={arr.dtype}")
    else:
        print(f"{filename}: NOT FOUND")

print("\n" + "=" * 70)
print("LINGUISTIC FEATURE SHAPES")
print("=" * 70)

for filename in [
    "train_linguistic_features.csv",
    "test_linguistic_features.csv",
]:
    path = PROJECT / "outputs" / "transcripts" / filename
    if path.exists():
        import pandas as pd
        df = pd.read_csv(path)
        print(f"{filename}: shape={df.shape}")
        print("Columns:", df.columns.tolist())
    else:
        print(f"{filename}: NOT FOUND")

print("\n" + "=" * 70)
print("SAVED MODEL EXPECTED FEATURES")
print("=" * 70)

for filename in [
    "final_hist_gradient_boosting.joblib",
    "final_hgb_audio_linguistic.joblib",
]:
    path = PROJECT / "outputs" / filename
    if not path.exists():
        print(f"{filename}: NOT FOUND")
        continue

    obj = joblib.load(path)
    print(f"\n{filename}")
    print("Model type:", type(obj).__name__)
    print("Expected input features:", getattr(obj, "n_features_in_", "unknown"))

print("\nInspection complete. No files were changed.")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------------------------------------------------
# 1. Set paths
# --------------------------------------------------
PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = PROJECT / "data" / "Dataset_Final"
FEATURES = PROJECT / "features"
OUTPUTS = PROJECT / "outputs"
TRANSCRIPTS = OUTPUTS / "transcripts"

# --------------------------------------------------
# 2. Load training labels and existing audio features
# --------------------------------------------------
train_df = pd.read_csv(DATA / "train.csv")

X_wavlm = np.load(FEATURES / "X_train_wavlm.npy")
X_acoustic = np.load(FEATURES / "X_train_acoustic.npy")
y = train_df["label"].to_numpy(dtype=np.float64)

assert len(y) == len(X_wavlm) == len(X_acoustic), (
    f"Row mismatch: labels={len(y)}, "
    f"WavLM={len(X_wavlm)}, acoustic={len(X_acoustic)}"
)

X_audio = np.hstack([X_wavlm, X_acoustic])

# --------------------------------------------------
# 3. Align linguistic features to train.csv filenames
# --------------------------------------------------
ling_df = pd.read_csv(TRANSCRIPTS / "train_linguistic_features.csv")

if "filename" not in train_df.columns or "filename" not in ling_df.columns:
    raise ValueError("Both training CSVs must contain a filename column.")

if train_df["filename"].duplicated().any():
    raise ValueError("Duplicate filenames in train.csv.")

if ling_df["filename"].duplicated().any():
    raise ValueError("Duplicate filenames in linguistic features.")

ling_cols = [
    "word_count",
    "sentence_count",
    "avg_words_per_sentence",
    "unique_word_count",
    "vocabulary_diversity",
    "repeated_adjacent_count",
    "repeated_word_rate",
    "filler_count",
    "incomplete_ending",
]

missing_cols = set(ling_cols) - set(ling_df.columns)
if missing_cols:
    raise ValueError(f"Missing linguistic columns: {sorted(missing_cols)}")

ling_aligned = train_df[["filename"]].merge(
    ling_df[["filename"] + ling_cols],
    on="filename",
    how="left",
    validate="one_to_one",
    sort=False,
)

if ling_aligned[ling_cols].isna().any().any():
    raise ValueError("Missing linguistic values after filename alignment.")

X_linguistic = ling_aligned[ling_cols].to_numpy(dtype=np.float32)
X_combined = np.hstack([X_audio, X_linguistic])

# --------------------------------------------------
# 4. Validate inputs
# --------------------------------------------------
for name, X in [
    ("Audio", X_audio),
    ("Audio + linguistic", X_combined),
]:
    if not np.isfinite(X).all():
        raise ValueError(f"{name} features contain NaN or infinity.")

if not np.isfinite(y).all():
    raise ValueError("Training labels contain NaN or infinity.")

print("Data checks passed.")
print("Training samples:", len(y))
print("Audio features:", X_audio.shape)
print("Audio + linguistic features:", X_combined.shape)

# --------------------------------------------------
# 5. Run the same five folds for both models
# --------------------------------------------------
cv = KFold(n_splits=5, shuffle=True, random_state=42)

oof_audio = np.full(len(y), np.nan)
oof_combined = np.full(len(y), np.nan)

def create_model():
    return HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42,
    )

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_audio), start=1):
    # Audio-only model
    model_audio = create_model()
    model_audio.fit(X_audio[train_idx], y[train_idx])
    oof_audio[valid_idx] = model_audio.predict(X_audio[valid_idx])

    # Audio + linguistic model
    model_combined = create_model()
    model_combined.fit(X_combined[train_idx], y[train_idx])
    oof_combined[valid_idx] = model_combined.predict(
        X_combined[valid_idx]
    )

    print(f"Completed fold {fold}/5")

# --------------------------------------------------
# 6. Calculate out-of-fold metrics
# --------------------------------------------------
results = []

for name, pred in [
    ("Audio-only", oof_audio),
    ("Audio + linguistic", oof_combined),
]:
    rmse = np.sqrt(mean_squared_error(y, pred))
    mae = mean_absolute_error(y, pred)
    pearson = np.corrcoef(y, pred)[0, 1]

    results.append({
        "model": name,
        "OOF_RMSE": rmse,
        "OOF_MAE": mae,
        "OOF_Pearson": pearson,
    })

    print(f"\n{name}")
    print(f"  OOF RMSE:    {rmse:.4f} (lower is better)")
    print(f"  OOF MAE:     {mae:.4f} (lower is better)")
    print(f"  OOF Pearson: {pearson:.4f} (higher is better)")

# --------------------------------------------------
# 7. Save results without overwriting old files
# --------------------------------------------------
OUTPUTS.mkdir(parents=True, exist_ok=True)

results_df = pd.DataFrame(results)
results_path = OUTPUTS / "controlled_model_comparison.csv"
predictions_path = OUTPUTS / "controlled_oof_predictions.csv"

results_df.to_csv(results_path, index=False)

pd.DataFrame({
    "filename": train_df["filename"],
    "actual": y,
    "oof_audio": oof_audio,
    "oof_audio_linguistic": oof_combined,
}).to_csv(predictions_path, index=False)

print("\nSaved validation results:", results_path)
print("Saved OOF predictions:", predictions_path)
print("\nExperiment complete. No submission was created.")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = PROJECT / "data" / "Dataset_Final"
FEATURES = PROJECT / "features"
OUTPUTS = PROJECT / "outputs"
TRANSCRIPTS = OUTPUTS / "transcripts"

# --------------------------------------------------
# 2. Load training and test data
# --------------------------------------------------
train_df = pd.read_csv(DATA / "train.csv")
test_df = pd.read_csv(DATA / "test.csv")
sample_df = pd.read_csv(DATA / "sample_submission.csv")

X_train_wavlm = np.load(FEATURES / "X_train_wavlm.npy")
X_test_wavlm = np.load(FEATURES / "X_test_wavlm.npy")
X_train_acoustic = np.load(FEATURES / "X_train_acoustic.npy")
X_test_acoustic = np.load(FEATURES / "X_test_acoustic.npy")

y = train_df["label"].to_numpy(dtype=np.float64)

# --------------------------------------------------
# 3. Align linguistic features by filename
# --------------------------------------------------
ling_cols = [
    "word_count",
    "sentence_count",
    "avg_words_per_sentence",
    "unique_word_count",
    "vocabulary_diversity",
    "repeated_adjacent_count",
    "repeated_word_rate",
    "filler_count",
    "incomplete_ending",
]

train_ling_df = pd.read_csv(TRANSCRIPTS / "train_linguistic_features.csv")
test_ling_df = pd.read_csv(TRANSCRIPTS / "test_linguistic_features.csv")

def align_linguistic_features(reference_df, linguistic_df, columns):
    if "filename" not in reference_df or "filename" not in linguistic_df:
        raise ValueError("Missing filename column.")
    if reference_df["filename"].duplicated().any():
        raise ValueError("Duplicate filenames in reference data.")
    if linguistic_df["filename"].duplicated().any():
        raise ValueError("Duplicate filenames in linguistic data.")

    aligned = reference_df[["filename"]].merge(
        linguistic_df[["filename"] + columns],
        on="filename",
        how="left",
        validate="one_to_one",
        sort=False,
    )

    if aligned[columns].isna().any().any():
        raise ValueError("Missing linguistic features after alignment.")

    return aligned[columns].to_numpy(dtype=np.float32)

X_train_ling = align_linguistic_features(
    train_df, train_ling_df, ling_cols
)
X_test_ling = align_linguistic_features(
    test_df, test_ling_df, ling_cols
)

# --------------------------------------------------
# 4. Construct combined features
# --------------------------------------------------
X_train_audio = np.hstack([X_train_wavlm, X_train_acoustic])
X_test_audio = np.hstack([X_test_wavlm, X_test_acoustic])

X_train = np.hstack([X_train_audio, X_train_ling])
X_test = np.hstack([X_test_audio, X_test_ling])

assert X_train.shape == (len(train_df), 824), X_train.shape
assert X_test.shape == (len(test_df), 824), X_test.shape
assert len(y) == len(train_df)
assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()
assert np.isfinite(y).all()

# --------------------------------------------------
# 5. Fit selected model on all training samples
# --------------------------------------------------
model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42,
)

model.fit(X_train, y)

predictions = model.predict(X_test)

# Grammar score bounds
predictions = np.clip(predictions, 0.0, 5.0)

# --------------------------------------------------
# 6. Create submission in test.csv order
# --------------------------------------------------
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": predictions,
})

output_path = OUTPUTS / "submission_hgb_audio_linguistic_v2.csv"
submission.to_csv(output_path, index=False)

# --------------------------------------------------
# 7. Validate saved submission
# --------------------------------------------------
saved = pd.read_csv(output_path)

assert len(saved) == len(test_df) == 216
assert saved["filename"].equals(test_df["filename"])
assert saved["filename"].is_unique
assert saved["label"].notna().all()
assert saved["label"].between(0, 5).all()

print("Submission created and validated.")
print("File:", output_path)
print("Rows:", len(saved))
print("Columns:", saved.columns.tolist())
print("\nPrediction summary:")
print(saved["label"].describe().round(4))
print("\nFirst 10 predictions:")
print(saved.head(10).to_string(index=False))
print("\nExisting submission files were not overwritten.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
OUTPUTS = PROJECT / "outputs"

files = [
    "submission_candidate.csv",
    "submission_audio_linguistic.csv",
    "submission_extratrees_audio.csv",
    "submission_hgb_audio_linguistic_v2.csv",
]

submissions = {}

for filename in files:
    path = OUTPUTS / filename
    if path.exists():
        df = pd.read_csv(path)
        if {"filename", "label"}.issubset(df.columns):
            submissions[filename] = df.set_index("filename")["label"]

print("PAIRWISE SUBMISSION COMPARISON")
print("=" * 75)

names = list(submissions)

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]

        aligned = pd.concat(
            [submissions[a].rename("a"), submissions[b].rename("b")],
            axis=1,
            join="inner",
        ).dropna()

        diff = aligned["a"] - aligned["b"]

        print(f"\n{a}  vs  {b}")
        print("Matching files:", len(aligned))
        print("Correlation:", round(aligned["a"].corr(aligned["b"]), 5))
        print("Mean absolute difference:", round(diff.abs().mean(), 5))
        print("Maximum absolute difference:", round(diff.abs().max(), 5))

print("\nComparison complete. No files were changed.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
OUTPUTS = PROJECT / "outputs"

# Load out-of-fold predictions generated on identical CV folds
path = OUTPUTS / "controlled_oof_predictions.csv"
df = pd.read_csv(path)

required = ["actual", "oof_audio", "oof_audio_linguistic"]
missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in controlled OOF file: {missing}")

y = df["actual"].to_numpy(dtype=float)
hgb_audio = df["oof_audio"].to_numpy(dtype=float)
hgb_combined = df["oof_audio_linguistic"].to_numpy(dtype=float)

# Load existing ExtraTrees combined-feature OOF predictions
et_df = pd.read_csv(OUTPUTS / "extratrees_revalidation_oof.csv")

if "actual" not in et_df or "extratrees_oof" not in et_df:
    raise ValueError("Unexpected columns in ExtraTrees OOF file.")

# Confirm the actual labels align row-by-row before blending
et_y = et_df["actual"].to_numpy(dtype=float)
if len(et_y) != len(y) or not np.allclose(et_y, y):
    raise ValueError(
        "ExtraTrees labels do not align with controlled OOF labels. "
        "Do not blend until row alignment is resolved."
    )

et_pred = et_df["extratrees_oof"].to_numpy(dtype=float)

# Predefined blend weights; 1.0 means HGB combined only
weights = [0.0, 0.25, 0.50, 0.75, 1.0]

results = []

print("OOF BLENDING EXPERIMENT")
print("=" * 78)
print("Blend = weight × HGB audio+linguistic + (1-weight) × ExtraTrees")
print("All metrics are computed on saved out-of-fold predictions.\n")

for weight in weights:
    pred = weight * hgb_combined + (1 - weight) * et_pred

    rmse = np.sqrt(mean_squared_error(y, pred))
    mae = mean_absolute_error(y, pred)
    pearson = np.corrcoef(y, pred)[0, 1]

    results.append({
        "HGB_weight": weight,
        "ExtraTrees_weight": 1 - weight,
        "OOF_RMSE": rmse,
        "OOF_MAE": mae,
        "OOF_Pearson": pearson,
    })

    print(
        f"HGB={weight:.2f}, ET={1-weight:.2f} | "
        f"RMSE={rmse:.4f} | MAE={mae:.4f} | Pearson={pearson:.4f}"
    )

results_df = pd.DataFrame(results)
save_path = OUTPUTS / "blend_oof_results.csv"
results_df.to_csv(save_path, index=False)

print("\nBest blend by RMSE:")
print(results_df.loc[results_df["OOF_RMSE"].idxmin()].to_string())

print("\nBest blend by Pearson:")
print(results_df.loc[results_df["OOF_Pearson"].idxmax()].to_string())

print("\nSaved results:", save_path)
print("No model was retrained. No Kaggle submission was created.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = PROJECT / "data" / "Dataset_Final"
FEATURES = PROJECT / "features"
OUTPUTS = PROJECT / "outputs"
TRANSCRIPTS = OUTPUTS / "transcripts"

# Load labels and audio features
train_df = pd.read_csv(DATA / "train.csv")
test_df = pd.read_csv(DATA / "test.csv")

X_train_wavlm = np.load(FEATURES / "X_train_wavlm.npy")
X_test_wavlm = np.load(FEATURES / "X_test_wavlm.npy")
X_train_acoustic = np.load(FEATURES / "X_train_acoustic.npy")
X_test_acoustic = np.load(FEATURES / "X_test_acoustic.npy")
y = train_df["label"].to_numpy(dtype=float)

# Align linguistic features by filename
ling_cols = [
    "word_count", "sentence_count", "avg_words_per_sentence",
    "unique_word_count", "vocabulary_diversity",
    "repeated_adjacent_count", "repeated_word_rate",
    "filler_count", "incomplete_ending",
]

train_ling_df = pd.read_csv(TRANSCRIPTS / "train_linguistic_features.csv")
test_ling_df = pd.read_csv(TRANSCRIPTS / "test_linguistic_features.csv")

def align_linguistic(reference, linguistic):
    return reference[["filename"]].merge(
        linguistic[["filename"] + ling_cols],
        on="filename",
        how="left",
        validate="one_to_one",
        sort=False,
    )[ling_cols].to_numpy(dtype=np.float32)

X_train_audio = np.hstack([X_train_wavlm, X_train_acoustic])
X_test_audio = np.hstack([X_test_wavlm, X_test_acoustic])

X_train_ling = align_linguistic(train_df, train_ling_df)
X_test_ling = align_linguistic(test_df, test_ling_df)

X_train_hgb = np.hstack([X_train_audio, X_train_ling])
X_test_hgb = np.hstack([X_test_audio, X_test_ling])

# Validate dimensions and finite values
assert X_train_hgb.shape == (len(train_df), 824)
assert X_test_hgb.shape == (len(test_df), 824)
assert X_train_audio.shape[1] == 815
assert np.isfinite(X_train_hgb).all()
assert np.isfinite(X_test_hgb).all()
assert np.isfinite(y).all()

# Train HGB using the configuration evaluated in cross-validation
hgb = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42,
)
hgb.fit(X_train_hgb, y)
pred_hgb = hgb.predict(X_test_hgb)

# Train ExtraTrees on the combined audio features
et = ExtraTreesRegressor(
    n_estimators=400,
    min_samples_leaf=2,
    max_features=1.0,
    random_state=42,
    n_jobs=-1,
)
et.fit(X_train_audio, y)
pred_et = et.predict(X_test_audio)

# Blend using the OOF-selected weights
pred_blend = 0.75 * pred_hgb + 0.25 * pred_et
pred_blend = np.clip(pred_blend, 0.0, 5.0)

submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": pred_blend,
})

output_path = OUTPUTS / "submission_blend_hgb75_et25.csv"
submission.to_csv(output_path, index=False)

# Validate
saved = pd.read_csv(output_path)
assert len(saved) == len(test_df) == 216
assert saved["filename"].equals(test_df["filename"])
assert saved["filename"].is_unique
assert saved["label"].notna().all()
assert saved["label"].between(0, 5).all()

print("Blended submission created:", output_path)
print("Rows:", len(saved))
print("\nPrediction summary:")
print(saved["label"].describe().round(4))
print("\nFirst 10 rows:")
print(saved.head(10).to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
OUTPUTS = PROJECT / "outputs"

files = [
    "submission_candidate.csv",
    "submission_audio_linguistic.csv",
    "submission_extratrees_audio.csv",
    "submission_blend_hgb75_et25.csv",
]

submissions = {}

for filename in files:
    path = OUTPUTS / filename
    if not path.exists():
        print("Missing:", filename)
        continue

    df = pd.read_csv(path)
    if not {"filename", "label"}.issubset(df.columns):
        print("Unexpected columns:", filename, df.columns.tolist())
        continue

    submissions[filename] = df.set_index("filename")["label"]

reference = "submission_blend_hgb75_et25.csv"
blend = submissions[reference]

print("BLEND COMPARISON")
print("=" * 75)

for filename, prediction in submissions.items():
    if filename == reference:
        continue

    aligned = pd.concat(
        [blend.rename("blend"), prediction.rename("other")],
        axis=1,
        join="inner",
    ).dropna()

    difference = aligned["blend"] - aligned["other"]

    print(f"\nBlend vs {filename}")
    print("Matching files:", len(aligned))
    print("Correlation:", round(aligned["blend"].corr(aligned["other"]), 5))
    print("Mean absolute difference:", round(difference.abs().mean(), 5))
    print("Maximum absolute difference:", round(difference.abs().max(), 5))

print("\nNo files were modified.")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = PROJECT / "data" / "Dataset_Final"
FEATURES = PROJECT / "features"
OUTPUTS = PROJECT / "outputs"

# 1. Load training labels and precomputed features
train_df = pd.read_csv(DATA / "train.csv")
y = train_df["label"].to_numpy(dtype=float)

X_wavlm = np.load(FEATURES / "X_train_wavlm.npy")
X_acoustic = np.load(FEATURES / "X_train_acoustic.npy")

X_combined = np.hstack([X_wavlm, X_acoustic])

assert len(y) == len(X_combined)
assert np.isfinite(y).all() and np.isfinite(X_combined).all()

# 2. Reproduce the previous ExtraTrees configuration
cv = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.full(len(y), np.nan)

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_combined), start=1):
    model = ExtraTreesRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_combined[train_idx], y[train_idx])
    oof[valid_idx] = model.predict(X_combined[valid_idx])

    fold_rmse = np.sqrt(
        mean_squared_error(y[valid_idx], oof[valid_idx])
    )
    print(f"Fold {fold}/5 | RMSE: {fold_rmse:.4f}")

# 3. Aggregate OOF metrics
rmse = np.sqrt(mean_squared_error(y, oof))
mae = mean_absolute_error(y, oof)
pearson = np.corrcoef(y, oof)[0, 1]

print("\nREPRODUCED EXTRATREES RESULTS")
print(f"OOF RMSE:    {rmse:.4f}")
print(f"OOF MAE:     {mae:.4f}")
print(f"OOF Pearson: {pearson:.4f}")

# 4. Compare against saved predictions
saved_path = OUTPUTS / "extratrees_revalidation_oof.csv"

if saved_path.exists():
    saved = pd.read_csv(saved_path)

    if len(saved) == len(y) and np.allclose(saved["actual"], y):
        old_oof = saved["extratrees_oof"].to_numpy(dtype=float)

        print("\nCOMPARISON WITH SAVED OOF PREDICTIONS")
        print("Correlation:", round(np.corrcoef(oof, old_oof)[0, 1], 6))
        print("Mean absolute difference:", round(np.mean(np.abs(oof - old_oof)), 6))
        print("Maximum absolute difference:", round(np.max(np.abs(oof - old_oof)), 6))
    else:
        print("\nSaved OOF rows or labels do not align; comparison skipped.")
else:
    print("\nSaved OOF file not found.")

print("\nNo submission was created. No existing files were changed.")

In [ ]:
from pathlib import Path
import json

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")

print("=" * 75)
print("MODEL AND FEATURE ARTIFACT INVENTORY")
print("=" * 75)

for folder_name in ["features", "outputs"]:
    folder = PROJECT / folder_name
    print(f"\n--- {folder_name.upper()} ---")

    if not folder.exists():
        print("Folder missing")
        continue

    for path in sorted(folder.rglob("*")):
        if not path.is_file():
            continue

        suffix = path.suffix.lower()
        if suffix in {".npy", ".npz", ".joblib", ".pkl", ".json", ".csv"}:
            size_mb = path.stat().st_size / (1024 * 1024)
            print(f"{path.relative_to(PROJECT)} | {size_mb:.3f} MB")

print("\n--- SAVED MODEL CONFIGURATIONS ---")

try:
    import joblib

    for path in sorted((PROJECT / "outputs").glob("*.joblib")):
        try:
            model = joblib.load(path)
            print(f"\n{path.name}")
            print("Type:", type(model).__name__)

            params = getattr(model, "get_params", None)
            if callable(params):
                print("Parameters:")
                print(json.dumps(params(), indent=2, default=str))

            print("Expected features:", getattr(model, "n_features_in_", "unknown"))

        except Exception as exc:
            print(f"Could not inspect {path.name}: {exc}")

except ImportError:
    print("joblib is not installed.")

print("\nInventory complete. No files were changed.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
FEATURES = PROJECT / "features"
TRANSCRIPTS = PROJECT / "outputs" / "transcripts"

train_df = pd.read_csv(PROJECT / "data" / "Dataset_Final" / "train.csv")
y = train_df["label"].to_numpy(dtype=float)

X_wavlm = np.load(FEATURES / "X_train_wavlm.npy")
X_acoustic = np.load(FEATURES / "X_train_acoustic.npy")
X_audio = np.hstack([X_wavlm, X_acoustic])

ling_cols = [
    "word_count", "sentence_count", "avg_words_per_sentence",
    "unique_word_count", "vocabulary_diversity",
    "repeated_adjacent_count", "repeated_word_rate",
    "filler_count", "incomplete_ending",
]

ling_df = pd.read_csv(TRANSCRIPTS / "train_linguistic_features.csv")
ling_df = train_df[["filename"]].merge(
    ling_df[["filename"] + ling_cols],
    on="filename",
    how="left",
    validate="one_to_one",
    sort=False,
)

X_combined = np.hstack([
    X_audio,
    ling_df[ling_cols].to_numpy(dtype=np.float32),
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def make_model():
    return HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42,
    )

results = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_audio), start=1):
    row = {"fold": fold}

    for name, X in [
        ("audio", X_audio),
        ("combined", X_combined),
    ]:
        model = make_model()
        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[valid_idx])
        row[f"{name}_rmse"] = np.sqrt(
            mean_squared_error(y[valid_idx], pred)
        )

    row["rmse_change"] = row["combined_rmse"] - row["audio_rmse"]
    results.append(row)

results_df = pd.DataFrame(results)

print(results_df.round(4).to_string(index=False))
print("\nFolds where linguistic features improved RMSE:",
      int((results_df["rmse_change"] < 0).sum()), "/ 5")
print("Mean RMSE change (combined - audio):",
      round(results_df["rmse_change"].mean(), 5))
print("\nNegative change means the combined model performed better.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
FEATURES = PROJECT / "features"
TRANSCRIPTS = PROJECT / "outputs" / "transcripts"

train_df = pd.read_csv(PROJECT / "data" / "Dataset_Final" / "train.csv")
y = train_df["label"].to_numpy(dtype=float)

X_wavlm = np.load(FEATURES / "X_train_wavlm.npy")
X_acoustic = np.load(FEATURES / "X_train_acoustic.npy")
X_audio = np.hstack([X_wavlm, X_acoustic])

ling_cols = [
    "word_count", "sentence_count", "avg_words_per_sentence",
    "unique_word_count", "vocabulary_diversity",
    "repeated_adjacent_count", "repeated_word_rate",
    "filler_count", "incomplete_ending",
]

ling_df = pd.read_csv(TRANSCRIPTS / "train_linguistic_features.csv")
ling_df = train_df[["filename"]].merge(
    ling_df[["filename"] + ling_cols],
    on="filename",
    how="left",
    validate="one_to_one",
    sort=False,
)

X_combined = np.hstack([
    X_audio,
    ling_df[ling_cols].to_numpy(dtype=np.float32),
])

def make_model():
    return HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42,
    )

cv = KFold(n_splits=5, shuffle=True, random_state=123)
oof_audio = np.full(len(y), np.nan)
oof_combined = np.full(len(y), np.nan)

for fold, (tr, va) in enumerate(cv.split(X_audio), start=1):
    model_audio = make_model()
    model_audio.fit(X_audio[tr], y[tr])
    oof_audio[va] = model_audio.predict(X_audio[va])

    model_combined = make_model()
    model_combined.fit(X_combined[tr], y[tr])
    oof_combined[va] = model_combined.predict(X_combined[va])

    print(f"Completed fold {fold}/5")

for name, pred in [
    ("Audio-only", oof_audio),
    ("Audio + linguistic", oof_combined),
]:
    rmse = np.sqrt(mean_squared_error(y, pred))
    mae = mean_absolute_error(y, pred)
    pearson = np.corrcoef(y, pred)[0, 1]
    print(
        f"\n{name}: RMSE={rmse:.4f}, "
        f"MAE={mae:.4f}, Pearson={pearson:.4f}"
    )

print("\nNo files were modified and no submission was created.")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
FEATURES = PROJECT / "features"
TRANSCRIPTS = PROJECT / "outputs" / "transcripts"

train_df = pd.read_csv(PROJECT / "data" / "Dataset_Final" / "train.csv")
y = train_df["label"].to_numpy(dtype=float)

# Reuse existing features
X_wavlm = np.load(FEATURES / "X_train_wavlm.npy")
X_acoustic = np.load(FEATURES / "X_train_acoustic.npy")
X_audio = np.hstack([X_wavlm, X_acoustic])

ling_cols = [
    "word_count", "sentence_count", "avg_words_per_sentence",
    "unique_word_count", "vocabulary_diversity",
    "repeated_adjacent_count", "repeated_word_rate",
    "filler_count", "incomplete_ending",
]

ling_df = pd.read_csv(TRANSCRIPTS / "train_linguistic_features.csv")
ling_df = train_df[["filename"]].merge(
    ling_df[["filename"] + ling_cols],
    on="filename",
    how="left",
    validate="one_to_one",
    sort=False,
)

X = np.hstack([
    X_audio,
    ling_df[ling_cols].to_numpy(dtype=np.float64),
])

models = {
    "HGB": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42,
    ),
    "Ridge_alpha_10": make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0),
    ),
}

results = []

for seed in [42, 123]:
    cv = KFold(n_splits=5, shuffle=True, random_state=seed)

    for model_name, prototype in models.items():
        oof = np.full(len(y), np.nan)

        for tr, va in cv.split(X):
            # Fresh model instance for each fold
            from sklearn.base import clone
            model = clone(prototype)
            model.fit(X[tr], y[tr])
            oof[va] = model.predict(X[va])

        results.append({
            "seed": seed,
            "model": model_name,
            "RMSE": np.sqrt(mean_squared_error(y, oof)),
            "MAE": mean_absolute_error(y, oof),
            "Pearson": np.corrcoef(y, oof)[0, 1],
        })

results_df = pd.DataFrame(results)
print(results_df.round(4).to_string(index=False))

print("\nLower RMSE/MAE and higher Pearson are better.")
print("No submission was created; no existing files were modified.")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
OUTPUTS = PROJECT / "outputs"

# Load genuine out-of-fold predictions
path = OUTPUTS / "controlled_oof_predictions.csv"
df = pd.read_csv(path)

required = {"actual", "oof_audio", "oof_audio_linguistic"}
if not required.issubset(df.columns):
    raise ValueError(
        f"Expected columns {sorted(required)}; found {df.columns.tolist()}"
    )

y = df["actual"].to_numpy(dtype=float)
pred = df["oof_audio_linguistic"].to_numpy(dtype=float)

if not np.isfinite(y).all() or not np.isfinite(pred).all():
    raise ValueError("Actual values or predictions contain non-finite values.")

# Positive error means overprediction; negative means underprediction
df["error"] = pred - y
df["absolute_error"] = np.abs(df["error"])
df["squared_error"] = df["error"] ** 2

rmse = np.sqrt(mean_squared_error(y, pred))
mae = mean_absolute_error(y, pred)
pearson = np.corrcoef(y, pred)[0, 1]

print("=" * 70)
print("OVERALL OUT-OF-FOLD ERROR ANALYSIS")
print("=" * 70)
print(f"Samples: {len(df)}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Pearson: {pearson:.4f}")
print(f"Mean signed error (prediction - actual): {df['error'].mean():.4f}")
print(f"Median signed error: {df['error'].median():.4f}")
print(f"Overpredicted samples: {(df['error'] > 0).sum()}")
print(f"Underpredicted samples: {(df['error'] < 0).sum()}")
print(f"Exact predictions: {(df['error'] == 0).sum()}")

# Group actual scores into meaningful ranges
bins = [-np.inf, 1.5, 2.5, 3.5, 4.5, np.inf]
labels = ["<=1.5", "(1.5, 2.5]", "(2.5, 3.5]", "(3.5, 4.5]", ">4.5"]
df["actual_range"] = pd.cut(
    y, bins=bins, labels=labels, include_lowest=True
)

grouped = df.groupby("actual_range", observed=False).agg(
    samples=("actual", "size"),
    mean_actual=("actual", "mean"),
    mean_prediction=("oof_audio_linguistic", "mean"),
    mean_error=("error", "mean"),
    MAE=("absolute_error", "mean"),
    RMSE=("squared_error", lambda s: np.sqrt(s.mean())),
).reset_index()

print("\n" + "=" * 70)
print("ERROR BY ACTUAL SCORE RANGE")
print("=" * 70)
print(grouped.round(4).to_string(index=False))

# Save diagnostics without overwriting prior results
diagnostic_path = OUTPUTS / "oof_error_analysis.csv"
grouped.to_csv(diagnostic_path, index=False)

print("\nSaved:", diagnostic_path)
print("No models were trained. No submission was created.")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

# Load existing out-of-fold predictions; no feature extraction or model training.
df = pd.read_csv("outputs/controlled_oof_predictions.csv")

y = df["actual"].to_numpy()
pred = df["oof_audio_linguistic"].to_numpy()

# Use the same five-fold split structure for a fair comparison.
cv = KFold(n_splits=5, shuffle=True, random_state=42)
calibrated = np.empty_like(pred, dtype=float)

for train_idx, valid_idx in cv.split(pred):
    # Fit the calibration only on the other four folds.
    calibrator = LinearRegression()
    calibrator.fit(pred[train_idx].reshape(-1, 1), y[train_idx])

    calibrated[valid_idx] = calibrator.predict(
        pred[valid_idx].reshape(-1, 1)
    )

def report(name, actual, prediction):
    rmse = np.sqrt(mean_squared_error(actual, prediction))
    mae = mean_absolute_error(actual, prediction)
    pearson = pearsonr(actual, prediction).statistic

    print(
        f"{name:22s} | "
        f"RMSE={rmse:.4f} | "
        f"MAE={mae:.4f} | "
        f"Pearson={pearson:.4f}"
    )

print("CROSS-VALIDATED CALIBRATION COMPARISON")
report("Original", y, pred)
report("Calibrated", y, calibrated)

print("\nCalibration prediction range:")
print(f"Min:  {calibrated.min():.4f}")
print(f"Max:  {calibrated.max():.4f}")
print(f"Mean: {calibrated.mean():.4f}")

print("\nAverage fold-specific calibration coefficients:")
print("The fitted slopes/intercepts vary by fold, so inspect each fold")
print("before deciding whether calibration is stable.")

In [ ]:
from pathlib import Path
import pandas as pd

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
oof_path = project / "outputs" / "controlled_oof_predictions.csv"

if not oof_path.exists():
    raise FileNotFoundError(
        f"OOF file not found at: {oof_path}\n"
        "Check whether the file exists in your project's outputs folder."
    )

df = pd.read_csv(oof_path)

print("Loaded:", oof_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

# 1. Load existing out-of-fold predictions
project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
oof_path = project / "outputs" / "controlled_oof_predictions.csv"

if not oof_path.exists():
    raise FileNotFoundError(f"OOF file not found: {oof_path}")

df = pd.read_csv(oof_path)

required_columns = {"actual", "oof_audio_linguistic"}
missing = required_columns - set(df.columns)

if missing:
    raise ValueError(f"Missing required columns: {missing}")

y = df["actual"].to_numpy(dtype=float)
pred = df["oof_audio_linguistic"].to_numpy(dtype=float)

if not np.isfinite(y).all() or not np.isfinite(pred).all():
    raise ValueError("The actual scores or predictions contain missing/non-finite values.")

# 2. Generate calibrated predictions without fitting on each sample's own label
cv = KFold(n_splits=5, shuffle=True, random_state=42)
calibrated = np.empty(len(y), dtype=float)
fold_results = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(pred), start=1):
    calibrator = LinearRegression()

    calibrator.fit(
        pred[train_idx].reshape(-1, 1),
        y[train_idx]
    )

    calibrated[valid_idx] = calibrator.predict(
        pred[valid_idx].reshape(-1, 1)
    )

    fold_results.append({
        "fold": fold,
        "slope": calibrator.coef_[0],
        "intercept": calibrator.intercept_,
        "validation_samples": len(valid_idx)
    })

# 3. Calculate evaluation metrics
def calculate_metrics(actual, prediction):
    return {
        "RMSE": np.sqrt(mean_squared_error(actual, prediction)),
        "MAE": mean_absolute_error(actual, prediction),
        "Pearson": pearsonr(actual, prediction).statistic
    }

original_metrics = calculate_metrics(y, pred)
calibrated_metrics = calculate_metrics(y, calibrated)

comparison = pd.DataFrame(
    [original_metrics, calibrated_metrics],
    index=["Original HGB audio + linguistic", "Cross-validated calibration"]
)

print("=" * 65)
print("CALIBRATION EXPERIMENT")
print("=" * 65)
print(f"Samples: {len(y)}")
print("\nMetric comparison:")
print(comparison.round(4).to_string())

print("\nChange after calibration (calibrated - original):")
for metric in ["RMSE", "MAE", "Pearson"]:
    change = calibrated_metrics[metric] - original_metrics[metric]
    print(f"{metric}: {change:+.4f}")

# 4. Inspect calibration stability across folds
fold_df = pd.DataFrame(fold_results)

print("\nFold-specific calibration coefficients:")
print(fold_df.round(4).to_string(index=False))

print("\nCalibration coefficient variability:")
print(f"Mean slope:     {fold_df['slope'].mean():.4f}")
print(f"Std. dev. slope:{fold_df['slope'].std():.4f}")
print(f"Mean intercept: {fold_df['intercept'].mean():.4f}")
print(f"Std. dev. intercept: {fold_df['intercept'].std():.4f}")

# 5. Save diagnostic results only
output_path = project / "outputs" / "calibration_experiment_results.csv"

pd.DataFrame({
    "filename": df["filename"],
    "actual": y,
    "original_oof": pred,
    "calibrated_oof": calibrated
}).to_csv(output_path, index=False)

print(f"\nSaved diagnostic predictions to: {output_path}")
print("\nNo audio models were retrained. No submission was created.")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

# 1. Load existing out-of-fold predictions
project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
oof_path = project / "outputs" / "controlled_oof_predictions.csv"

if not oof_path.exists():
    raise FileNotFoundError(f"File not found: {oof_path}")

df = pd.read_csv(oof_path)

required = {
    "filename", "actual", "oof_audio", "oof_audio_linguistic"
}
if not required.issubset(df.columns):
    raise ValueError(f"Missing columns: {required - set(df.columns)}")

# 2. Calculate errors for both existing models
for model in ["oof_audio", "oof_audio_linguistic"]:
    df[f"{model}_error"] = df[model] - df["actual"]
    df[f"{model}_abs_error"] = df[f"{model}_error"].abs()

def metrics(actual, prediction):
    return {
        "RMSE": np.sqrt(mean_squared_error(actual, prediction)),
        "MAE": mean_absolute_error(actual, prediction),
        "Pearson": pearsonr(actual, prediction).statistic
    }

comparison = pd.DataFrame({
    "Audio only": metrics(df["actual"], df["oof_audio"]),
    "Audio + linguistic": metrics(
        df["actual"], df["oof_audio_linguistic"]
    )
}).T

print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)
print(comparison.round(4))

# 3. Compare performance by actual grammar score range
bins = [-np.inf, 1.5, 2.5, 3.5, 4.5, np.inf]
labels = ["<=1.5", "(1.5,2.5]", "(2.5,3.5]", "(3.5,4.5]", ">4.5"]

df["actual_range"] = pd.cut(
    df["actual"], bins=bins, labels=labels, right=True
)

range_results = []

for score_range, group in df.groupby(
    "actual_range", observed=False
):
    if group.empty:
        continue

    for model, display_name in [
        ("oof_audio", "Audio only"),
        ("oof_audio_linguistic", "Audio + linguistic")
    ]:
        actual = group["actual"].to_numpy()
        prediction = group[model].to_numpy()

        range_results.append({
            "actual_range": str(score_range),
            "model": display_name,
            "samples": len(group),
            "mean_actual": actual.mean(),
            "mean_prediction": prediction.mean(),
            "mean_error": (prediction - actual).mean(),
            "MAE": mean_absolute_error(actual, prediction),
            "RMSE": np.sqrt(mean_squared_error(actual, prediction))
        })

range_df = pd.DataFrame(range_results)

print("\n" + "=" * 70)
print("ERROR BY ACTUAL SCORE RANGE")
print("=" * 70)
print(range_df.round(4).to_string(index=False))

# 4. Find samples where adding linguistic features helped or hurt most
df["linguistic_improvement"] = (
    df["oof_audio_abs_error"]
    - df["oof_audio_linguistic_abs_error"]
)

print("\n" + "=" * 70)
print("10 SAMPLES WHERE LINGUISTIC FEATURES HELPED MOST")
print("=" * 70)
print(
    df.nlargest(10, "linguistic_improvement")[
        [
            "filename", "actual", "oof_audio",
            "oof_audio_linguistic", "linguistic_improvement"
        ]
    ].round(4).to_string(index=False)
)

print("\n" + "=" * 70)
print("10 SAMPLES WHERE LINGUISTIC FEATURES HURT MOST")
print("=" * 70)
print(
    df.nsmallest(10, "linguistic_improvement")[
        [
            "filename", "actual", "oof_audio",
            "oof_audio_linguistic", "linguistic_improvement"
        ]
    ].round(4).to_string(index=False)
)

# 5. Save diagnostics
output_dir = project / "outputs"
range_df.to_csv(output_dir / "oof_range_model_comparison.csv", index=False)
df.to_csv(output_dir / "oof_sample_error_diagnosis.csv", index=False)

print("\nSaved:")
print(output_dir / "oof_range_model_comparison.csv")
print(output_dir / "oof_sample_error_diagnosis.csv")
print("\nNo models were trained. No submission was created.")

In [ ]:
from pathlib import Path
import pandas as pd

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
path = project / "outputs" / "oof_sample_error_diagnosis.csv"

df = pd.read_csv(path)

print("=" * 65)
print("10 SAMPLES WHERE LINGUISTIC FEATURES HELPED MOST")
print("=" * 65)

print(
    df.nlargest(10, "linguistic_improvement")[
        ["filename", "actual", "oof_audio",
         "oof_audio_linguistic", "linguistic_improvement"]
    ].round(4).to_string(index=False)
)

print("\n" + "=" * 65)
print("10 SAMPLES WHERE LINGUISTIC FEATURES HURT MOST")
print("=" * 65)

print(
    df.nsmallest(10, "linguistic_improvement")[
        ["filename", "actual", "oof_audio",
         "oof_audio_linguistic", "linguistic_improvement"]
    ].round(4).to_string(index=False)
)

In [ ]:
from pathlib import Path

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"

print("Searching project for existing linguistic feature files...\n")

matches = []

for folder in [project / "outputs", project / "features", project / "data"]:
    if folder.exists():
        for path in folder.rglob("*.csv"):
            name = path.name.lower()
            if "linguistic" in name or "transcript" in name:
                matches.append(path)

if matches:
    for path in matches:
        print(path)
else:
    print("No linguistic/transcript CSV found in outputs, features, or data.")

print("\nExisting NumPy feature files:")
for path in (project / "features").glob("*.npy"):
    print(path.name, path.stat().st_size, "bytes")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"

# Load existing cached features
X_wavlm = np.load(project / "features" / "X_train_wavlm.npy")
X_acoustic = np.load(project / "features" / "X_train_acoustic.npy")
y = np.load(project / "features" / "y_train.npy")

# Correct path to linguistic features
ling_path = (
    project / "outputs" / "transcripts" / "train_linguistic_features.csv"
)
ling = pd.read_csv(ling_path)

train_csv = pd.read_csv(
    project / "data" / "Dataset_Final" / "train.csv"
)

# Align linguistic features to the training CSV by filename
ling = train_csv[["filename"]].merge(
    ling, on="filename", how="left", validate="one_to_one"
)

ling_cols = [
    "word_count",
    "sentence_count",
    "avg_words_per_sentence",
    "unique_word_count",
    "vocabulary_diversity",
    "repeated_adjacent_count",
    "repeated_word_rate",
    "filler_count",
    "incomplete_ending",
]

if ling[ling_cols].isna().any().any():
    raise ValueError("Missing linguistic features after filename alignment.")

X = np.hstack([
    X_wavlm,
    X_acoustic,
    ling[ling_cols].to_numpy(dtype=float),
])

if len(X) != len(y):
    raise ValueError(f"Feature/label mismatch: {len(X)} vs {len(y)}")

print("Feature matrix:", X.shape)
print("Labels:", y.shape)

configs = {
    "Current HGB": {
        "max_iter": 200,
        "learning_rate": 0.05,
        "max_leaf_nodes": 15,
        "l2_regularization": 1.0,
    },
    "Simpler HGB": {
        "max_iter": 150,
        "learning_rate": 0.04,
        "max_leaf_nodes": 7,
        "l2_regularization": 3.0,
    },
    "More regularized HGB": {
        "max_iter": 200,
        "learning_rate": 0.04,
        "max_leaf_nodes": 11,
        "l2_regularization": 5.0,
    },
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, params in configs.items():
    oof = np.zeros(len(y), dtype=float)

    for train_idx, valid_idx in cv.split(X):
        model = HistGradientBoostingRegressor(
            **params,
            early_stopping=False,
            random_state=42,
        )
        model.fit(X[train_idx], y[train_idx])
        oof[valid_idx] = model.predict(X[valid_idx])

    results.append({
        "model": name,
        "RMSE": np.sqrt(mean_squared_error(y, oof)),
        "MAE": mean_absolute_error(y, oof),
        "Pearson": pearsonr(y, oof).statistic,
    })

results_df = pd.DataFrame(results).sort_values("RMSE")

print("\nMODEL COMPLEXITY COMPARISON")
print(results_df.round(4).to_string(index=False))

output_path = project / "outputs" / "hgb_complexity_comparison.csv"
results_df.to_csv(output_path, index=False)
print("\nSaved:", output_path)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"

# Reuse the feature matrix construction from the previous cell.
X_wavlm = np.load(project / "features" / "X_train_wavlm.npy")
X_acoustic = np.load(project / "features" / "X_train_acoustic.npy")
y = np.load(project / "features" / "y_train.npy")

ling = pd.read_csv(
    project / "outputs" / "transcripts" / "train_linguistic_features.csv"
)
train_csv = pd.read_csv(
    project / "data" / "Dataset_Final" / "train.csv"
)

ling = train_csv[["filename"]].merge(
    ling, on="filename", how="left", validate="one_to_one"
)

ling_cols = [
    "word_count", "sentence_count", "avg_words_per_sentence",
    "unique_word_count", "vocabulary_diversity",
    "repeated_adjacent_count", "repeated_word_rate",
    "filler_count", "incomplete_ending"
]

X = np.hstack([
    X_wavlm,
    X_acoustic,
    ling[ling_cols].to_numpy(dtype=float)
])

configs = {
    "Current HGB": {
        "max_iter": 200,
        "learning_rate": 0.05,
        "max_leaf_nodes": 15,
        "l2_regularization": 1.0,
    },
    "More regularized HGB": {
        "max_iter": 200,
        "learning_rate": 0.04,
        "max_leaf_nodes": 11,
        "l2_regularization": 5.0,
    },
}

cv = KFold(n_splits=5, shuffle=True, random_state=123)
results = []

for name, params in configs.items():
    oof = np.zeros(len(y), dtype=float)

    for train_idx, valid_idx in cv.split(X):
        model = HistGradientBoostingRegressor(
            **params,
            early_stopping=False,
            random_state=42
        )
        model.fit(X[train_idx], y[train_idx])
        oof[valid_idx] = model.predict(X[valid_idx])

    results.append({
        "model": name,
        "RMSE": np.sqrt(mean_squared_error(y, oof)),
        "MAE": mean_absolute_error(y, oof),
        "Pearson": pearsonr(y, oof).statistic,
    })

print("SECOND VALIDATION SPLIT (seed=123)")
print(pd.DataFrame(results).round(4).to_string(index=False))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

# Project paths
project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
features_dir = project / "features"
data_dir = project / "data" / "Dataset_Final"
output_dir = project / "outputs"

# Load cached features: no audio feature extraction
X_train_wavlm = np.load(features_dir / "X_train_wavlm.npy")
X_test_wavlm = np.load(features_dir / "X_test_wavlm.npy")
X_train_acoustic = np.load(features_dir / "X_train_acoustic.npy")
X_test_acoustic = np.load(features_dir / "X_test_acoustic.npy")
y = np.load(features_dir / "y_train.npy")

train_df = pd.read_csv(data_dir / "train.csv")
test_df = pd.read_csv(data_dir / "test.csv")

# Load cached linguistic features
ling_train = pd.read_csv(
    output_dir / "transcripts" / "train_linguistic_features.csv"
)
ling_test = pd.read_csv(
    output_dir / "transcripts" / "test_linguistic_features.csv"
)

ling_cols = [
    "word_count",
    "sentence_count",
    "avg_words_per_sentence",
    "unique_word_count",
    "vocabulary_diversity",
    "repeated_adjacent_count",
    "repeated_word_rate",
    "filler_count",
    "incomplete_ending",
]

# Align features to the exact CSV filename order
ling_train = train_df[["filename"]].merge(
    ling_train, on="filename", how="left", validate="one_to_one"
)
ling_test = test_df[["filename"]].merge(
    ling_test, on="filename", how="left", validate="one_to_one"
)

assert not ling_train[ling_cols].isna().any().any(), \
    "Missing training linguistic features"
assert not ling_test[ling_cols].isna().any().any(), \
    "Missing test linguistic features"

# Combine all cached features
X_train = np.hstack([
    X_train_wavlm,
    X_train_acoustic,
    ling_train[ling_cols].to_numpy(dtype=float),
])

X_test = np.hstack([
    X_test_wavlm,
    X_test_acoustic,
    ling_test[ling_cols].to_numpy(dtype=float),
])

assert X_train.shape == (769, 824), X_train.shape
assert X_test.shape == (216, 824), X_test.shape
assert len(y) == len(train_df) == 769

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

# Train the selected model on all labeled samples
model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42,
)

model.fit(X_train, y)

# Predict and enforce the valid score range
predictions = np.clip(model.predict(X_test), 0, 5)

# Create submission in test.csv order
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": predictions,
})

assert len(submission) == 216
assert submission["filename"].is_unique
assert submission["filename"].equals(test_df["filename"])
assert submission["label"].notna().all()
assert submission["label"].between(0, 5).all()

save_path = output_dir / "submission_final_hgb_audio_linguistic.csv"
submission.to_csv(save_path, index=False)

print("\nCandidate saved:", save_path)
print("Rows:", len(submission))
print("Prediction mean:", round(submission["label"].mean(), 4))
print("Prediction std:", round(submission["label"].std(), 4))
print("Prediction min:", round(submission["label"].min(), 4))
print("Prediction max:", round(submission["label"].max(), 4))
display(submission.head(10))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.cwd()
while not (project / "data" / "Dataset_Final" / "test.csv").exists():
    if project.parent == project:
        raise FileNotFoundError("Could not locate Dataset_Final/test.csv")
    project = project.parent

out = project / "outputs"
test_df = pd.read_csv(project / "data" / "Dataset_Final" / "test.csv")
candidate = pd.read_csv(out / "submission_final_hgb_audio_linguistic.csv")
best = pd.read_csv(out / "submission_audio_linguistic.csv")

for name, df in [("Candidate", candidate), ("Previous best", best)]:
    valid = (
        len(df) == len(test_df)
        and df.columns.tolist() == ["filename", "label"]
        and df["filename"].equals(test_df["filename"])
        and df["label"].notna().all()
        and df["label"].between(0, 5).all()
    )
    print(f"{name} valid: {valid} | rows: {len(df)}")

if candidate["filename"].equals(best["filename"]):
    diff = np.abs(candidate["label"].to_numpy() - best["label"].to_numpy())
    print("Predictions identical:", np.array_equal(diff, np.zeros(len(diff))))
    print("Maximum absolute difference:", diff.max())
else:
    print("Filename order differs; investigate before submission.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"

# List only existing OOF files; do not rerun models.
oof_files = sorted(out.glob("*oof*.csv"))
print("Available OOF files:")
for f in oof_files:
    print(f.name)

In [ ]:
from pathlib import Path
import pandas as pd

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"

files_to_check = [
    "controlled_oof_predictions.csv",
    "model_tuning_oof_predictions.csv",
    "oof_predictions.csv",
    "calibration_oof_predictions.csv",
    "extratrees_revalidation_oof.csv",
]

for filename in files_to_check:
    path = out / filename
    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{filename}")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(2).to_string(index=False))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
df = pd.read_csv(project / "outputs" / "controlled_oof_predictions.csv")

y = df["actual"].to_numpy()
predictions = {
    "HGB audio": df["oof_audio"].to_numpy(),
    "HGB audio + linguistic": df["oof_audio_linguistic"].to_numpy(),
}

def report(name, actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    corr = pearsonr(actual, pred).statistic
    print(f"{name:32s} RMSE={rmse:.4f} | MAE={mae:.4f} | Pearson={corr:.4f}")

for name, raw in predictions.items():
    # Diagnostic only: fit and evaluate on the same OOF targets.
    # This is optimistic and must not be used to select a submission.
    calibrator = LinearRegression().fit(raw.reshape(-1, 1), y)
    calibrated = np.clip(calibrator.predict(raw.reshape(-1, 1)), 0, 5)

    print(f"\n{name}")
    print(f"Calibration: prediction = {calibrator.intercept_:.4f} "
          f"+ {calibrator.coef_[0]:.4f} * raw_prediction")
    report("Raw OOF", y, raw)
    report("In-sample calibrated OOF", y, calibrated)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
df = pd.read_csv(project / "outputs" / "controlled_oof_predictions.csv")

y = df["actual"].to_numpy()
raw = df["oof_audio_linguistic"].to_numpy()

calibrated = np.empty_like(raw, dtype=float)
kf = KFold(n_splits=5, shuffle=True, random_state=2026)

for train_idx, valid_idx in kf.split(raw):
    calibrator = LinearRegression()
    calibrator.fit(raw[train_idx].reshape(-1, 1), y[train_idx])
    calibrated[valid_idx] = calibrator.predict(
        raw[valid_idx].reshape(-1, 1)
    )

calibrated = np.clip(calibrated, 0, 5)

def metrics(name, pred):
    print(
        f"{name:22s} "
        f"RMSE={np.sqrt(mean_squared_error(y, pred)):.4f} | "
        f"MAE={mean_absolute_error(y, pred):.4f} | "
        f"Pearson={pearsonr(y, pred).statistic:.4f}"
    )

metrics("Raw HGB OOF", raw)
metrics("Cross-fitted calibration", calibrated)

print("\nCalibration fold coefficients:")
for fold, (train_idx, valid_idx) in enumerate(
    KFold(n_splits=5, shuffle=True, random_state=2026).split(raw), 1
):
    model = LinearRegression().fit(
        raw[train_idx].reshape(-1, 1), y[train_idx]
    )
    print(
        f"Fold {fold}: intercept={model.intercept_:.4f}, "
        f"slope={model.coef_[0]:.4f}"
    )

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"

source_path = out / "submission_final_hgb_audio_linguistic.csv"
target_path = out / "submission_calibrated_hgb_audio_linguistic.csv"

submission = pd.read_csv(source_path)
assert submission.columns.tolist() == ["filename", "label"]
assert len(submission) == 216
assert submission["label"].between(0, 5).all()

calibrated = submission.copy()
calibrated["label"] = np.clip(
    -0.1919 + 1.0611 * submission["label"].to_numpy(),
    0,
    5,
)

calibrated.to_csv(target_path, index=False)

print("Calibrated candidate saved:", target_path)
print("Original mean:", round(submission["label"].mean(), 4))
print("Calibrated mean:", round(calibrated["label"].mean(), 4))
print("Original std:", round(submission["label"].std(), 4))
print("Calibrated std:", round(calibrated["label"].std(), 4))
print(calibrated.head(10).to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"

print("EXISTING SUBMISSION FILES")
for path in sorted(out.glob("submission*.csv")):
    try:
        df = pd.read_csv(path)
        print(
            f"{path.name:48s} "
            f"rows={len(df):3d} "
            f"columns={df.columns.tolist()}"
        )
    except Exception as e:
        print(path.name, "READ ERROR:", e)

print("\nEXISTING MODEL FILES")
for path in sorted(out.glob("*.joblib")):
    print(f"{path.name:48s} {path.stat().st_size / 1024:.1f} KB")

print("\nEXISTING OOF / VALIDATION FILES")
for path in sorted(out.glob("*oof*.csv")):
    print(path.name)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"

df = pd.read_csv(out / "model_tuning_oof_predictions.csv")
y = df["actual"].to_numpy()

results = []

for col in df.columns:
    if col == "actual":
        continue

    pred = df[col].to_numpy()
    results.append({
        "Model": col,
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "MAE": mean_absolute_error(y, pred),
        "Pearson": pearsonr(y, pred).statistic,
    })

results = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

print("Saved OOF model comparison — lower RMSE/MAE is better")
display(results.round(4))

print("\nCurrent controlled HGB reference:")
controlled = pd.read_csv(out / "controlled_oof_predictions.csv")
for col in ["oof_audio", "oof_audio_linguistic"]:
    pred = controlled[col].to_numpy()
    actual = controlled["actual"].to_numpy()
    print(
        f"{col}: RMSE={np.sqrt(mean_squared_error(actual, pred)):.4f}, "
        f"MAE={mean_absolute_error(actual, pred):.4f}, "
        f"Pearson={pearsonr(actual, pred).statistic:.4f}"
    )

In [ ]:
try:
    import catboost
    print("CatBoost is installed.")
    print("Version:", catboost.__version__)
except ImportError:
    print("CatBoost is not installed.")

In [ ]:
%pip install catboost

In [ ]:
import catboost
print("CatBoost version:", catboost.__version__)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

# Paths
project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
features_dir = project / "features"
data_dir = project / "data" / "Dataset_Final"
out = project / "outputs"

# Load cached features
X_wavlm = np.load(features_dir / "X_train_wavlm.npy")
X_acoustic = np.load(features_dir / "X_train_acoustic.npy")
y = np.load(features_dir / "y_train.npy")

train_df = pd.read_csv(data_dir / "train.csv")
ling = pd.read_csv(out / "transcripts" / "train_linguistic_features.csv")

ling_cols = [
    "word_count", "sentence_count", "avg_words_per_sentence",
    "unique_word_count", "vocabulary_diversity",
    "repeated_adjacent_count", "repeated_word_rate",
    "filler_count", "incomplete_ending",
]

ling = train_df[["filename"]].merge(
    ling, on="filename", how="left", validate="one_to_one"
)

assert not ling[ling_cols].isna().any().any()
X = np.hstack([
    X_wavlm,
    X_acoustic,
    ling[ling_cols].to_numpy(dtype=float),
])

assert X.shape == (769, 824)
assert len(y) == len(train_df)

# Same five-fold split used for the existing controlled OOF results
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cat_oof = np.zeros(len(y), dtype=float)

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):
    model = CatBoostRegressor(
        iterations=800,
        depth=6,
        learning_rate=0.03,
        loss_function="RMSE",
        l2_leaf_reg=5,
        random_seed=42,
        verbose=False,
        thread_count=-1,
        allow_writing_files=False,
    )

    model.fit(
        X[train_idx],
        y[train_idx],
        eval_set=(X[valid_idx], y[valid_idx]),
        early_stopping_rounds=80,
        verbose=False,
    )

    cat_oof[valid_idx] = model.predict(X[valid_idx])
    fold_rmse = np.sqrt(mean_squared_error(y[valid_idx], cat_oof[valid_idx]))
    print(
        f"Fold {fold}: RMSE={fold_rmse:.4f}, "
        f"best_iteration={model.get_best_iteration()}"
    )

def report(name, actual, pred):
    print(
        f"{name:24s} "
        f"RMSE={np.sqrt(mean_squared_error(actual, pred)):.4f} | "
        f"MAE={mean_absolute_error(actual, pred):.4f} | "
        f"Pearson={pearsonr(actual, pred).statistic:.4f}"
    )

print("\nOverall OOF comparison")
report("CatBoost combined", y, cat_oof)

controlled = pd.read_csv(out / "controlled_oof_predictions.csv")
assert np.allclose(y, controlled["actual"].to_numpy())

report(
    "HGB audio + linguistic",
    y,
    controlled["oof_audio_linguistic"].to_numpy(),
)

# Save predictions for analysis; this does not create a submission
pd.DataFrame({
    "filename": train_df["filename"],
    "actual": y,
    "catboost_oof": cat_oof,
}).to_csv(out / "catboost_combined_oof.csv", index=False)

print("\nSaved OOF predictions to outputs/catboost_combined_oof.csv")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
features_dir = project / "features"
data_dir = project / "data" / "Dataset_Final"
out = project / "outputs"

# Load cached train and test features
X_train_wavlm = np.load(features_dir / "X_train_wavlm.npy")
X_test_wavlm = np.load(features_dir / "X_test_wavlm.npy")
X_train_acoustic = np.load(features_dir / "X_train_acoustic.npy")
X_test_acoustic = np.load(features_dir / "X_test_acoustic.npy")
y = np.load(features_dir / "y_train.npy")

train_df = pd.read_csv(data_dir / "train.csv")
test_df = pd.read_csv(data_dir / "test.csv")

ling_train = pd.read_csv(out / "transcripts" / "train_linguistic_features.csv")
ling_test = pd.read_csv(out / "transcripts" / "test_linguistic_features.csv")

ling_cols = [
    "word_count", "sentence_count", "avg_words_per_sentence",
    "unique_word_count", "vocabulary_diversity",
    "repeated_adjacent_count", "repeated_word_rate",
    "filler_count", "incomplete_ending",
]

ling_train = train_df[["filename"]].merge(
    ling_train, on="filename", how="left", validate="one_to_one"
)
ling_test = test_df[["filename"]].merge(
    ling_test, on="filename", how="left", validate="one_to_one"
)

assert not ling_train[ling_cols].isna().any().any()
assert not ling_test[ling_cols].isna().any().any()

X_train = np.hstack([
    X_train_wavlm,
    X_train_acoustic,
    ling_train[ling_cols].to_numpy(dtype=float),
])
X_test = np.hstack([
    X_test_wavlm,
    X_test_acoustic,
    ling_test[ling_cols].to_numpy(dtype=float),
])

assert X_train.shape == (769, 824)
assert X_test.shape == (216, 824)

# Train the final model using all labeled samples
model = CatBoostRegressor(
    iterations=800,
    depth=6,
    learning_rate=0.03,
    loss_function="RMSE",
    l2_leaf_reg=5,
    random_seed=42,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False,
)
model.fit(X_train, y)

pred = np.clip(model.predict(X_test), 0, 5)

submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": pred,
})

assert len(submission) == 216
assert submission["filename"].equals(test_df["filename"])
assert submission["filename"].is_unique
assert submission["label"].notna().all()
assert submission["label"].between(0, 5).all()

save_path = out / "submission_catboost_combined.csv"
submission.to_csv(save_path, index=False)

print("\nSaved:", save_path)
print("Rows:", len(submission))
print("Mean:", round(submission["label"].mean(), 4))
print("Std:", round(submission["label"].std(), 4))
print("Min:", round(submission["label"].min(), 4))
print("Max:", round(submission["label"].max(), 4))
print(submission.head(10).to_string(index=False))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"

cat = pd.read_csv(out / "catboost_combined_oof.csv")
hgb = pd.read_csv(out / "controlled_oof_predictions.csv")

# Align HGB predictions to CatBoost's filename order
hgb = cat[["filename"]].merge(
    hgb[["filename", "actual", "oof_audio_linguistic"]],
    on="filename",
    how="left",
    validate="one_to_one",
)

assert hgb["oof_audio_linguistic"].notna().all()
y = cat["actual"].to_numpy()
cat_pred = cat["catboost_oof"].to_numpy()
hgb_pred = hgb["oof_audio_linguistic"].to_numpy()

assert np.allclose(y, hgb["actual"].to_numpy())

def evaluate(name, pred):
    return {
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "MAE": mean_absolute_error(y, pred),
        "Pearson": pearsonr(y, pred).statistic,
    }

results = [
    evaluate("CatBoost", cat_pred),
    evaluate("HGB combined", hgb_pred),
]

for cat_weight in np.arange(0.1, 1.0, 0.1):
    blend = cat_weight * cat_pred + (1 - cat_weight) * hgb_pred
    results.append(
        evaluate(f"CatBoost {cat_weight:.1f} + HGB {1-cat_weight:.1f}", blend)
    )

results = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print(results.round(4).to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"

cat = pd.read_csv(out / "submission_catboost_combined.csv")
hgb = pd.read_csv(out / "submission_final_hgb_audio_linguistic.csv")
test_df = pd.read_csv(project / "data" / "Dataset_Final" / "test.csv")

# Align HGB predictions to CatBoost filename order
hgb = cat[["filename"]].merge(
    hgb,
    on="filename",
    how="left",
    validate="one_to_one",
)

assert len(cat) == len(test_df) == 216
assert cat["filename"].equals(test_df["filename"])
assert hgb["label"].notna().all()
assert hgb["filename"].equals(cat["filename"])

# Blend predictions
blend = cat.copy()
blend["label"] = (
    0.70 * cat["label"].to_numpy()
    + 0.30 * hgb["label"].to_numpy()
)

blend["label"] = np.clip(blend["label"], 0, 5)

assert blend["label"].between(0, 5).all()
assert blend["filename"].is_unique

save_path = out / "submission_blend_catboost70_hgb30.csv"
blend.to_csv(save_path, index=False)

print("Saved:", save_path)
print("Rows:", len(blend))
print("Mean:", round(blend["label"].mean(), 4))
print("Std:", round(blend["label"].std(), 4))
print(blend.head(10).to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"

cat = pd.read_csv(out / "submission_catboost_combined.csv")
hgb = pd.read_csv(out / "submission_final_hgb_audio_linguistic.csv")
test_df = pd.read_csv(project / "data" / "Dataset_Final" / "test.csv")

# Align both prediction files by filename
hgb = cat[["filename"]].merge(
    hgb, on="filename", how="left", validate="one_to_one"
)

assert cat["filename"].equals(test_df["filename"])
assert hgb["label"].notna().all()

# Candidate: 60% CatBoost + 40% HGB
blend = cat.copy()
blend["label"] = np.clip(
    0.60 * cat["label"].to_numpy()
    + 0.40 * hgb["label"].to_numpy(),
    0, 5
)

assert len(blend) == 216
assert blend["filename"].equals(test_df["filename"])
assert blend["label"].between(0, 5).all()

save_path = out / "submission_blend_catboost60_hgb40.csv"
blend.to_csv(save_path, index=False)

print("Saved:", save_path)
print("Rows:", len(blend))
print("Mean:", round(blend["label"].mean(), 4))
print("Std:", round(blend["label"].std(), 4))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
df = pd.read_csv(project / "outputs" / "catboost_combined_oof.csv")

y = df["actual"].to_numpy()
raw = df["catboost_oof"].to_numpy()
calibrated = np.empty_like(raw, dtype=float)

kf = KFold(n_splits=5, shuffle=True, random_state=2026)

for train_idx, valid_idx in kf.split(raw):
    calibrator = LinearRegression()
    calibrator.fit(raw[train_idx].reshape(-1, 1), y[train_idx])
    calibrated[valid_idx] = calibrator.predict(
        raw[valid_idx].reshape(-1, 1)
    )

calibrated = np.clip(calibrated, 0, 5)

def report(name, pred):
    print(
        f"{name:24s} "
        f"RMSE={np.sqrt(mean_squared_error(y, pred)):.4f} | "
        f"MAE={mean_absolute_error(y, pred):.4f} | "
        f"Pearson={pearsonr(y, pred).statistic:.4f}"
    )

report("Raw CatBoost OOF", raw)
report("Calibrated CatBoost OOF", calibrated)

print("\nCalibration coefficients:")
for fold, (train_idx, valid_idx) in enumerate(
    KFold(n_splits=5, shuffle=True, random_state=2026).split(raw), 1
):
    model = LinearRegression().fit(
        raw[train_idx].reshape(-1, 1), y[train_idx]
    )
    print(
        f"Fold {fold}: intercept={model.intercept_:.4f}, "
        f"slope={model.coef_[0]:.4f}"
    )

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"

source = out / "submission_catboost_combined.csv"
target = out / "submission_calibrated_catboost.csv"

submission = pd.read_csv(source)
test_df = pd.read_csv(project / "data" / "Dataset_Final" / "test.csv")

assert len(submission) == 216
assert submission["filename"].equals(test_df["filename"])
assert submission["label"].between(0, 5).all()

# Average of the five cross-fitted calibration coefficients
intercept = -0.35514
slope = 1.10728

calibrated = submission.copy()
calibrated["label"] = np.clip(
    intercept + slope * submission["label"].to_numpy(),
    0,
    5,
)

assert calibrated["filename"].equals(test_df["filename"])
assert calibrated["label"].between(0, 5).all()
assert calibrated["label"].notna().all()

calibrated.to_csv(target, index=False)

print("Saved:", target)
print("Rows:", len(calibrated))
print("Raw mean:", round(submission["label"].mean(), 4))
print("Calibrated mean:", round(calibrated["label"].mean(), 4))
print("Raw std:", round(submission["label"].std(), 4))
print("Calibrated std:", round(calibrated["label"].std(), 4))
print(calibrated.head(10).to_string(index=False))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"
test_df = pd.read_csv(project / "data" / "Dataset_Final" / "test.csv")

# Load the existing predictions
cat_cal = pd.read_csv(out / "submission_calibrated_catboost.csv")
hgb = pd.read_csv(out / "submission_final_hgb_audio_linguistic.csv")

# Align HGB rows to calibrated CatBoost filename order
hgb = cat_cal[["filename"]].merge(
    hgb,
    on="filename",
    how="left",
    validate="one_to_one",
)

# Validate inputs before blending
assert len(cat_cal) == len(test_df) == 216
assert cat_cal["filename"].equals(test_df["filename"])
assert hgb["label"].notna().all()
assert hgb["filename"].equals(cat_cal["filename"])

# Blend: 80% calibrated CatBoost + 20% HGB
blend = cat_cal.copy()
blend["label"] = np.clip(
    0.80 * cat_cal["label"].to_numpy()
    + 0.20 * hgb["label"].to_numpy(),
    0,
    5,
)

assert blend["label"].notna().all()
assert blend["label"].between(0, 5).all()
assert blend["filename"].equals(test_df["filename"])

save_path = out / "submission_blend_calibrated_catboost80_hgb20.csv"
blend.to_csv(save_path, index=False)

print("Saved:", save_path)
print("Rows:", len(blend))
print("Mean:", round(blend["label"].mean(), 4))
print("Std:", round(blend["label"].std(), 4))
print(blend.head(10).to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

# Locate project files
project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"

best_file = out / "submission_blend_calibrated_catboost80_hgb20.csv"
test_file = project / "data" / "Dataset_Final" / "test.csv"

# Load files
submission = pd.read_csv(best_file)
test_df = pd.read_csv(test_file)

# Validate submission structure
assert list(submission.columns) == ["filename", "label"], \
    f"Unexpected columns: {submission.columns.tolist()}"

assert len(submission) == len(test_df) == 216, \
    f"Row count mismatch: submission={len(submission)}, test={len(test_df)}"

assert submission["filename"].equals(test_df["filename"]), \
    "Filename order does not match test.csv"

assert submission["label"].notna().all(), "Missing predictions found"
assert np.isfinite(submission["label"]).all(), "Non-finite predictions found"
assert submission["label"].between(0, 5).all(), "Predictions outside [0, 5]"

# Preserve a separate copy of the winning submission
backup_file = out / "FINAL_BEST_submission_0.4897.csv"
shutil.copy2(best_file, backup_file)

print("VALIDATION PASSED")
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
print("Prediction range:",
      round(submission["label"].min(), 4),
      "to",
      round(submission["label"].max(), 4))
print("Mean prediction:", round(submission["label"].mean(), 4))
print("Best public score: 0.4897")
print("Backup saved to:", backup_file)

In [ ]:
from pathlib import Path

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"

folders_to_check = [
    project,
    project / "notebooks",
    project / "src",
    project / "outputs",
    project / "features",
    project / "data",
]

for folder in folders_to_check:
    print(f"\n{'=' * 60}")
    print(f"FOLDER: {folder}")

    if not folder.exists():
        print("Not found")
        continue

    for item in sorted(folder.iterdir()):
        if item.is_file():
            size_mb = item.stat().st_size / (1024 * 1024)
            print(f"  FILE  {item.name} ({size_mb:.2f} MB)")
        else:
            print(f"  DIR   {item.name}/")

print("\nPROJECT ROOT EXISTS:", project.exists())

In [ ]:
from pathlib import Path
import json

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
notebook_path = project / "notebooks" / "01_baseline.ipynb"
requirements_path = project / "requirements.txt"

# Inspect notebook structure and code cells
with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("NOTEBOOK CELL SUMMARY")
for i, cell in enumerate(notebook.get("cells", [])):
    source = "".join(cell.get("source", []))
    if cell.get("cell_type") == "markdown":
        print(f"\n[Markdown {i}] {source[:250].replace(chr(10), ' ')}")
    else:
        first_line = source.strip().splitlines()[0] if source.strip() else "(empty)"
        print(f"\n[Code {i}] {first_line[:180]}")

print("\n\nREQUIREMENTS.TXT")
print(requirements_path.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
requirements_path = project / "requirements.txt"

raw = requirements_path.read_bytes()

if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
    requirements = raw.decode("utf-16")
elif raw.startswith(b"\xef\xbb\xbf"):
    requirements = raw.decode("utf-8-sig")
else:
    requirements = raw.decode("utf-8")

print("REQUIREMENTS.TXT")
print("-" * 40)
print(requirements)

In [ ]:
import ast
import json
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
notebook_path = project / "notebooks" / "01_baseline.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

imports = set()

for cell in notebook.get("cells", []):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    try:
        tree = ast.parse(source)
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                imports.update(alias.name.split(".")[0] for alias in node.names)
            elif isinstance(node, ast.ImportFrom) and node.module:
                imports.add(node.module.split(".")[0])
    except SyntaxError:
        pass

print("NOTEBOOK IMPORTS")
print(", ".join(sorted(imports)))

packages = [
    "numpy", "pandas", "scikit-learn", "torch", "transformers",
    "librosa", "catboost", "joblib", "soundfile", "openai-whisper"
]

print("\nMAIN PACKAGE VERSIONS")
for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed in this notebook environment")

In [ ]:
from pathlib import Path

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"

requirements = """\
numpy==2.5.3
pandas==3.0.6
scikit-learn==1.9.1
torch==2.14.1
transformers==5.19.0
librosa==1.0.0
catboost==1.2.10
joblib==1.6.0
soundfile==0.14.0
scipy
matplotlib
"""

requirements_path = project / "requirements-github.txt"
requirements_path.write_text(requirements, encoding="utf-8")

print("Created:", requirements_path)
print("\nContents:")
print(requirements_path.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"

gitignore_content = r"""
# Python environments
.venv/
venv/
env/
__pycache__/
*.py[cod]

# Jupyter temporary files
.ipynb_checkpoints/

# Competition data and audio
data/
*.wav
*.mp3
*.flac

# Generated features and caches
features/
checkpoints/
logs/
outputs/acoustic_cache/
outputs/wavlm_cache/
outputs/transcripts/

# Model artifacts
*.joblib
*.pkl
*.pickle
*.pt
*.pth
*.safetensors

# Private competition submissions and predictions
submissions/
outputs/submission*.csv
outputs/FINAL_BEST*.csv
outputs/*oof*.csv
outputs/*predictions*.csv
outputs/*error*.csv

# Secrets and local configuration
.env
.env.*
*.key

# OS and editor files
.DS_Store
Thumbs.db
.vscode/
"""

gitignore_path = project / ".gitignore"
gitignore_path.write_text(gitignore_content.strip() + "\n", encoding="utf-8")

print("Created:", gitignore_path)
print("\nGit ignore rules:")
print(gitignore_path.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import subprocess

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"

def run_git(args):
    result = subprocess.run(
        ["git", *args],
        cwd=project,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )
    return result

check = run_git(["rev-parse", "--is-inside-work-tree"])

if check.returncode != 0:
    print("This folder is not yet a Git repository.")
    print("No files were changed.")
else:
    print("GIT STATUS")
    status = run_git(["status", "--short", "--branch"])
    print(status.stdout or "(clean working tree)")

    print("\nIGNORED FILE CHECK")
    ignored = run_git([
        "check-ignore", "-v",
        "data",
        ".venv",
        "features",
        "checkpoints",
        "outputs/FINAL_BEST_submission_0.4897.csv",
        "submissions"
    ])
    print(ignored.stdout or "No matching ignored paths reported.")

In [ ]:
import shutil
import subprocess
from pathlib import Path

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"

git_path = shutil.which("git")

print("Project folder exists:", project.exists())
print("Git executable:", git_path or "NOT FOUND")

if git_path:
    result = subprocess.run(
        [git_path, "--version"],
        capture_output=True,
        text=True
    )
    print("Git version:", result.stdout.strip())
else:
    print(
        "\nGit is not available on this notebook's PATH.\n"
        "We need to install Git for Windows or add its executable to PATH."
    )

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
data = project / "data" / "Dataset_Final"
out = project / "outputs"

train = pd.read_csv(data / "train.csv")
test = pd.read_csv(data / "test.csv")
sample = pd.read_csv(data / "sample_submission.csv")
best_path = out / "FINAL_BEST_submission_0.4897.csv"

print("=== DATA SHAPES ===")
print(f"Train: {train.shape} | Test: {test.shape} | Sample submission: {sample.shape}")
print("\n=== TRAIN COLUMNS ===")
print(train.columns.tolist())
print("\n=== LABEL DISTRIBUTION ===")
y = pd.to_numeric(train["label"], errors="coerce")
print("Missing/non-numeric labels:", int(y.isna().sum()))
print("Unique label count:", int(y.nunique()))
print("Exact label counts (first 30):")
print(y.value_counts(dropna=False).sort_index().head(30).to_string())
print(f"Mean={y.mean():.4f} | SD={y.std():.4f} | Min={y.min():.4f} | Max={y.max():.4f}")
print("Labels near zero (<=0.5):", int((y <= 0.5).sum()))

print("\n=== FILENAME SET COMPARISON ===")
for name, df in [("train", train), ("test", test), ("sample_submission", sample)]:
    print(f"{name}: filename column present = {'filename' in df.columns}")
    if "filename" in df.columns:
        print(f"  rows={len(df)}, unique filenames={df['filename'].nunique()}, duplicates={df['filename'].duplicated().sum()}")

if "filename" in test.columns and "filename" in sample.columns:
    test_names = set(test["filename"].astype(str))
    sample_names = set(sample["filename"].astype(str))
    print("In test but missing from sample:", len(test_names - sample_names))
    print(sorted(test_names - sample_names)[:20])
    print("In sample but missing from test:", len(sample_names - test_names))
    print(sorted(sample_names - test_names)[:20])

print("\n=== SAVED WINNING SUBMISSION ===")
if best_path.exists():
    best = pd.read_csv(best_path)
    print("Found:", best_path.name, "| shape:", best.shape)
    if "label" in best.columns:
        p = pd.to_numeric(best["label"], errors="coerce")
        print(f"Prediction min={p.min():.4f} | max={p.max():.4f} | mean={p.mean():.4f}")
else:
    print("Backup file not found at expected path.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
best_path = ROOT / "outputs" / "FINAL_BEST_submission_0.4897.csv"

def normalize_filename(series):
    return series.astype(str).str.strip().str.replace("\\", "/", regex=False).str.split("/").str[-1].str.lower()

train_names = normalize_filename(train["filename"])
test_names = normalize_filename(test["filename"])
sample_names = normalize_filename(sample["filename"])

print("=== DUPLICATES ===")
print("Train:", int(train_names.duplicated().sum()))
print("Test:", int(test_names.duplicated().sum()))
print("Sample submission:", int(sample_names.duplicated().sum()))

train_set, test_set, sample_set = set(train_names), set(test_names), set(sample_names)

print("\n=== FILENAME OVERLAPS ===")
print("Train ∩ Test:", len(train_set & test_set))
print("Test ∩ Sample:", len(test_set & sample_set))
print("Test files missing from sample:", len(test_set - sample_set))
print("Sample files absent from test:", len(sample_set - test_set))

print("\nFirst 25 test files missing from sample:")
print(sorted(test_set - sample_set)[:25])

print("\nFirst 25 sample files absent from test:")
print(sorted(sample_set - test_set)[:25])

print("\n=== SAVED BEST SUBMISSION ===")
if best_path.exists():
    best = pd.read_csv(best_path)
    print("Columns:", best.columns.tolist(), "| Shape:", best.shape)

    if "filename" in best.columns and "label" in best.columns:
        best_names = normalize_filename(best["filename"])
        best_set = set(best_names)

        print("Duplicate filenames:", int(best_names.duplicated().sum()))
        print("Saved submission files missing from test:", len(best_set - test_set))
        print("Test files missing from saved submission:", len(test_set - best_set))
        print("Filename order exactly matches test:", best_names.tolist() == test_names.tolist())
        print("Missing predictions:", int(best["label"].isna().sum()))
        print("Non-finite predictions:", int((~np.isfinite(pd.to_numeric(best["label"], errors="coerce"))).sum()))
        print("Predictions outside [0, 5]:", int(((best["label"] < 0) | (best["label"] > 5)).sum()))
else:
    print("Saved submission not found:", best_path)

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import pandas as pd

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")

# Index WAV files by lowercase basename.
audio_index = defaultdict(list)
for p in DATA.rglob("*.wav"):
    audio_index[p.name.strip().lower()].append(p)

train_names = set(train["filename"].astype(str).str.strip().str.lower())
test_names = set(test["filename"].astype(str).str.strip().str.lower())
overlap = sorted(train_names & test_names)

print("=== AUDIO FILE LOCATIONS ===")
print("WAV files indexed:", sum(map(len, audio_index.values())))
print("Unique WAV basenames:", len(audio_index))
print("Train CSV filenames found on disk:",
      sum(name in audio_index for name in train_names))
print("Test CSV filenames found on disk:",
      sum(name in audio_index for name in test_names))

print("\n=== EXAMPLE PATHS ===")
for name in overlap[:8]:
    print(name, "->", [str(p.relative_to(ROOT)) for p in audio_index.get(name, [])[:4]])

# Hash only shared-basename files, to check exact content equality.
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

identical_pairs = []
ambiguous_names = []
unmatched_names = []

for name in overlap:
    paths = audio_index.get(name, [])
    if len(paths) < 2:
        unmatched_names.append(name)
        continue
    if len(paths) > 2:
        ambiguous_names.append((name, len(paths)))
    hashes = {sha256(p) for p in paths}
    if len(hashes) == 1:
        identical_pairs.append(name)

print("\n=== SAME-BASENAME CONTENT CHECK ===")
print("Overlapping CSV basenames:", len(overlap))
print("Names with fewer than 2 matching disk paths:", len(unmatched_names))
print("Names with more than 2 matching disk paths:", len(ambiguous_names))
print("Names whose matching files are all byte-identical:", len(identical_pairs))
print("Examples of byte-identical names:", identical_pairs[:15])
print("Examples of ambiguous names:", ambiguous_names[:10])

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")

def audio_path(split, filename):
    return DATA / split / Path(str(filename)).name

def inspect_audio(split, df):
    rows = []
    for _, row in df.iterrows():
        p = audio_path(split, row["filename"])
        try:
            info = sf.info(str(p))
            rows.append({
                "filename": Path(str(row["filename"])).name,
                "label": row["label"] if "label" in df.columns else np.nan,
                "duration_s": info.duration,
                "sample_rate": info.samplerate,
                "channels": info.channels,
                "subtype": info.subtype,
                "size_mb": p.stat().st_size / (1024 ** 2),
                "read_ok": True,
            })
        except Exception as e:
            rows.append({
                "filename": Path(str(row["filename"])).name,
                "label": row["label"] if "label" in df.columns else np.nan,
                "read_ok": False,
                "error": str(e)[:120],
            })
    return pd.DataFrame(rows)

train_meta = inspect_audio("train", train)
test_meta = inspect_audio("test", test)

print("=== READABILITY ===")
print("Train metadata failures:", int((~train_meta["read_ok"]).sum()))
print("Test metadata failures:", int((~test_meta["read_ok"]).sum()))

for name, meta in [("TRAIN", train_meta), ("TEST", test_meta)]:
    ok = meta[meta["read_ok"]]
    print(f"\n=== {name} AUDIO METADATA ===")
    print("Sample rates:\n", ok["sample_rate"].value_counts().sort_index().to_string())
    print("Channels:\n", ok["channels"].value_counts().sort_index().to_string())
    print("Subtype counts:\n", ok["subtype"].value_counts().to_string())
    print("Duration (s):\n", ok["duration_s"].describe(
        percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
    ).round(3).to_string())
    print("File size (MB):\n", ok["size_mb"].describe().round(3).to_string())

print("\n=== ZERO-SCORE AUDIO ===")
zero = train_meta[(train_meta["read_ok"]) & (pd.to_numeric(train_meta["label"], errors="coerce") == 0)]
print("Count:", len(zero))
print(zero[["filename", "duration_s", "sample_rate", "channels", "subtype", "size_mb"]]
      .sort_values("duration_s").head(15).round(3).to_string(index=False))

print("\n=== ZERO-SCORE VS OTHER TRAINING AUDIO ===")
train_meta["label_num"] = pd.to_numeric(train_meta["label"], errors="coerce")
train_meta["zero_score"] = train_meta["label_num"].eq(0)
print(train_meta.groupby("zero_score")[["duration_s", "size_mb"]].agg(["count", "mean", "median", "min", "max"]).round(3).to_string())

print("\n=== TRAIN–TEST METADATA SHIFT ===")
for col in ["duration_s", "size_mb"]:
    a = train_meta.loc[train_meta["read_ok"], col].dropna()
    b = test_meta.loc[test_meta["read_ok"], col].dropna()
    print(f"{col}: train median={a.median():.3f}, test median={b.median():.3f}, "
          f"train mean={a.mean():.3f}, test mean={b.mean():.3f}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.stats import ks_2samp, spearmanr

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")

def get_durations(split, df):
    values = []
    for filename in df["filename"]:
        path = DATA / split / Path(str(filename)).name
        values.append(sf.info(str(path)).duration)
    return np.asarray(values, dtype=float)

train["duration_s"] = get_durations("train", train)
test["duration_s"] = get_durations("test", test)

print("=== LABEL VS DURATION ===")
print("Spearman correlation:", round(spearmanr(train["label"], train["duration_s"]).statistic, 4))
print("\nDuration by label:")
print(train.groupby("label")["duration_s"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2).to_string())

print("\n=== ZERO VS NONZERO SCORES ===")
train["score_group"] = np.where(train["label"].eq(0), "zero", "nonzero")
print(train.groupby("score_group")["duration_s"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2).to_string())

zero = train.loc[train["label"].eq(0), "duration_s"]
nonzero = train.loc[train["label"].ne(0), "duration_s"]
print("Zero vs nonzero duration KS statistic:",
      round(ks_2samp(zero, nonzero).statistic, 4))
print("Zero vs nonzero duration KS p-value:",
      round(ks_2samp(zero, nonzero).pvalue, 6))

print("\n=== TRAIN–TEST DURATION SHIFT ===")
print("Train duration quantiles:")
print(train["duration_s"].quantile([0, .1, .25, .5, .75, .9, 1]).round(2).to_string())
print("Test duration quantiles:")
print(test["duration_s"].quantile([0, .1, .25, .5, .75, .9, 1]).round(2).to_string())

ks = ks_2samp(train["duration_s"], test["duration_s"])
print(f"Train vs test KS statistic={ks.statistic:.4f}, p-value={ks.pvalue:.6g}")
print("Test duration below 50 seconds:",
      f"{(test['duration_s'] < 50).mean():.1%}")
print("Train duration below 50 seconds:",
      f"{(train['duration_s'] < 50).mean():.1%}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"
FEATURES = ROOT / "features"
LING = ROOT / "outputs" / "transcripts"

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")

def load_array(name):
    path = FEATURES / name
    if not path.exists():
        print(f"MISSING: {path}")
        return None
    arr = np.load(path, allow_pickle=False)
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}, "
          f"non-finite={int((~np.isfinite(arr)).sum())}")
    return arr

print("=== CACHED ARRAY INTEGRITY ===")
Xw_train = load_array("X_train_wavlm.npy")
Xw_test  = load_array("X_test_wavlm.npy")
Xa_train = load_array("X_train_acoustic.npy")
Xa_test  = load_array("X_test_acoustic.npy")
y_cache  = load_array("y_train.npy")

print("\n=== ROW-COUNT / TARGET ALIGNMENT ===")
for name, arr, expected in [
    ("WavLM train", Xw_train, len(train)),
    ("WavLM test", Xw_test, len(test)),
    ("Acoustic train", Xa_train, len(train)),
    ("Acoustic test", Xa_test, len(test)),
    ("Cached target", y_cache, len(train)),
]:
    if arr is not None:
        print(f"{name}: rows={arr.shape[0]}, expected={expected}, "
              f"match={arr.shape[0] == expected}")

if y_cache is not None:
    y_flat = np.asarray(y_cache).reshape(-1)
    y_csv = train["label"].to_numpy(dtype=float)
    print("Cached y exactly matches train.csv order:",
          len(y_flat) == len(y_csv) and np.allclose(y_flat, y_csv))

print("\n=== FEATURE DISTRIBUTION SHIFT ===")
def compare_features(name, A, B):
    if A is None or B is None or A.ndim != 2 or B.ndim != 2:
        print(name, ": skipped (missing or invalid array)")
        return
    if A.shape[1] != B.shape[1]:
        print(name, ": skipped (feature dimensions differ)")
        return

    # Compare per-feature means and standard deviations.
    mean_a, mean_b = np.nanmean(A, axis=0), np.nanmean(B, axis=0)
    std_a, std_b = np.nanstd(A, axis=0), np.nanstd(B, axis=0)
    pooled = np.sqrt((std_a**2 + std_b**2) / 2)
    valid = np.isfinite(mean_a) & np.isfinite(mean_b) & (pooled > 1e-12)
    effect = np.zeros_like(mean_a, dtype=float)
    effect[valid] = (mean_b[valid] - mean_a[valid]) / pooled[valid]

    print(f"{name}:")
    print("  Features:", A.shape[1])
    print("  Features with |standardized mean shift| > 0.5:",
          int((np.abs(effect[valid]) > 0.5).sum()))
    print("  Features with |standardized mean shift| > 1.0:",
          int((np.abs(effect[valid]) > 1.0).sum()))
    if valid.any():
        print("  Median absolute standardized shift:",
              round(float(np.median(np.abs(effect[valid]))), 4))
        print("  Maximum absolute standardized shift:",
              round(float(np.max(np.abs(effect[valid]))), 4))

compare_features("WavLM", Xw_train, Xw_test)
compare_features("Acoustic", Xa_train, Xa_test)

print("\n=== LINGUISTIC FEATURE CSV ALIGNMENT ===")
for split, expected in [("train", train), ("test", test)]:
    path = LING / f"{split}_linguistic_features.csv"
    if not path.exists():
        print(f"{split}: missing {path}")
        continue

    df = pd.read_csv(path)
    print(f"{split}: shape={df.shape}, columns={df.columns.tolist()}")

    if "filename" in df.columns:
        expected_names = expected["filename"].astype(str).str.strip().str.lower().tolist()
        actual_names = df["filename"].astype(str).str.strip().str.lower().tolist()
        print("  Filename order matches CSV:", actual_names == expected_names)
        print("  Duplicate linguistic filenames:", int(df["filename"].duplicated().sum()))
        print("  Missing expected filenames:",
              len(set(expected_names) - set(actual_names)))
        print("  Unexpected filenames:",
              len(set(actual_names) - set(expected_names)))
    else:
        print("  WARNING: no filename column; row alignment cannot be verified here.")
        

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
SEARCH_ROOTS = [ROOT / "notebooks", ROOT / "src"]

terms = (
    "X_train_wavlm", "X_test_wavlm",
    "X_train_acoustic", "X_test_acoustic",
    "np.save", "sorted(", "os.listdir",
    "glob(", "train.csv", "test.csv",
    "iterrows", "filename"
)

print("=== FEATURE CREATION / ORDERING CODE ===")

for base in SEARCH_ROOTS:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if path.suffix == ".ipynb":
            try:
                notebook = json.loads(path.read_text(encoding="utf-8"))
            except Exception as e:
                print(f"\nCould not read {path.name}: {e}")
                continue

            print(f"\n--- NOTEBOOK: {path.relative_to(ROOT)} ---")
            for i, cell in enumerate(notebook.get("cells", [])):
                if cell.get("cell_type") != "code":
                    continue
                source = "".join(cell.get("source", []))
                if any(term in source for term in terms):
                    print(f"\n[CELL {i}]")
                    print(source[:3500])

        elif path.suffix == ".py":
            try:
                source = path.read_text(encoding="utf-8", errors="replace")
            except Exception:
                continue
            if any(term in source for term in terms):
                print(f"\n--- PYTHON FILE: {path.relative_to(ROOT)} ---")
                lines = source.splitlines()
                for i, line in enumerate(lines):
                    if any(term in line for term in terms):
                        start, end = max(0, i - 3), min(len(lines), i + 5)
                        print(f"\nLines {start + 1}-{end}:")
                        print("\n".join(
                            f"{j + 1}: {lines[j]}" for j in range(start, end)
                        ))

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
notebook_path = ROOT / "notebooks" / "01_baseline.ipynb"

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

keywords = (
    "X_train_wavlm", "X_test_wavlm",
    "X_train_acoustic", "X_test_acoustic",
    "wavlm_cache", "np.save", "np.load",
    "extract_wavlm", "extract_acoustic",
    "feature_rows", "audio_paths", "audio_files",
    "sorted(", "iterrows()", "os.listdir",
    "glob(", "train_files", "test_files",
    "train_embeddings", "test_embeddings",
)

print("Notebook code cells:", sum(c.get("cell_type") == "code" for c in nb["cells"]))

matches = []
for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue
    source = "".join(cell.get("source", []))
    if any(k.lower() in source.lower() for k in keywords):
        matches.append((i, source))

print("Matching cells:", len(matches))
for i, source in matches:
    lines = source.splitlines()
    relevant = [
        j for j, line in enumerate(lines)
        if any(k.lower() in line.lower() for k in keywords)
    ]
    keep = set()
    for j in relevant:
        keep.update(range(max(0, j - 2), min(len(lines), j + 3)))

    print(f"\n--- CELL {i} ({len(lines)} lines) ---")
    for j in sorted(keep):
        print(f"{j + 1}: {lines[j]}")

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
path = ROOT / "notebooks" / "01_baseline.ipynb"
nb = json.loads(path.read_text(encoding="utf-8"))

terms = [
    "X_train_wavlm.npy", "X_test_wavlm.npy",
    "X_train_acoustic.npy", "X_test_acoustic.npy",
    "y_train.npy",
    "X_train_wavlm", "X_test_wavlm",
    "X_train_acoustic", "X_test_acoustic",
    "np.save(", "np.load(",
]

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    hits = [term for term in terms if term.lower() in source.lower()]

    if hits:
        print(f"\n{'=' * 25} CELL {i} | MATCHES: {hits} {'=' * 25}")
        print(source)

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
nb_path = ROOT / "notebooks" / "01_baseline.ipynb"
nb = json.loads(nb_path.read_text(encoding="utf-8"))

source = "".join(nb["cells"][20]["source"])
lines = source.splitlines()

keywords = [
    "glob(", "rglob(", "sorted(", "iterrows", "filename",
    "train_audio", "test_audio", "train_files", "test_files",
    "np.save", "np.load", "wavlm", "embedding", "append(",
    "train.csv", "test.csv", "resume", "cache"
]

print("Cell 20 total lines:", len(lines))

shown = set()
for i, line in enumerate(lines):
    if any(k.lower() in line.lower() for k in keywords):
        shown.update(range(max(0, i - 3), min(len(lines), i + 4)))

for i in sorted(shown):
    print(f"{i + 1:03d}: {lines[i]}")

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
nb = json.loads(
    (ROOT / "notebooks" / "01_baseline.ipynb").read_text(encoding="utf-8")
)

lines = "".join(nb["cells"][20]["source"]).splitlines()

print("=== CELL 20: LINES 35–190 ===")
for i in range(34, min(190, len(lines))):
    print(f"{i + 1:03d}: {lines[i]}")

print("\n=== CELL 20: LINES 190–290 ===")
for i in range(189, min(290, len(lines))):
    print(f"{i + 1:03d}: {lines[i]}")

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
nb = json.loads(
    (ROOT / "notebooks" / "01_baseline.ipynb").read_text(encoding="utf-8")
)
lines = "".join(nb["cells"][20]["source"]).splitlines()

# Print the short sections responsible for manifests, file order,
# per-file cache naming, and assembling the final arrays.
ranges = [(57, 125), (126, 205), (205, 287), (288, 338)]

for start, end in ranges:
    print(f"\n=== LINES {start}–{end} ===")
    for i in range(start - 1, min(end, len(lines))):
        line = lines[i]
        # Keep output compact while preserving the relevant code.
        print(f"{i + 1:03d}: {line}")

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
nb = json.loads(
    (ROOT / "notebooks" / "01_baseline.ipynb").read_text(encoding="utf-8")
)
lines = "".join(nb["cells"][20]["source"]).splitlines()

terms = [
    "def extract_split", "def extract_one", "for _, row",
    "for filename", "df.iterrows", "df[", "row[",
    "glob(", "sorted(", "cache_path", "cache_file",
    "np.save", "np.load", "np.stack", "np.vstack",
    "np.concatenate", "manifest", "embedding_path",
    "train_failures", "test_failures",
]

hits = [i for i, line in enumerate(lines)
        if any(term.lower() in line.lower() for term in terms)]

shown = set()
for i in hits:
    shown.update(range(max(0, i - 1), min(len(lines), i + 2)))

print("=== ALIGNMENT-CRITICAL LINES ===")
for i in sorted(shown):
    print(f"{i + 1:03d}: {lines[i]}")

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
nb = json.loads(
    (ROOT / "notebooks" / "01_baseline.ipynb").read_text(encoding="utf-8")
)
lines = "".join(nb["cells"][20]["source"]).splitlines()

for start, end in [(80, 115), (145, 195), (295, 338)]:
    print(f"\n=== CELL 20, LINES {start}-{end} ===")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:03d}: {lines[i]}")

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
nb = json.loads(
    (ROOT / "notebooks" / "01_baseline.ipynb").read_text(encoding="utf-8")
)
lines = "".join(nb["cells"][20]["source"]).splitlines()

# Print exact sections: checkpoint validation and extraction/assembly.
for start, end in [(94, 125), (152, 185), (300, 330)]:
    print(f"\n=== LINES {start}-{end} ===")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:03d}: {lines[i]}")

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
nb = json.loads(
    (ROOT / "notebooks" / "01_baseline.ipynb").read_text(encoding="utf-8")
)
lines = "".join(nb["cells"][20]["source"]).splitlines()

for start, end in [(150, 180), (295, 326)]:
    print(f"\n=== CELL 20, LINES {start}-{end} ===")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:03d}: {lines[i]}")

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
nb = json.loads(
    (ROOT / "notebooks" / "01_baseline.ipynb").read_text(encoding="utf-8")
)
lines = "".join(nb["cells"][20]["source"]).splitlines()

# Inspect the remainder of extract_split and the final extraction calls.
for start, end in [(163, 230), (230, 295), (295, 338)]:
    print(f"\n=== CELL 20, LINES {start}-{end} ===")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:03d}: {lines[i]}")

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
nb = json.loads(
    (ROOT / "notebooks" / "01_baseline.ipynb").read_text(encoding="utf-8")
)
lines = "".join(nb["cells"][20]["source"]).splitlines()

keywords = [
    "valid_checkpoint", "np.load", "np.save",
    "completed", "skipped", "failures.append",
    "return ", "extract_split(", "embedding",
    "all_failures", "valid_train", "valid_test",
]

print("=== EXTRACTION AND ASSEMBLY LOGIC ===")
for i in range(180, min(338, len(lines))):
    line = lines[i]
    if any(k.lower() in line.lower() for k in keywords):
        print(f"{i + 1:03d}: {line}")

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
nb = json.loads(
    (ROOT / "notebooks" / "01_baseline.ipynb").read_text(encoding="utf-8")
)

terms = [
    "KFold", "StratifiedKFold", "GroupKFold",
    "CatBoostRegressor", "HistGradientBoostingRegressor",
    "cross_val_predict", "oof", "out_of_fold",
    "early_stopping_rounds", "eval_set",
    "calibrat", "intercept", "slope",
    "blend", "0.8", "0.7", "RandomForest",
    "Ridge", "SVR", "submission",
]

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    lines = "".join(cell.get("source", [])).splitlines()
    hits = [
        j for j, line in enumerate(lines)
        if any(term.lower() in line.lower() for term in terms)
    ]
    if not hits:
        continue

    shown = set()
    for j in hits:
        shown.update(range(max(0, j - 2), min(len(lines), j + 3)))

    print(f"\n=== CELL {i} ===")
    for j in sorted(shown):
        print(f"{j + 1:03d}: {lines[j]}")

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
nb = json.loads(
    (ROOT / "notebooks" / "01_baseline.ipynb").read_text(encoding="utf-8")
)

# Cell 24 defines a separate KFold workflow; inspect it in full.
print("========== CELL 24: FULL SOURCE ==========")
print("".join(nb["cells"][24].get("source", [])))

# Find cells containing the specific modeling and evaluation operations.
terms = [
    "CatBoost", "HistGradientBoosting", "out-of-fold",
    "oof_", "oof =", "StratifiedKFold", "KFold(",
    "early_stopping_rounds", "eval_set",
    "calibration", "IsotonicRegression",
    "LinearRegression", "blend_weights",
    "Pearson", "mean_squared_error",
]

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code" or i == 24:
        continue

    source = "".join(cell.get("source", []))
    matches = [term for term in terms if term.lower() in source.lower()]

    if matches:
        print(f"\n========== CELL {i} | MATCHES: {matches} ==========")
        print(source)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Project paths
ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"
FEATURES = ROOT / "features"
OUTPUTS = ROOT / "outputs"

train_df = pd.read_csv(DATA / "train.csv")
X_wavlm = np.load(FEATURES / "X_train_wavlm.npy")
X_acoustic = np.load(FEATURES / "X_train_acoustic.npy")
y = np.load(FEATURES / "y_train.npy")

# Align linguistic features explicitly by filename
ling = pd.read_csv(OUTPUTS / "transcripts" / "train_linguistic_features.csv")
ling = train_df[["filename"]].merge(
    ling, on="filename", how="left", validate="one_to_one"
)

ling_cols = [
    "word_count", "sentence_count", "avg_words_per_sentence",
    "unique_word_count", "vocabulary_diversity",
    "repeated_adjacent_count", "repeated_word_rate",
    "filler_count", "incomplete_ending",
]

assert not ling[ling_cols].isna().any().any()
assert len(y) == len(train_df) == len(X_wavlm) == len(X_acoustic)

X = np.hstack([
    X_wavlm,
    X_acoustic,
    ling[ling_cols].to_numpy(dtype=float),
])

assert X.shape == (769, 824)
assert np.isfinite(X).all()
assert np.isfinite(y).all()

# Same folds as the existing controlled comparison
cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Ridge_alpha100": make_pipeline(
        StandardScaler(),
        Ridge(alpha=100.0),
    ),
    "RBF_SVR_C10": make_pipeline(
        StandardScaler(),
        SVR(kernel="rbf", C=10.0, epsilon=0.2, gamma="scale"),
    ),
}

results = []

for name, estimator in models.items():
    oof = np.full(len(y), np.nan)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X), start=1):
        model = clone(estimator)
        model.fit(X[train_idx], y[train_idx])
        oof[valid_idx] = model.predict(X[valid_idx])

        fold_rmse = np.sqrt(
            mean_squared_error(y[valid_idx], oof[valid_idx])
        )
        print(f"{name} | fold {fold}/5 | RMSE={fold_rmse:.4f}")

    results.append({
        "Model": name,
        "OOF_RMSE": np.sqrt(mean_squared_error(y, oof)),
        "OOF_MAE": mean_absolute_error(y, oof),
        "OOF_Pearson": pearsonr(y, oof).statistic,
    })

    # Save predictions for later error analysis
    OUTPUTS.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        "filename": train_df["filename"],
        "actual": y,
        "prediction": oof,
    }).to_csv(OUTPUTS / f"{name}_oof.csv", index=False)

print("\n=== CONTROLLED MODEL COMPARISON ===")
print(pd.DataFrame(results).round(4).to_string(index=False))
print("\nReference results from the existing run:")
print("CatBoost combined: RMSE=0.6301 | MAE=0.4903 | Pearson=0.8649")
print("HGB combined:      RMSE=0.6421 | MAE=0.4984 | Pearson=0.8565")
print("70/30 blend:       RMSE=0.6256 | MAE=0.4882 | Pearson=0.8669")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"

train_df = pd.read_csv(DATA / "train.csv")
y = pd.to_numeric(train_df["label"], errors="raise").to_numpy()

print("=== TARGET INTEGRITY ===")
print(f"Rows: {len(y)}")
print(f"Missing labels: {np.isnan(y).sum()}")
print(f"Unique scores: {np.sort(np.unique(y))}")
print(f"Min={y.min():.3f} | Max={y.max():.3f} | "
      f"Mean={y.mean():.3f} | Std={y.std(ddof=1):.3f}")

print("\n=== EXACT SCORE COUNTS ===")
counts = (
    pd.Series(y)
    .value_counts()
    .sort_index()
    .rename_axis("score")
    .to_frame("count")
)
counts["percentage"] = 100 * counts["count"] / len(y)
print(counts.round(2).to_string())

print("\n=== LOW-SCORE COUNTS ===")
for threshold in [0, 0.25, 0.5, 1.0, 1.5, 2.0]:
    n = int((y <= threshold).sum())
    print(f"Score <= {threshold:>4}: {n:>3} ({100*n/len(y):5.2f}%)")

print("\n=== QUANTILES ===")
print(pd.Series(y).quantile(
    [0, .01, .05, .10, .25, .50, .75, .90, .95, .99, 1]
).to_string())

print("\n=== LOWEST-SCORE TRAINING ROWS ===")
cols = ["filename", "label"]
print(
    train_df.loc[y <= 0.5, cols]
    .sort_values("label")
    .to_string(index=False)
)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import mannwhitneyu

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"
FEATURES = ROOT / "features"
OUTPUTS = ROOT / "outputs"

train_df = pd.read_csv(DATA / "train.csv")
y = np.load(FEATURES / "y_train.npy")
wavlm = np.load(FEATURES / "X_train_wavlm.npy")
acoustic = np.load(FEATURES / "X_train_acoustic.npy")

ling_df = pd.read_csv(OUTPUTS / "transcripts" / "train_linguistic_features.csv")
ling = train_df[["filename"]].merge(
    ling_df, on="filename", how="left", validate="one_to_one"
)

ling_cols = [
    "word_count", "sentence_count", "avg_words_per_sentence",
    "unique_word_count", "vocabulary_diversity",
    "repeated_adjacent_count", "repeated_word_rate",
    "filler_count", "incomplete_ending",
]

zero = (y == 0)
nonzero = ~zero

print("=== GROUP SIZES ===")
print(f"Zero-score: {zero.sum()} | Other scores: {nonzero.sum()}")
print(f"Missing linguistic values in zero-score group: "
      f"{ling.loc[zero, ling_cols].isna().sum().sum()}")

print("\n=== LINGUISTIC FEATURES: ZERO VS NONZERO ===")
rows = []
for col in ling_cols:
    a = pd.to_numeric(ling.loc[zero, col], errors="coerce").dropna()
    b = pd.to_numeric(ling.loc[nonzero, col], errors="coerce").dropna()

    if len(a) and len(b):
        p = mannwhitneyu(a, b, alternative="two-sided").pvalue
        rows.append({
            "feature": col,
            "zero_median": a.median(),
            "other_median": b.median(),
            "zero_mean": a.mean(),
            "other_mean": b.mean(),
            "MW_p": p,
        })

print(pd.DataFrame(rows).round(4).to_string(index=False))

print("\n=== ACOUSTIC / WAVLM FEATURE SUMMARY ===")
for name, arr in [("Acoustic", acoustic), ("WavLM", wavlm)]:
    zero_mean = arr[zero].mean(axis=0)
    other_mean = arr[nonzero].mean(axis=0)
    pooled_std = arr.std(axis=0) + 1e-8
    standardized_gap = np.abs(zero_mean - other_mean) / pooled_std

    print(
        f"{name}: mean absolute standardized feature gap="
        f"{standardized_gap.mean():.4f}; "
        f"largest gap={standardized_gap.max():.4f}"
    )

print("\n=== ZERO-SCORE SAMPLE LINGUISTIC VALUES ===")
print(
    train_df.loc[zero, ["filename", "label"]]
    .merge(
        ling[["filename"] + ling_cols],
        on="filename",
        how="left",
        validate="one_to_one",
    )
    .sort_values("filename")
    .to_string(index=False, max_rows=45)
)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, (X_train, X_test) in blocks.items():
    X_all = np.vstack([X_train, X_test])
    domain = np.r_[np.zeros(len(X_train)), np.ones(len(X_test))]

    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced"),
    )

    aucs = cross_val_score(
        model,
        X_all,
        domain,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1,  # Avoid parallel worker processes on this machine
    )

    results.append({
        "Feature block": name,
        "CV AUC mean": aucs.mean(),
        "CV AUC SD": aucs.std(ddof=1),
        "Fold AUCs": np.round(aucs, 3).tolist(),
    })

print("=== TRAIN/TEST ADVERSARIAL VALIDATION ===")
print("Domain label: 0=train, 1=test")
print(pd.DataFrame(results).round(4).to_string(index=False))

print("\n=== FEATURE SHAPES / NONFINITE VALUES ===")
for name, (X_train, X_test) in blocks.items():
    print(
        f"{name}: train={X_train.shape}, test={X_test.shape}, "
        f"nonfinite_train={(~np.isfinite(X_train)).sum()}, "
        f"nonfinite_test={(~np.isfinite(X_test)).sum()}"
    )

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Reuse the combined cached features from the previous diagnostic cell.
X_train, X_test = blocks["Combined"]
X_all = np.vstack([X_train, X_test])
domain = np.r_[np.zeros(len(X_train), dtype=int),
               np.ones(len(X_test), dtype=int)]

domain_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
domain_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced"),
)

# Every probability is out-of-fold.
p_test = cross_val_predict(
    domain_model,
    X_all,
    domain,
    cv=domain_cv,
    method="predict_proba",
    n_jobs=1,
)[:, 1]

auc = roc_auc_score(domain, p_test)
p_train_rows = p_test[:len(train_df)]
p_test_rows = p_test[len(train_df):]

print(f"OOF domain-classifier AUC: {auc:.4f}")

print("\n=== TRAIN RECORDINGS MOST TEST-LIKE ===")
train_rank = train_df[["filename", "label"]].copy()
train_rank["predicted_test_probability"] = p_train_rows
print(
    train_rank.sort_values("predicted_test_probability", ascending=False)
    .head(15)
    .round(4)
    .to_string(index=False)
)

print("\n=== TEST RECORDINGS MOST TRAIN-LIKE ===")
test_rank = test_df[["filename"]].copy()
test_rank["predicted_test_probability"] = p_test_rows
print(
    test_rank.sort_values("predicted_test_probability", ascending=True)
    .head(15)
    .round(4)
    .to_string(index=False)
)

print("\n=== DOMAIN PROBABILITY SUMMARY ===")
print("Training rows:")
print(pd.Series(p_train_rows).describe(percentiles=[.1, .25, .5, .75, .9]).round(4))
print("\nTest rows:")
print(pd.Series(p_test_rows).describe(percentiles=[.1, .25, .5, .75, .9]).round(4))

In [ ]:
from pathlib import Path
import pandas as pd
import soundfile as sf

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"

train_df = pd.read_csv(DATA / "train.csv")
test_df = pd.read_csv(DATA / "test.csv")

# Index WAV files once so nested train/test directories are both supported.
wav_paths = {}
for path in DATA.rglob("*"):
    if path.is_file() and path.suffix.lower() == ".wav":
        wav_paths.setdefault(path.name.lower(), path)

print(f"WAV files found under dataset directory: {len(wav_paths)}")

rows = []
missing = []

for split, frame in [("train", train_df), ("test", test_df)]:
    for filename in frame["filename"].astype(str):
        path = wav_paths.get(filename.lower())

        if path is None:
            missing.append((split, filename))
            continue

        try:
            info = sf.info(str(path))
            rows.append({
                "split": split,
                "filename": filename,
                "duration_s": info.duration,
                "sample_rate": info.samplerate,
                "channels": info.channels,
                "subtype": info.subtype,
                "file_size_mb": path.stat().st_size / (1024 ** 2),
            })
        except Exception as exc:
            rows.append({
                "split": split,
                "filename": filename,
                "metadata_error": str(exc),
            })

meta = pd.DataFrame(rows)

print(f"Metadata records collected: {len(meta)}")
print(f"Missing WAV references: {len(missing)}")
if missing:
    print("First missing references:", missing[:20])

if not meta.empty:
    print("\n=== METADATA COUNTS BY SPLIT ===")
    for col in ["sample_rate", "channels", "subtype"]:
        if col in meta.columns:
            print(f"\n{col}:")
            print(pd.crosstab(meta["split"], meta[col], dropna=False).to_string())

    print("\n=== NUMERIC METADATA SUMMARY ===")
    numeric_cols = ["duration_s", "file_size_mb"]
    print(meta.groupby("split")[numeric_cols].agg(
        ["count", "mean", "std", "min", "median", "max"]
    ).round(3).to_string())

    print("\n=== DURATION QUANTILES ===")
    print(meta.groupby("split")["duration_s"].quantile(
        [0, .1, .25, .5, .75, .9, 1]
    ).unstack().round(3).to_string())

    print("\n=== METADATA ERRORS ===")
    if "metadata_error" in meta.columns:
        errors = meta[meta["metadata_error"].notna()]
        print(errors[["split", "filename", "metadata_error"]].head(15).to_string(index=False))
    else:
        print("None")

In [ ]:
from pathlib import Path
import pandas as pd
from collections import Counter

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"

train_df = pd.read_csv(DATA / "train.csv")
test_df = pd.read_csv(DATA / "test.csv")

train_names = train_df["filename"].astype(str)
test_names = test_df["filename"].astype(str)

train_set = set(train_names.str.lower())
test_set = set(test_names.str.lower())
overlap = sorted(train_set & test_set)

print("=== FILENAME INTEGRITY ===")
print(f"Train rows: {len(train_names)} | unique filenames: {train_names.str.lower().nunique()}")
print(f"Test rows:  {len(test_names)} | unique filenames: {test_names.str.lower().nunique()}")
print(f"Train/test filename overlap: {len(overlap)}")
print(f"Duplicate rows within train: {train_names.str.lower().duplicated().sum()}")
print(f"Duplicate rows within test:  {test_names.str.lower().duplicated().sum()}")

if overlap:
    print("\nFirst overlapping filenames:")
    print(overlap[:30])

print("\n=== WAV FILE INVENTORY ===")
wav_paths = [
    p for p in DATA.rglob("*")
    if p.is_file() and p.suffix.lower() == ".wav"
]
basename_counts = Counter(p.name.lower() for p in wav_paths)

print(f"Physical WAV files found: {len(wav_paths)}")
print(f"Unique WAV basenames: {len(basename_counts)}")
print(f"Basenames appearing more than once: "
      f"{sum(count > 1 for count in basename_counts.values())}")

if overlap:
    print("\n=== OVERLAPPING ROWS AND LABELS ===")
    print(
        train_df.assign(_key=train_names.str.lower())
        .merge(
            test_df.assign(_key=test_names.str.lower()),
            on="_key",
            how="inner",
            suffixes=("_train", "_test"),
        )
        [["filename_train", "label", "filename_test"]]
        .head(30)
        .to_string(index=False)
    )

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import pandas as pd

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"

train_df = pd.read_csv(DATA / "train.csv")
test_df = pd.read_csv(DATA / "test.csv")

train_names = set(train_df["filename"].astype(str).str.lower())
test_names = set(test_df["filename"].astype(str).str.lower())
overlap = train_names & test_names

paths_by_name = defaultdict(list)
for path in DATA.rglob("*"):
    if path.is_file() and path.suffix.lower() == ".wav":
        paths_by_name[path.name.lower()].append(path)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

records = []
for name in sorted(overlap):
    paths = paths_by_name.get(name, [])
    for path in paths:
        records.append({
            "filename": name,
            "relative_path": str(path.relative_to(DATA)),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })

files_df = pd.DataFrame(records)

print("=== PHYSICAL FILES FOR OVERLAPPING FILENAMES ===")
print(f"Overlapping basenames: {len(overlap)}")
print(f"Physical WAV files examined: {len(files_df)}")
print(f"Overlapping names with no matching WAV: "
      f"{sum(name not in paths_by_name for name in overlap)}")
print(f"Overlapping names with exactly one physical file: "
      f"{sum(len(paths_by_name.get(name, [])) == 1 for name in overlap)}")
print(f"Overlapping names with multiple physical files: "
      f"{sum(len(paths_by_name.get(name, [])) > 1 for name in overlap)}")

print("\n=== EXAMPLE FILE PATHS AND HASHES ===")
print(files_df.head(12).to_string(index=False))

print("\n=== HASH COMPARISON BY BASENAME ===")
summary = []
for name in sorted(overlap):
    group = files_df[files_df["filename"] == name]
    if len(group) > 1:
        summary.append({
            "filename": name,
            "physical_copies": len(group),
            "unique_hashes": group["sha256"].nunique(),
            "identical_copies": group["sha256"].nunique() == 1,
            "relative_paths": " | ".join(group["relative_path"].tolist()),
        })

summary_df = pd.DataFrame(summary)
if len(summary_df):
    print(summary_df.head(25).to_string(index=False))
    print("\nDuplicate-basename summary:")
    print(f"Names with multiple copies: {len(summary_df)}")
    print(f"All copies identical: {summary_df['identical_copies'].sum()}")
    print(f"Copies differ by hash: {(~summary_df['identical_copies']).sum()}")
else:
    print("No duplicate physical copies found for the overlapping names.")

In [ ]:
from pathlib import Path
import pandas as pd
import soundfile as sf

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
DATA = ROOT / "data" / "Dataset_Final"

train_df = pd.read_csv(DATA / "train.csv")
test_df = pd.read_csv(DATA / "test.csv")

def collect_metadata(frame, split):
    folder = DATA / split
    records = []
    missing = []

    for filename in frame["filename"].astype(str):
        path = folder / filename

        if not path.is_file():
            missing.append(filename)
            continue

        info = sf.info(str(path))
        records.append({
            "split": split,
            "filename": filename,
            "duration_s": info.duration,
            "sample_rate": info.samplerate,
            "channels": info.channels,
            "file_size_mb": path.stat().st_size / (1024 ** 2),
        })

    return pd.DataFrame(records), missing

train_meta, train_missing = collect_metadata(train_df, "train")
test_meta, test_missing = collect_metadata(test_df, "test")
meta = pd.concat([train_meta, test_meta], ignore_index=True)

print("=== EXPLICIT-PATH CHECK ===")
print(f"Train metadata rows: {len(train_meta)} / {len(train_df)}")
print(f"Test metadata rows:  {len(test_meta)} / {len(test_df)}")
print(f"Missing train WAVs: {len(train_missing)}")
print(f"Missing test WAVs:  {len(test_missing)}")

print("\n=== CORRECT DURATION SUMMARY ===")
print(meta.groupby("split")["duration_s"].agg(
    ["count", "mean", "std", "min", "median", "max"]
).round(3).to_string())

print("\n=== FORMAT COUNTS ===")
for col in ["sample_rate", "channels"]:
    print(f"\n{col}:")
    print(pd.crosstab(meta["split"], meta[col]).to_string())

print("\n=== OVERLAPPING BASENAME DURATION COMPARISON ===")
paired = train_meta[["filename", "duration_s"]].merge(
    test_meta[["filename", "duration_s"]],
    on="filename",
    suffixes=("_train", "_test"),
    validate="one_to_one",
)
paired["duration_difference_s"] = (
    paired["duration_s_train"] - paired["duration_s_test"]
).abs()

print(f"Paired filenames: {len(paired)}")
print("Absolute duration difference (seconds):")
print(paired["duration_difference_s"].describe(
    percentiles=[.25, .5, .75, .9]
).round(3).to_string())

print("\nPairs with largest duration differences:")
print(
    paired.sort_values("duration_difference_s", ascending=False)
    .head(10)
    .round(3)
    .to_string(index=False)
)

In [ ]:
import json
from pathlib import Path

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
NOTEBOOK = ROOT / "notebooks" / "01_baseline.ipynb"

with NOTEBOOK.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "wavlm", "librosa.load", "sf.read", "soundfile",
    "rglob", "glob(", "os.walk", "audio_path",
    "wav_path", ".wav", "X_train_acoustic",
    "X_train_wavlm", "feature extraction",
]

print(f"Notebook: {NOTEBOOK}")
print(f"Total cells: {len(notebook.get('cells', []))}")

matches = 0
for idx, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()
    matching_lines = [
        (line_no, line)
        for line_no, line in enumerate(lines, start=1)
        if any(keyword.lower() in line.lower() for keyword in keywords)
    ]

    if matching_lines:
        matches += 1
        print(f"\n{'=' * 72}")
        print(f"CELL {idx} — {len(lines)} lines")
        for line_no, line in matching_lines:
            start = max(1, line_no - 2)
            end = min(len(lines), line_no + 2)
            print(f"\n  Context around line {line_no}:")
            for j in range(start, end + 1):
                print(f"  {j:>3}: {lines[j - 1]}")

print(f"\nMatching code cells: {matches}")

In [ ]:
import json
from pathlib import Path

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
NOTEBOOK = ROOT / "notebooks" / "01_baseline.ipynb"

with NOTEBOOK.open("r", encoding="utf-8") as f:
    nb = json.load(f)

for cell_idx in [20, 21, 22, 23]:
    cell = nb["cells"][cell_idx]
    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    print(f"\n{'=' * 24} CELL {cell_idx} {'=' * 24}")

    if cell_idx == 20:
        # Show configuration, cache naming, and split processing.
        ranges = [(0, 85), (85, 145), (140, 265)]
    elif cell_idx == 21:
        ranges = [(0, len(lines))]
    elif cell_idx == 22:
        ranges = [(0, 25), (155, 225)]
    else:
        ranges = [(0, 100)]

    printed = set()
    for start, end in ranges:
        for i in range(start, min(end, len(lines))):
            if i not in printed:
                print(f"{i + 1:>4}: {lines[i]}")
                printed.add(i)
        print("...")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
OUT = ROOT / "outputs"

print("Relevant saved prediction/metric files:")
for p in sorted(OUT.rglob("*")):
    if p.is_file() and any(
        key in p.name.lower()
        for key in ["oof", "prediction", "calibrat", "cv_", "comparison"]
    ):
        print(f"\n{p.relative_to(ROOT)} | {p.stat().st_size:,} bytes")
        if p.suffix.lower() == ".csv":
            try:
                df = pd.read_csv(p, nrows=3)
                print("Columns:", df.columns.tolist())
                print(df.to_string(index=False, max_cols=8))
            except Exception as e:
                print("Could not preview:", type(e).__name__, str(e)[:120])

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
path = ROOT / "outputs" / "catboost_combined_oof.csv"

df = pd.read_csv(path)
df = df[["actual", "catboost_oof"]].dropna()

y = df["actual"].to_numpy(dtype=float)
p = df["catboost_oof"].to_numpy(dtype=float)
resid = y - p

# Regress actual scores on OOF predictions.
slope, intercept = np.polyfit(p, y, 1)

# Bootstrap paired rows to estimate uncertainty.
rng = np.random.default_rng(42)
n = len(y)
boot = []

for _ in range(3000):
    idx = rng.integers(0, n, size=n)
    if np.std(p[idx]) > 1e-10:
        b1, b0 = np.polyfit(p[idx], y[idx], 1)
        boot.append((b0, b1))

boot = np.asarray(boot)
ci_intercept = np.percentile(boot[:, 0], [2.5, 97.5])
ci_slope = np.percentile(boot[:, 1], [2.5, 97.5])

print("Rows:", n)
print(f"OOF RMSE: {np.sqrt(np.mean(resid**2)):.4f}")
print(f"OOF MAE:  {np.mean(np.abs(resid)):.4f}")
print(f"Mean residual (actual - prediction): {np.mean(resid):+.4f}")
print(f"Calibration intercept: {intercept:+.4f}")
print(f"Intercept bootstrap 95% CI: [{ci_intercept[0]:+.4f}, {ci_intercept[1]:+.4f}]")
print(f"Calibration slope: {slope:.4f}")
print(f"Slope bootstrap 95% CI: [{ci_slope[0]:.4f}, {ci_slope[1]:.4f}]")

# Equal-sized bins based on predicted score.
bins = pd.qcut(pd.Series(p), q=8, duplicates="drop")
reliability = pd.DataFrame({
    "predicted": p,
    "actual": y,
    "residual": resid,
    "bin": bins
}).groupby("bin", observed=True).agg(
    n=("actual", "size"),
    mean_prediction=("predicted", "mean"),
    mean_actual=("actual", "mean"),
    mean_residual=("residual", "mean"),
    rmse=("residual", lambda x: np.sqrt(np.mean(x**2)))
).reset_index(drop=True)

print("\nReliability by predicted-score bin:")
print(reliability.round(4).to_string(index=False))

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
OUT = ROOT / "outputs"

files = [
    "calibration_comparison.csv",
    "calibration_experiment_results.csv",
    "calibration_oof_predictions.csv",
]

for name in files:
    path = OUT / name
    print(f"\n{'=' * 18} {name} {'=' * 18}")

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(5).to_string(index=False))

    if name == "calibration_oof_predictions.csv":
        print("\nMissing values:")
        print(df.isna().sum().to_dict())

        numeric = df.select_dtypes(include="number")
        print("\nNumeric summary:")
        print(numeric.describe().round(4).to_string())

In [ ]:
import json
from pathlib import Path

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
NOTEBOOK = ROOT / "notebooks" / "01_baseline.ipynb"

with NOTEBOOK.open("r", encoding="utf-8") as f:
    nb = json.load(f)

terms = (
    "calibrat",
    "calibration_oof_predictions",
    "calibration_experiment_results",
    "calibration_comparison",
    "catboost_combined_oof",
    "LinearRegression",
    "np.polyfit",
    "IsotonicRegression",
)

for idx, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    if any(term.lower() in source.lower() for term in terms):
        print(f"\n{'=' * 20} CELL {idx} {'=' * 20}")
        lines = source.splitlines()

        # Print the full source for short cells; otherwise show context
        # around matching lines so the output remains manageable.
        if len(lines) <= 100:
            print(source)
        else:
            selected = set()
            for i, line in enumerate(lines):
                if any(term.lower() in line.lower() for term in terms):
                    selected.update(range(max(0, i - 8), min(len(lines), i + 13)))

            previous = -2
            for i in sorted(selected):
                if i > previous + 1:
                    print("...")
                print(f"{i + 1:>4}: {lines[i]}")
                previous = i

In [ ]:
import json
from pathlib import Path

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
NOTEBOOK = ROOT / "notebooks" / "01_baseline.ipynb"

with NOTEBOOK.open("r", encoding="utf-8") as f:
    nb = json.load(f)

terms = [
    "calibration_oof_predictions",
    "calibration_experiment_results",
    "calibration_comparison",
    "catboost_combined_oof",
    "IsotonicRegression",
    "LinearRegression",
    "calibrated_oof",
    "outer_train_idx",
    "outer_valid_idx",
]

found = 0

for idx, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    matches = [
        (line_no, line)
        for line_no, line in enumerate(source.splitlines(), start=1)
        if any(term.lower() in line.lower() for term in terms)
    ]

    if matches:
        found += 1
        print(f"\n--- Notebook cell {idx} | matching source lines ---")
        for line_no, line in matches:
            print(f"{line_no:>4}: {line[:240]}")

print(f"\nTotal matching code cells: {found}")

In [ ]:
import json
from pathlib import Path

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
NOTEBOOK = ROOT / "notebooks" / "01_baseline.ipynb"

with NOTEBOOK.open("r", encoding="utf-8") as f:
    nb = json.load(f)

cell_index = 96
cell = nb["cells"][cell_index]

print(f"--- Complete source of notebook cell {cell_index} ---\n")
print("".join(cell.get("source", [])))

In [ ]:
import json
from pathlib import Path

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
NOTEBOOK = ROOT / "notebooks" / "01_baseline.ipynb"

with NOTEBOOK.open("r", encoding="utf-8") as f:
    nb = json.load(f)

terms = [
    "submission_blend_calibrated_catboost80_hgb20",
    "FINAL_BEST_submission_0.4897",
    "catboost80",
    "calibrated_cat",
    "blend_calibrated",
]

for idx, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    matches = [
        line.strip()
        for line in source.splitlines()
        if any(term.lower() in line.lower() for term in terms)
    ]

    if matches:
        print(f"\n--- Notebook cell {idx} ---")
        for line in matches:
            print(line[:250])

In [ ]:
import json
from pathlib import Path

ROOT = Path(r"C:\Users\SRIKANTH\Downloads\SHL_Grammar_Assessment")
NOTEBOOK = ROOT / "notebooks" / "01_baseline.ipynb"

with NOTEBOOK.open("r", encoding="utf-8") as f:
    nb = json.load(f)

for idx in [97, 98, 99]:
    cell = nb["cells"][idx]
    print(f"\n{'=' * 20} CELL {idx} {'=' * 20}\n")
    print("".join(cell.get("source", [])))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.home() / "Downloads" / "SHL_Grammar_Assessment"
out = project / "outputs"

test_df = pd.read_csv(
    project / "data" / "Dataset_Final" / "test.csv"
)

files = {
    "Raw CatBoost": out / "submission_catboost_combined.csv",
    "Calibrated CatBoost": out / "submission_calibrated_catboost.csv",
    "Blended submission": out / "submission_blend_calibrated_catboost80_hgb20.csv",
    "Backup": out / "FINAL_BEST_submission_0.4897.csv",
}

loaded = {}

for name, path in files.items():
    print(f"\n--- {name} ---")

    if not path.exists():
        print("MISSING:", path)
        continue

    df = pd.read_csv(path)
    loaded[name] = df

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("Filename order matches test.csv:",
          len(df) == len(test_df)
          and "filename" in df.columns
          and df["filename"].equals(test_df["filename"]))

    if "label" in df.columns:
        print("Missing predictions:", int(df["label"].isna().sum()))
        print("Finite predictions:",
              bool(np.isfinite(df["label"].to_numpy(dtype=float)).all()))
        print("Range:",
              round(df["label"].min(), 4),
              "to",
              round(df["label"].max(), 4))
        print("Mean:", round(df["label"].mean(), 4))

if "Blended submission" in loaded and "Backup" in loaded:
    a = loaded["Blended submission"]
    b = loaded["Backup"]

    print("\n--- Backup comparison ---")
    print("Identical filename order:", a["filename"].equals(b["filename"]))
    print("Predictions identical:",
          np.allclose(a["label"], b["label"], rtol=0, atol=0, equal_nan=True))